# RAG answer reviews

The latest prompt, table-extraction and selective-rewriting experiments appear first. Earlier Qwen/hybrid and baseline comparisons follow. This notebook makes no model calls; ratings are assistant source reviews, not independent human labels.


In [1]:
from pathlib import Path
from types import SimpleNamespace
import sys
from IPython.display import Markdown, display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'evaluation/manual_review.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run from inside the repository.')
sys.path.insert(0, str(ROOT / 'evaluation'))
import manual_review


In [2]:
display(Markdown((ROOT / "evaluation/results/prompt_tables_rewrite_assessment.md").read_text()))

# Prompt, table and query-rewriting experiments

**Selective query rewriting is the most promising candidate, but none of these changes clears the existing adoption threshold.** It raises complete answers from 20 to 21 and adds no new major errors. The tighter prompt removes the existing major error but drops completeness to 19; adding table chunks drops completeness to 18. Keep the accepted Qwen + hybrid coverage configuration as the default. Preserve rewriting as an opt-in experiment for broader validation, and use table chunks selectively in a future test rather than enabling them globally.

| Configuration | Complete | Partial | Major error | Correct abstention | Mean answer words |
|---|---:|---:|---:|---:|---:|
| Current Qwen + hybrid | 20 | 2 | 1 | 2 | 101.6 |
| Focused prompt | 19 | 4 | 0 | 2 | 40.5 |
| Table-aware retrieval | 18 | 4 | 1 | 2 | 88.7 |
| Selective rewriting | 21 | 1 | 1 | 2 | 102.7 |

**Interpretation:** 23 questions are answerable and two intentionally unsupported. Complete requires the requested content without a material factual defect; citation placement is tracked separately. Paper IDs without recognizable names remain Partial under the established presentation criterion. These are assistant source reviews on a small development set, not independent human ratings or statistically significant estimates.

## What each experiment showed

1. **Tighter answer policy — mixed, not adopted.** The full run is shorter and avoids the clinician-preference count attribution error, but a comparison caveat changes stronger *systems* to stronger *submissions*, and a cross-paper answer becomes too vague about costs. A feedback-adjusted second policy still ignores some instructions and attributes non-expert comprehensibility ratings to clinicians. That candidate was stopped after four completed cases; the interrupted fifth response is not counted.
2. **Table-aware extraction/chunking — better extraction, worse overall answers, not adopted.** Visual review confirms that table headers, row labels, percentages and judge groups are recovered correctly for the inspected tables. The TMI accuracy answer now includes the correct baseline and improved values explicitly. However, new table chunks also alter ranking on non-table questions: the PMRI definition answer cites an unrelated results table and conflates task definition with training supervision; another cross-paper answer loses paper names and compute-budget detail. Clearer source formatting alone does not guarantee better selection or generation.
3. **Selective rewriting — modest positive signal, still experimental.** Eight short rewrites change four questions' retrieved contexts. The Router answer now covers ordering, stopping and final selection without explicitly describing specialist execution as the computation of reservation prices; it remains repetitive. The other three changed answers retain their previous grades. Rewrite generation takes 16.5 seconds in total (about 2.1 seconds per selected query) in this warm local run, excluding retrieval and answer generation; this is not a controlled latency benchmark.
4. **Remaining issues:** The accepted baseline's clinician-preference numerical attribution error remains in the table and rewrite variants because those inputs are unchanged. Paper-ID-only answers and misplaced citations also remain. None of these trials enables an LLM judge, corrective loop or HyDE.

**Recommended next decision:** validate selective rewriting on the 15 additional questions before making it the default. For output quality, a deterministic mapping from known cited paper IDs to their titles is a lower-risk way to improve business readability than adding more prompt instructions. For tables, first try routing only explicit metric/comparison questions to table chunks while preserving explanatory prose.

## Controls and limits

- Qwen 3.5 9B, temperature 0, seed 42, citation-only schema, thinking disabled, no repair, context 12,288 and output limit 1,800 throughout. Five retrieved passages maximum, fetch 60, per-paper cap 4 and reranker gap 8.
- Prompt experiment: all 25 answers were freshly generated on exactly the baseline hybrid evidence. Only the prompt policy changed. A second prompt variant was stopped after four completed regression cases; it is not counted as a full run.
- Table and rewriting experiments independently retain the accepted coverage policy. They are not stacked on the rejected focused prompt. A fresh control retrieval run reproduced all 25 original hybrid contexts byte-for-byte.
- Table experiment: eight confident numeric tables produce 21 additional chunks across two PDFs. Captions, header context, row labels and group labels are preserved. Unsupported layouts are skipped; original prose is retained. Three questions retrieve a table chunk, but eight contexts change because new corpus entries also affect hybrid rankings. Only those eight answers were regenerated.
- Rewriting: a fixed question-text rule selects broad mechanisms, comparisons and multi-paper questions. Eight rewrites produce four changed contexts. Original and rewritten hybrid candidates are fused, then reranked against the original question. Only real source passages go to the answer generator. Four changed answers are regenerated; unchanged answers are reused.
- Assembly verifies identical questions, grouped context, citation IDs, source lists and generation settings before reuse. The baseline engine hash was reproduced exactly after removing only the two added, unused experimental policy assignments; generation code for coverage is unchanged.
- Literal evidence-phrase coverage: baseline 25/28, tables 24/28, rewriting 25/28. These are wording checks, not correctness scores.
- Acceptance threshold remains +2 complete answers or removal of two major errors, without new major errors or unsupported answers. No automatic production promotion is based on shorter output or more matching phrases alone.
- The 15-question holdout was not run here: this task tested the requested items 1, 2 and 4. No HyDE or corrective-RAG calls were added.
- 150 tests passed, including original-query reranking, table row/header/group association, and strict checks for reusing prior outputs.

## Source-table example

The layout-aware extractor stores explicit metric/value pairs, for example:

```text
Workflow summary: Cov.=54.07; Exec.=59.35; Safe.=91.74; Align.=66.18; Acc.=14.29
Ours: Cov.=54.95; Exec.=67.65; Safe.=90.57; Align.=67.99; Acc.=18.57
```

The chunk repeats the caption defining Acc. as held-out task accuracy and the values as percentages. This extraction improvement did not produce a net answer-quality gain when added indiscriminately to retrieval.

## Question-by-question outcomes

| Question | Current | Focused | Tables | Rewriting |
|---|---|---|---|---|
| q01 | Partial | Partial | Partial | Complete |
| q02 | Complete | Complete | Complete | Complete |
| q03 | Complete | Complete | Complete | Complete |
| q04 | Complete | Complete | Complete | Complete |
| q05 | Complete | Complete | Complete | Complete |
| q06 | Complete | Complete | Complete | Complete |
| q07 | Complete | Complete | Complete | Complete |
| q08 | Complete | Partial | Complete | Complete |
| q09 | Complete | Complete | Complete | Complete |
| q10 | Complete | Complete | Complete | Complete |
| q11 | Complete | Complete | Complete | Complete |
| q12 | Complete | Complete | Complete | Complete |
| q13 | Complete | Complete | Complete | Complete |
| q14 | Complete | Complete | Complete | Complete |
| q15 | Complete | Complete | Complete | Complete |
| q16 | Complete | Complete | Complete | Complete |
| q17 | Complete | Complete | Partial | Complete |
| q18 | Complete | Complete | Complete | Complete |
| q19 | Complete | Complete | Complete | Complete |
| q20 | Major issue | Complete | Major issue | Major issue |
| q21 | Partial | Partial | Partial | Partial |
| q22 | Complete | Partial | Partial | Complete |
| q23 | Complete | Complete | Complete | Complete |
| q24 | Correct abstention | Correct abstention | Correct abstention | Correct abstention |
| q25 | Correct abstention | Correct abstention | Correct abstention | Correct abstention |

## Explanations and raw outputs

The following details preserve model outputs; no generated answers were edited to improve their grades.

### Focused prompt

<details><summary>q01 — Partial: How does Pandora's Router decide when an expensive specialist estimate is worth paying for?</summary>

Still conflates running a specialist with calculating reservation prices, and omits explicit ordering and final selection.

- Pandora's Router decides when an expensive specialist estimate is worth paying for by computing reservation prices that quantify the expected value-of-information for querying a more costly and accurate value estimate. [2]
- The system computes reservation prices by running the specialist for a few reasoning tokens and checking if it is on the right track. [2]
- Pandora's Router stops querying expensive estimates when no remaining query is worth its cost. [2]

Citation concern: The first claim cites [2][4] rather than the explicit mechanism in [1].

</details>

<details><summary>q02 — Complete: What is the difference between Pandora's Router and Pandora's Bidder?</summary>

Explains centralized Router versus decentralized Bidder and global welfare versus local surplus, without turning high costs into automatic refusal.

- Pandora's Router queries the costly specialist only when the value of information exceeds the cost, interpolating between baselines that either always query or never query. [2][3]
- Pandora's Bidder operates in a decentralized setting where specialists face posted prices and must decide whether to invest in more accurate self-assessments before claiming queries. [3]
- In Pandora's Bidder, maximizing local surplus does not always lead to better overall allocative efficiency because strategic bidders may capture higher surplus at the cost of global welfare. [3]
- The tension between local surplus maximization and global allocative efficiency is absent in Pandora's Router, where the router internalizes all costs and optimizes global welfare directly. [3]
- Pandora's Bidder corresponds to a single stage of an ascending-price mechanism with costly preference elicitation, isolating core components of decentralized prompt allocation under private value estimation with profit incentives. [4]

</details>

<details><summary>q03 — Complete: In which three domains was the costly-value-estimation routing policy evaluated?</summary>

Names all three evaluation domains.

- The costly-value-estimation routing policy was evaluated across three domains: a multi-LLM routing benchmark (EmbedLLM), retrieval-augmented specialists for factoid QA, and inference-time computation for mathematical reasoning. [2][3]

</details>

<details><summary>q04 — Complete: What limitation appears when competing specialists' estimates are noisy?</summary>

Correctly states that noisy competing estimates can benefit one specialist at others' expense.

- When competing estimates are noisy, value-of-information reasoning can increase the strategic specialist's utility at the expense of others. [2]

Citation concern: The statement is supported by passage [1], but cites [2].

</details>

<details><summary>q05 — Complete: How many frozen repositories and training algorithm families are in AI4AI-Bench?</summary>

Correct ten repositories and ten algorithm families.

- AI4AI-Bench contains ten frozen research repositories. [2][3]
- These ten repositories cover ten distinct families of training algorithms. [3]

</details>

<details><summary>q06 — Complete: What is the AI4AI-Bench time budget for an agent and for the replay?</summary>

Correct four-hour agent budget on one B300 and up-to-twelve-hour replay.

- The agent has a time budget of 4 hours on one B300 GPU to rewrite the training algorithm. [2][4]
- The replay (rerun) has a time budget of up to 12 hours. [2][4]

</details>

<details><summary>q07 — Complete: How are the incommensurable AI4AI-Bench task metrics normalized?</summary>

Correct 0, 0.1 and 1.0 anchors, but the cited passage is wrong.

- The incommensurable AI4AI-Bench task metrics are normalized such that 0 represents an uninformative model, 0.1 represents the algorithm the repository ships, and 1.0 represents the task optimum. [2]

Citation concern: The only answer claim cites unrelated passage [2] rather than the scale definition in [4].

</details>

<details><summary>q08 — Partial: How do submissions that change the learning algorithm compare with those that do not?</summary>

Correct 0.226 versus 0.126, but changes the caveat from stronger systems to stronger submissions. This preserves a factual attribution imprecision.

- Submissions that change the learning algorithm achieve a mean score of 0.226, while those that do not change it achieve a mean score of 0.126. [2][3]
- The algorithmic layer is where the distance to a better algorithm is closed, and most submissions never go there. [3]
- Submissions that reach the learning procedure more often are the stronger ones to begin with. [2][3]

Citation concern: No numeric warning, but semantic caveat attribution is wrong.

</details>

<details><summary>q09 — Complete: What two kinds of task structure does TMI recover from computer-use traces?</summary>

Correct objective hierarchy and execution-control-flow model.

- TMI recovers task models that pair a hierarchical objective model of recursive goal decomposition with a procedure model of the control flow that organized the execution. [2][3]

</details>

<details><summary>q10 — Complete: How well does task model induction group interleaved tasks?</summary>

Correct 0.974 grouping agreement.

- The method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings. [2]

</details>

<details><summary>q11 — Complete: How much of observed execution does TMI reconstruct?</summary>

Correct 74.9% of observed execution steps.

- TMI reconstructs 74.9% of the observed execution steps. [2][4]

</details>

<details><summary>q12 — Complete: What was the held-out task accuracy gain from skills derived from TMI?</summary>

Correct 30.0% held-out accuracy improvement, without unrelated table statistics.

- Skills derived from TMI's task models improve held-out task accuracy by 30.0% over the strongest baseline. [2][5]

</details>

<details><summary>q13 — Complete: How many commuters and respondent-scenario observations did the weather travel survey collect?</summary>

Correct 92 commuters and 454 respondent-scenario observations.

- The weather travel survey collected mode choices from 92 student commuters. [2][3]
- The survey yielded 454 respondent–scenario observations across five predefined weather scenarios. [1][3]

</details>

<details><summary>q14 — Complete: Which travel modes responded most clearly to adverse weather and snow?</summary>

Correct cycling sensitivity and increased transit use under snow.

- Cycling was particularly sensitive to adverse weather, while public transit use increased substantially under the Snowy scenario. [2][4]

</details>

<details><summary>q15 — Complete: How did random forest and the best text-only zero-shot LLM compare on five-class travel prediction?</summary>

Correct 69.6% versus 69.9% without asserting statistical significance.

- Random forest achieved the highest trained-model accuracy of 69.6% five-class accuracy, while the best text-only zero-shot LLM produced a comparable point estimate of 69.9% without task-specific fitting to the survey records. [2][3]

</details>

<details><summary>q16 — Complete: What was the best vision-based travel mode prediction accuracy?</summary>

Correct best observed vision accuracy of 71.5%.

- The best vision-based configuration reached a five-class accuracy of 71.5%. [2][3]

</details>

<details><summary>q17 — Complete: What is the patient-oriented medical report interpretation task introduced with G-CARL?</summary>

Correct patient-facing multimodal interpretation using the user query and dialogue history.

- Patient-oriented Medical Report Interpretation (PMRI) is a novel open-ended multimodal generation task that requires models to explain medical reports in accurate and accessible language based on a user's query and dialogue history. [2][3]
- PMRI must align professional medical knowledge with patient-centered communication by transforming report evidence into accessible explanations, addressing user-specific concerns, and offering clinically cautious guidance. [4]
- Unlike short, deterministic outputs of medical VQA or conventional report generation, PMRI produces long-form, patient-facing explanations grounded in multimodal reports and extended patient-doctor dialogue. [4]

</details>

<details><summary>q18 — Complete: How does G-CARL supervise factuality and response coverage?</summary>

Explains multi-source verification and weighted checklists in two distinct claims.

- G-CARL supervises factuality through a retrieval-grounded branch that verifies report-grounded medical claims against evidence retrieved from a multi-source medical datastore. [2][4]
- G-CARL supervises response coverage through a case-specific checklist branch that constructs a weighted checklist via MLLM generation followed by clinician-guided refinement to evaluate checklist coverage of the generated response. [2][4]

</details>

<details><summary>q19 — Complete: What benchmark and evaluation dimensions accompany G-CARL?</summary>

Names MMedReport and all three dimensions; extra factual evaluation details are supported.

- G-CARL is evaluated on the MMedReport benchmark, which contains 2,450 instances of real-world multimodal patient-oriented medical report interpretation tasks. [2][4]
- The evaluation protocol for G-CARL includes three clinician-defined subjective dimensions: Medical Accuracy, Demand Satisfaction, and Expression Quality. [1]
- In addition to subjective evaluation, G-CARL is assessed using an external benchmark called CMB, which evaluates medical QA accuracy and the professionalism of open-ended clinical interpretation under an LLM-as-a-judge protocol. [1]
- Objective evaluation metrics are also reported alongside holistic assessment for G-CARL. [1]

Citation concern: The 2,450-instance claim cites [2][4] instead of supporting [1]. The subjective/objective protocol claim cites [4] rather than [1].

</details>

<details><summary>q20 — Complete: What did clinician pairwise preferences find about G-CARL?</summary>

Correct clinician preference direction and dimensions; omits the earlier unsupported count-to-baseline mapping.

- Clinician pairwise preferences found that G-CARL is consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction. [2][1]

</details>

<details><summary>q21 — Partial: Which papers address auditable AI workflows derived from user activity or survey responses?</summary>

Identifies the correct two paper IDs and relevance, but still omits recognizable paper titles.

- Paper ID 2608.20320 addresses auditable AI workflows derived from survey responses by defining agents as software components that produce structured and auditable outputs. [2][3]
- Paper ID 2608.20319 addresses auditable AI workflows derived from user activity by introducing TASKMODELINDUCTION(TMI) to derive symbolic, auditable, and reusable models of how everyday work is done from naturalistic computer-use traces. [1]

Citation concern: Paper-ID numeric warnings are metadata false positives, not fabricated result numbers.

</details>

<details><summary>q22 — Partial: Which papers evaluate AI agents or model specialists using benchmark tasks and costs?</summary>

Names both papers but largely repeats the question, without explaining the agent compute/time budget or specialist estimation-cost trade-off.

- Pandora's AI Model Routing Box evaluates model specialists using benchmark tasks and costs. [2][5]
- AI4AI-Bench evaluates AI agents using benchmark tasks. [3]

</details>

<details><summary>q23 — Complete: Which papers discuss evidence-grounded explanations and auditable task models?</summary>

Names TMI and G-CARL and correctly assigns their distinct relevance.

- The paper Inducing Task Models from Computer-Use Traces discusses auditable task models. [2][4]
- The paper G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation discusses evidence-grounded explanations. [3]

</details>

<details><summary>q24 — Correct abstention: What is KPMG's 2026 revenue forecast?</summary>

Correctly declines to invent a KPMG forecast.

The provided papers do not address this question.

</details>

<details><summary>q25 — Correct abstention: Which of these papers reports the stock price of NVIDIA on October 14, 2026?</summary>

Correctly declines to invent a NVIDIA stock price.

The provided papers do not address this question.

</details>

### Table-aware retrieval

<details><summary>q09 — Complete: What two kinds of task structure does TMI recover from computer-use traces?</summary>

Correct objective hierarchy and procedure/control-flow structure; repeated definitions remain.

- TMI recovers two complementary task structures: an objective model that hierarchically decomposes the task's goal into subobjectives, and a procedure model that composes control-flow operators to specify how execution is organized through sequencing and iteration. [2][4]
- The objective model represents a hierarchical decomposition of the task's objective into subobjectives that the user pursues. [4]
- The procedure model represents a trace-grounded composition of control-flow operators that specify how the execution of the task is organized through sequencing and iteration. [4]
- The final task model is structured as a tree where every node pairs an objective with a control-flow operator over its children. [4]

</details>

<details><summary>q12 — Complete: What was the held-out task accuracy gain from skills derived from TMI?</summary>

Correct 30% relative improvement and explicit 14.29% to 18.57% comparison. The cleaner table is retrieved, but the baseline already answered this correctly.

- Skills derived from TMI's task models improve held-out task accuracy by 30.0% over the strongest baseline. [2][5]
- The held-out accuracy improved from 14.29% for the strongest baseline (Workflow summary) to 18.57% for TMI, representing a 30% relative accuracy gain. [1][2]

</details>

<details><summary>q17 — Partial: What is the patient-oriented medical report interpretation task introduced with G-CARL?</summary>

Correct PMRI definition and user/dialogue context, but an extra claim assigns the framework's structured reward supervision to the task itself.

- The patient-oriented medical report interpretation task is defined as an open-ended multimodal generation task that requires models to explain medical reports in accurate and accessible language based on a user's query and dialogue history. [2]
- This task requires both evidence-grounded medical factuality and context-dependent patient communication, which existing medical vision-language tasks do not adequately capture. [2]
- The task involves two objectives that differ fundamentally in their verifiability but remain tightly coupled, making them difficult to optimize jointly under conventional supervised fine-tuning and holistic reinforcement learning paradigms. [2]
- The task aims to provide structured supervision for factuality, user-demand satisfaction, and expression quality without constraining response diversity. [2]

Citation concern: Every claim cites [2], which is a CMB results table. The actual task definition is in [4].

</details>

<details><summary>q18 — Complete: How does G-CARL supervise factuality and response coverage?</summary>

Correct multi-source claim verification and weighted checklist coverage in two clear claims.

- G-CARL supervises factuality by verifying report-grounded medical claims against evidence retrieved from a multi-source medical datastore to produce a factuality reward. [2][4]
- G-CARL supervises response coverage by constructing a case-specific weighted checklist through MLLM generation followed by clinician refinement to produce a checklist reward. [2][4]

</details>

<details><summary>q19 — Complete: What benchmark and evaluation dimensions accompany G-CARL?</summary>

Includes MMedReport, CMB and the three requested evaluation dimensions. Extra study details are supported, though it leads with the secondary benchmark.

- G-CARL is evaluated on the CMB benchmark, which assesses both medical QA accuracy and the professionalism of open-ended clinical interpretation under an LLM-as-a-judge protocol. [2][3]
- The evaluation dimensions for PMRI responses include Medical Accuracy, Demand Satisfaction, and Expression Quality, each scored according to a clinician-authored rubric ranging from -2 to 3. [1]
- In addition to the subjective dimensions, G-CARL also undergoes objective evaluation alongside holistic assessment on the MMedReport benchmark. [1]
- Human preference evaluation involves three professional clinicians comparing anonymized responses across the three evaluation dimensions and overall preference, with majority voting used for the final decision. [3]
- A separate group of 50 participants without medical training evaluates the comprehensibility of the generated interpretations to ensure patient understandability. [3]

Citation concern: The CMB protocol claim cites [2][3], while the full protocol is described in [1].

</details>

<details><summary>q21 — Partial: Which papers address auditable AI workflows derived from user activity or survey responses?</summary>

Correct two paper IDs and relevance, but still omits recognizable titles.

- Paper ID 2608.20320 addresses an auditable AI workflow derived from survey responses by defining agents as goal-directed, researcher-supervised software components that produce structured and auditable outputs. [2][3]
- Paper ID 2608.20319 addresses an auditable AI workflow derived from user activity by introducing TASKMODELINDUCTION (TMI), which derives symbolic, auditable, and reusable models of how everyday work is done from naturalistic computer-use traces. [1]

Citation concern: Paper-ID warnings are metadata false positives.

</details>

<details><summary>q22 — Partial: Which papers evaluate AI agents or model specialists using benchmark tasks and costs?</summary>

Gives paper IDs instead of titles and does not explain the fixed compute/time budget for the agent benchmark; specialist estimation costs are described.

- Paper ID 2608.20318 evaluates AI agents using benchmark tasks that measure their ability to design training algorithms for recursive self-improvement, where each task involves rewriting a training algorithm within a frozen repository and scoring the rerun against an original baseline. [2][4]
- Paper ID 2608.20316 evaluates model specialists by routing queries to them based on estimated value, where the cost of estimation varies between cheap noisy predictors and expensive accurate ones, and the system trades off these costs to achieve allocative performance. [1][3][5]

Citation concern: Paper-ID numeric warnings are metadata false positives.

</details>

<details><summary>q23 — Complete: Which papers discuss evidence-grounded explanations and auditable task models?</summary>

Identifies both TMI and G-CARL and their distinct relevance, but repeats TMI across five claims.

- The paper titled Inducing Task Models from Computer-Use Traces discusses auditable task models derived from naturalistic computer-use traces. [2][1]
- The paper titled Inducing Task Models from Computer-Use Traces discusses inducing symbolic, auditable, and reusable models of how everyday work is done. [2][1]
- The paper titled Inducing Task Models from Computer-Use Traces discusses inducing task models that are auditable and reusable. [2][1]
- The paper titled Inducing Task Models from Computer-Use Traces discusses inducing task models that are symbolic and auditable. [2][1]
- The paper titled Inducing Task Models from Computer-Use Traces discusses inducing task models that are reusable. [2][1]
- The paper titled G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation discusses evidence-grounded explanations. [3]
- The paper titled G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation discusses evidence-grounded medical factuality. [4]

</details>

### Selective rewriting

<details><summary>q01 — Complete: How does Pandora's Router decide when an expensive specialist estimate is worth paying for?</summary>

Now covers cheap estimates, reservation-price ordering, stopping and final selection, and describes reservation prices as deciding whether to run more reasoning rather than being computed by running it. Still ten repetitive claims and unrequested Bidder detail.

- Pandora's Router decides that an expensive specialist estimate is worth paying for by computing reservation prices that quantify the expected value-of-information for querying a more accurate value estimate, minus total estimation cost. [2][4]
- The system stops querying expensive estimates when no remaining query is worth its cost, effectively choosing the maximizer of the observed value estimate. [1]
- The mechanism involves specialists generating cheap, noisy value estimates first, after which the system computes reservation prices to decide whether to run a specialist for additional reasoning tokens to check if it is on the right track. [1]
- The decision rule balances cheap screening and costly reasoning by deciding which expensive estimates are worth computing based on the cost-adjusted reward, which may combine graded output quality with production costs such as inference cost or API fees. [1]
- Under a Gaussian signal model, the reservation-price-based policy has a closed form that is straightforward to compute, allowing the system to efficiently trade off expensive and cheap estimates. [4]
- The process follows a sequence where a concrete prompt is received, multiple specialists generate initial estimates, reservation prices are computed for the next stage of estimation, and then the system decides whether to proceed with expensive estimates. [1]
- The centralized policy inspects specialists in reservation price order, stopping when no remaining query is worth its cost, while the decentralized policy selects the high bidder in an auction where each specialist refines its own value estimate only when the market price makes inspection worthwhile. [1]
- Both Pandora's Router and Pandora's Bidder are able to gracefully trade-off expensive and cheap estimates to achieve strong allocative performance relative to baselines across a range of estimation costs. [4]
- The unified principle supports allocation by centralized routing and decentralized auctions, applying the same logic of balancing cheap screening and costly reasoning for routing. [1]
- The system computes reservation prices that quantify the expected value-of-information for querying a more costly and accurate value estimate, such as by running the specialist for a few reasoning tokens and checking if it is on the right track. [1]

Citation concern: The opening mechanism claim cites [2][4] rather than the explicit process in [1].

</details>

<details><summary>q21 — Partial: Which papers address auditable AI workflows derived from user activity or survey responses?</summary>

Correct papers and relevance, but still identifies them only by numeric paper ID.

- Paper ID 2608.20320 addresses auditable AI workflows derived from survey responses by defining agents as software components that produce structured and auditable outputs while maintaining researcher control over survey constructs and data processing. [2][4]
- Paper ID 2608.20319 addresses auditable AI workflows derived from user activity by introducing TASKMODELINDUCTION (TMI), which derives symbolic, auditable, and reusable models of how everyday work is done from naturalistic computer-use traces. [1]

Citation concern: Paper-ID warnings are metadata false positives.

</details>

<details><summary>q22 — Complete: Which papers evaluate AI agents or model specialists using benchmark tasks and costs?</summary>

Names both papers and explains specialist estimation costs and the benchmark agent/replay compute budgets.

- Pandora's AI Model Routing Box evaluates model specialists using benchmark tasks and costs by formalizing model routing with costly value estimation as an instance of Pandora's Box, where cheap estimators are fast but noisy while accurate estimators are expensive, and experiments across three domains show that the proposed policies trade off expensive and cheap estimates to achieve strong allocative performance relative to baselines. [2][5]
- AI4AI-Bench evaluates AI agents using benchmark tasks and costs by presenting 10 frozen research repositories spanning 10 training algorithm families, where an agent has 4 hours on one B300 to rewrite the training algorithm, its code is then rerun from scratch for up to 12 hours and scored by a fixed evaluator hidden from the agent against the repository's original algorithm under the same procedure. [3][4]

</details>

<details><summary>q23 — Complete: Which papers discuss evidence-grounded explanations and auditable task models?</summary>

Names both papers and correctly distinguishes auditable task models from evidence-grounded medical explanations.

- The paper Inducing Task Models from Computer-Use Traces discusses auditable task models derived from naturalistic computer-use traces. [2][1]
- The paper G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation discusses evidence-grounded explanations for medical reports. [3][4]

</details>



In [3]:
display(Markdown((ROOT / "evaluation/results/qwen_hybrid_assessment.md").read_text()))

# Full 25-question Qwen and hybrid-search comparison

**Hybrid search is worth continuing; the Qwen configuration is not ready to replace the current default.** Hybrid raises Qwen from 18 to 20 complete answers and reduces major errors from two to one. It meets the preselected +2-complete-answer threshold against Qwen with vector retrieval, without new major errors. Against the existing Llama configuration, however, it gains only one complete answer and introduces a major numerical-attribution error. Keep the current production settings while testing a more restrained answer policy with hybrid retrieval.

| Configuration | Complete | Partial | Major issue | Correct abstention |
|---|---:|---:|---:|---:|
| Existing Llama + concise prompt | 19 | 4 | 0 | 2 |
| Qwen + coverage policy + vector retrieval | 18 | 3 | 2 | 2 |
| Qwen + coverage policy + hybrid retrieval | 20 | 2 | 1 | 2 |

Complete means all requested content is covered without a material factual defect; citation placement is assessed separately. Two of the 25 questions intentionally have no answer in the papers. Ratings are assistant source reviews, not independent human labels.

## What changed

- **Fixed by hybrid — q12:** The vector answer begins with the correct 30% relative gain, then invents comparisons by confusing table rows and columns. The hybrid answer gives the correct 30.0% improvement without those extra claims.
- **Fixed by hybrid — q23:** Vector repeats TMI ten times and omits G-CARL. Hybrid names both papers and correctly separates auditable task models from evidence-grounded medical explanations.
- **Still wrong — q20:** Both configurations use identical evidence and the same reused answer. It gives the right direction of clinician preference, but assigns 136:85 and 106:64 to specific baselines without support, then assigns them to dimensions. The PDF's Figure 4 distinguishes dimensions and comparison groups. Correct numbers are insufficient when their meaning changes.
- **Still incomplete/imprecise — q01:** Explains the ordering and stopping rule, but describes partial specialist reasoning as the computation of reservation prices themselves.
- **Presentation gap — q21:** Both identify the correct two paper IDs and explain their relevance, but omit recognizable titles. Under the existing name-the-papers criterion this is Partial. If unique IDs count as sufficient identification, both Qwen complete counts increase by one; the hybrid improvement stays +2 and the factual-error conclusion is unchanged.
- **Key success retained — q08:** Correct 0.226 versus 0.126, with a noncausal caveat about stronger systems. This question has identical vector/hybrid evidence, so hybrid deserves no credit for the improvement over the older Llama response.
- **Citation caveat:** Structured output eliminates invalid citation IDs, not wrong citation placement. q08 and q19 still attach some correct numbers to the wrong passage; hybrid q07 also misplaces citations. The mechanical checker cannot verify semantic attribution.
- **Next focused experiment:** Keep hybrid as the retrieval candidate, but make the answer stop once all requested parts are covered. Require one explicit comparison per requested metric and forbid unrelated table statistics and repeated claims. Test that policy on these failures before another full run; then evaluate the 15 held-out questions. Do not add corrective RAG yet.

## Comparison controls and limits

- Hybrid combines BM25 keyword search and vector search using reciprocal-rank fusion, then the same cross-encoder reranker. Both use up to five passages, fetch 60 candidates, per-paper cap 4, score gap 8, and grouped paper context. Hybrid uses fused rank where vector uses dense rank for final presentation; this tests the whole hybrid configuration, including ordering.
- Both Qwen variants use qwen3.5:9b, the same coverage policy and citation-only JSON schema, temperature 0, seed 42, thinking disabled, context 12,288 and output limit 1,800. No repair or judge calls.
- All 25 vector answers were freshly generated. Hybrid generated 16 changed contexts and reused nine answers whose question, context, passage IDs, source list, model and settings were identical. The combined file records provenance per answer. This avoids redundant generation; it is not an independent repeatability trial.
- The earlier Llama baseline used a different prompt, model, context size and output limit. That comparison evaluates complete configurations, not the isolated effect of changing models.
- Both retrieve the relevant papers for all 23 answerable questions. Literal evidence-phrase hits rise from 23/28 to 25/28, but q12 already had the 30% result and q19 already had all three dimensions. These phrase hits do not establish new answerable content.
- The preselected adoption threshold is at least two net additional complete answers without new major factual errors or incorrect answers to unsupported questions. This is a practical threshold, not statistical significance. The 15-question holdout has not been run in this comparison.
- 140 repository tests passed. Default notebook settings remain unchanged pending the recommendation above.

## Question-by-question review

| Question | Vector | Hybrid | What the hybrid answer gets right or wrong |
|---|---|---|---|
| q01 | Partial | Partial | Includes value/cost, reservation-price order, stopping and selection, but incorrectly describes running specialist reasoning as how reservation prices themselves are computed. Repetitive ten-claim answer. |
| q02 | Complete | Complete | Explains centralized Router versus decentralized Bidder and global welfare versus local surplus, without turning high costs into automatic refusal. |
| q03 | Complete | Complete | Names the same three correct evaluation domains; no completeness change. |
| q04 | Complete | Complete | Correctly states that noisy competing estimates can benefit the strategic specialist at others' expense. |
| q05 | Complete | Complete | Correct ten repositories and ten algorithm families; no completeness change. |
| q06 | Complete | Complete | Correct four-hour agent budget on one B300 and twelve-hour clean replay. |
| q07 | Complete | Complete | Preserves all three correct normalized anchors, but repeats the explanation. |
| q08 | Complete | Complete | Correct 0.226 versus 0.126 and the stronger-systems/noncausal caveat; extra group shares are correct. |
| q09 | Complete | Complete | Correct objective hierarchy and execution-control-flow models; still repeats definitions. |
| q10 | Complete | Complete | Correct 0.974 agreement with task groupings, without confusing this with execution or task success. |
| q11 | Complete | Complete | Correct 74.9% of observed execution steps. |
| q12 | Major issue | Complete | Correct 30.0% held-out accuracy improvement, without the vector run's incorrect table comparisons. |
| q13 | Complete | Complete | Correct 92 commuters and 454 respondent-scenario observations. |
| q14 | Complete | Complete | Correct cycling sensitivity and increased transit use under snow; extra descriptive observations are supported. |
| q15 | Complete | Complete | Correct 69.6% random forest versus 69.9% text-only zero-shot LLM, without asserting significance; much shorter than vector answer. |
| q16 | Complete | Complete | Correct best observed vision-based five-class accuracy of 71.5%. |
| q17 | Complete | Complete | Correct patient-facing, accurate and accessible multimodal report interpretation grounded in the user query and dialogue history. |
| q18 | Complete | Complete | Preserves multi-source atomic claim verification and instance-specific weighted checklists; now repeats the explanation across nine claims. |
| q19 | Complete | Complete | Correct MMedReport and three dimensions, with supported extra evaluation details. |
| q20 | Major issue | Major issue | Correct preference direction, but assigns 136:85 to SFT and 106:64 to MLLM-as-a-Judge, then also assigns the same numbers to evaluation dimensions. The source passage does not support that baseline-specific mapping; PDF Figure 4 presents these as metric-specific comparisons. Repeats the result ten times. |
| q21 | Partial | Partial | Correctly identifies both papers by arXiv ID and explains their relevance, but still omits recognizable paper names required by the presentation checklist. |
| q22 | Complete | Complete | Names both Pandora and AI4AI-Bench and correctly explains benchmark tasks and estimation/compute costs. |
| q23 | Partial | Complete | Names both TMI and G-CARL and assigns auditable task models and evidence-grounded medical explanations to the correct paper; fixes vector omission/repetition. |
| q24 | Correct abstention | Correct abstention | Does not invent a KPMG forecast. |
| q25 | Correct abstention | Correct abstention | Does not invent a NVIDIA stock price. |

## Saved answers and citation concerns

Details below preserve the raw generated answers; no answers were edited to improve their scores.

<details><summary>q01: How does Pandora's Router decide when an expensive specialist estimate is worth paying for?</summary>

**Vector — Partial:** Includes value/cost, reservation-price order, stopping and selection, but incorrectly describes running specialist reasoning as how reservation prices themselves are computed. Repetitive ten-claim answer.

- Pandora's Router decides that an expensive specialist estimate is worth paying for by computing reservation prices that quantify the expected value-of-information for querying a more accurate estimate, minus total estimation cost. [2][4]
- The system stops querying expensive estimates when no remaining query is worth its cost, effectively choosing the maximizer of the observed value estimate. [1]
- The router computes reservation prices by running a specialist for a few reasoning tokens to check if it is on the right track, thereby quantifying the expected value-of-information before committing to a costly and accurate estimate. [1]
- The decision mechanism balances cheap screening and costly reasoning by deciding which expensive estimates are worth computing based on the cost-adjusted reward, which may combine graded output quality with production costs such as inference cost or API fees. [1]
- Under a Gaussian signal model, the reservation-price-based policy has a closed form that is straightforward to compute, allowing the system to efficiently determine when an expensive inspection is justified. [4]
- The centralized policy inspects specialists in reservation price order, stopping the process when no remaining query is worth its cost, while selecting the maximizer of the observed value estimate. [1]
- The unified principle supporting allocation by centralized routing involves computing reservation prices that quantify the expected value-of-information for querying a more costly and accurate value estimate, such as by running the specialist for a few reasoning tokens. [1]
- The system receives a concrete prompt, generates cheap noisy value estimates from multiple specialists, and then computes reservation prices to decide whether to proceed with expensive estimates. [1]
- Pandora's Router formalizes model routing with costly value estimation as an instance of Pandora's Box with non-obligatory inspection, yielding a reservation-price-based policy that determines when expensive estimates are worth paying for. [4]
- The reward function used in the decision process may combine the graded quality of the output with the cost of producing it, which could include inference cost or API fees for tools called. [1]

**Hybrid — Partial:** Includes value/cost, reservation-price order, stopping and selection, but incorrectly describes running specialist reasoning as how reservation prices themselves are computed. Repetitive ten-claim answer.

- Pandora's Router decides that an expensive specialist estimate is worth paying for by computing reservation prices that quantify the expected value-of-information for querying a more accurate estimate, minus total estimation cost. [2][4]
- The system stops querying expensive estimates when no remaining query is worth its cost, effectively choosing the maximizer of the observed value estimate. [1]
- The router computes reservation prices by running a specialist for a few reasoning tokens to check if it is on the right track, thereby quantifying the expected value-of-information before committing to a costly and accurate estimate. [1]
- The decision mechanism balances cheap screening and costly reasoning by deciding which expensive estimates are worth computing based on the cost-adjusted reward, which may combine graded output quality with production costs such as inference cost or API fees. [1]
- Under a Gaussian signal model, the reservation-price-based policy has a closed form that is straightforward to compute, allowing the system to efficiently determine when an expensive inspection is justified. [4]
- The centralized policy inspects specialists in reservation price order, stopping the process when no remaining query is worth its cost, while selecting the maximizer of the observed value estimate. [1]
- The unified principle supporting allocation by centralized routing involves computing reservation prices that quantify the expected value-of-information for querying a more costly and accurate value estimate, such as by running the specialist for a few reasoning tokens. [1]
- The system receives a concrete prompt, generates cheap noisy value estimates from multiple specialists, and then computes reservation prices to decide whether to proceed with expensive estimates. [1]
- Pandora's Router formalizes model routing with costly value estimation as an instance of Pandora's Box with non-obligatory inspection, yielding a reservation-price-based policy that determines when expensive estimates are worth paying for. [4]
- The reward function used in the decision process may combine the graded quality of the output with the cost of producing it, which could include inference cost or API fees for tools called. [1]

Vector citation concern: The first claim cites [2][4] rather than the explicit mechanism in [1].

Hybrid citation concern: The first claim cites [2][4] rather than the explicit mechanism in [1].

</details>

<details><summary>q02: What is the difference between Pandora's Router and Pandora's Bidder?</summary>

**Vector — Complete:** Explains centralized Router versus decentralized Bidder and global welfare versus local surplus, without turning high costs into automatic refusal.

- Pandora's Router operates as a centralized policy that computes reservation prices to decide when querying a more accurate value estimate is worthwhile, interpolating between baselines by querying only when the value of information exceeds the cost. [2][3]
- Pandora's Bidder operates as a decentralized policy where the high bidder is selected in an auction, and each specialist refines its own value estimate only when the market price makes inspection worthwhile. [1][3]
- In the decentralized setting of Pandora's Bidder, maximizing local surplus does not always lead to better overall allocative efficiency, particularly when prices are inaccurate or queries are overpriced. [3]
- The tension between protecting a bidder's surplus by declining overpriced queries and assigning inputs to less capable specialists is absent in the centralized setting of Pandora's Router, where the router internalizes all costs and optimizes global welfare directly. [3]
- Pandora's Bidder corresponds to a single stage of an ascending-price mechanism with costly preference elicitation, isolating core components of decentralized prompt allocation under private value estimation with profit incentives. [4]
- Both Pandora's Router and Pandora's Bidder are derived from the same principle of balancing cheap screening and costly reasoning for routing, supporting allocation by either centralized routing or decentralized auctions. [1]

**Hybrid — Complete:** Explains centralized Router versus decentralized Bidder and global welfare versus local surplus, without turning high costs into automatic refusal.

- Pandora's Router operates as a centralized policy that computes reservation prices to decide when querying a more accurate value estimate is worthwhile, interpolating between baselines by querying only when the value of information exceeds the cost. [2][3]
- Pandora's Bidder operates as a decentralized policy where the high bidder is selected in an auction, and each specialist refines its own value estimate only when the market price makes inspection worthwhile. [1][3]
- In the decentralized setting of Pandora's Bidder, maximizing local surplus does not always lead to better overall allocative efficiency, particularly when prices are inaccurate or queries are overpriced. [3]
- The tension between protecting a bidder's surplus by declining overpriced queries and assigning inputs to less capable specialists is absent in the centralized setting of Pandora's Router, where the router internalizes all costs and optimizes global welfare directly. [3]
- Pandora's Bidder corresponds to a single stage of an ascending-price mechanism with costly preference elicitation, isolating core components of decentralized prompt allocation under private value estimation with profit incentives. [4]
- Both Pandora's Router and Pandora's Bidder are derived from the same principle of balancing cheap screening and costly reasoning for routing, supporting allocation by either centralized routing or decentralized auctions. [1]

</details>

<details><summary>q03: In which three domains was the costly-value-estimation routing policy evaluated?</summary>

**Vector — Complete:** Names all three evaluation domains.

- The costly-value-estimation routing policy was evaluated in three domains: a multi-LLM routing benchmark (EmbedLLM), retrieval-augmented specialists for factoid QA, and inference-time computation for mathematical reasoning. [2][3]

**Hybrid — Complete:** Names the same three correct evaluation domains; no completeness change.

- The costly-value-estimation routing policy was evaluated across three domains: a multi-LLM routing benchmark (EmbedLLM), retrieval-augmented specialists for factoid QA, and inference-time computation for mathematical reasoning. [2][3]

</details>

<details><summary>q04: What limitation appears when competing specialists' estimates are noisy?</summary>

**Vector — Complete:** Correctly states that noisy competing estimates can benefit the strategic specialist at others' expense.

- When competing estimates are noisy, value-of-information reasoning can increase the strategic specialist's utility at the expense of others. [2]

**Hybrid — Complete:** Correctly states that noisy competing estimates can benefit the strategic specialist at others' expense.

- When competing estimates are noisy, value-of-information reasoning can increase the strategic specialist's utility at the expense of others. [2]

Vector citation concern: The statement is supported by passage [1], but cites [2].

Hybrid citation concern: The statement is supported by passage [1], but cites [2].

</details>

<details><summary>q05: How many frozen repositories and training algorithm families are in AI4AI-Bench?</summary>

**Vector — Complete:** Correct ten repositories and ten algorithm families.

- AI4AI-Bench contains ten frozen research repositories that collectively cover ten distinct families of training algorithms. [2][3]

**Hybrid — Complete:** Correct ten repositories and ten algorithm families; no completeness change.

- AI4AI-Bench contains ten frozen research repositories that collectively cover ten distinct families of training algorithms. [2][3]

</details>

<details><summary>q06: What is the AI4AI-Bench time budget for an agent and for the replay?</summary>

**Vector — Complete:** Correct four-hour agent budget on one B300 and twelve-hour clean replay.

- The agent is given a time budget of four hours on one B300 GPU to rewrite the training algorithm. [2][4]
- The submitted repository is then run from scratch under a fixed budget of twelve hours. [2][4]

**Hybrid — Complete:** Correct four-hour agent budget on one B300 and twelve-hour clean replay.

- The agent is given a time budget of four hours on one B300 GPU to rewrite the training algorithm. [2][4]
- The submitted repository is then run from scratch under a fixed budget of twelve hours. [2][4]

</details>

<details><summary>q07: How are the incommensurable AI4AI-Bench task metrics normalized?</summary>

**Vector — Complete:** Correct normalized anchors: 0 uninformative, 0.1 shipped algorithm, 1.0 optimum.

- The incommensurable AI4AI-Bench task metrics are normalized onto a single scale where 0 represents an uninformative model, 0.1 represents the algorithm the repository ships, and 1.0 represents the task optimum. [2][5]
- The ten metrics are incommensurable and cannot be averaged as they stand; every per-task number is reported in its own units against its own baseline. [1]

**Hybrid — Complete:** Preserves all three correct normalized anchors, but repeats the explanation.

- The incommensurable AI4AI-Bench task metrics are normalized onto a single scale where 0 represents an uninformative model, 0.1 represents the algorithm shipped by the repository, and 1.0 represents the task optimum. [2]
- Every per-task number in the AI4AI-Bench paper is reported in its own units against its own baseline because the ten metrics are incommensurable and cannot be averaged as they stand. [3]
- Because the ten metrics are incommensurable, every task is mapped onto one scale on which 0 is an uninformative model, 0.1 is the algorithm the repository ships, and 1.0 is the task optimum. [4]

Hybrid citation concern: The first anchors claim cites unrelated [2]; its repetition correctly cites [4]. The statement about incommensurable metrics cites [3] rather than explicit [1].

</details>

<details><summary>q08: How do submissions that change the learning algorithm compare with those that do not?</summary>

**Vector — Complete:** Correct 0.226 versus 0.126 and the stronger-systems/noncausal caveat; extra group shares are correct.

- Submissions that change the learning algorithm achieve a mean score of 0.226, whereas submissions that do not change the learning procedure achieve a mean score of 0.126. [2][3]
- The gap between submissions that reach the learning algorithm layer and those that do not represents the difference between the two groups rather than an effect caused by reaching that layer, as the systems that reach it more often are stronger to begin with. [3]
- Submissions that change the learning algorithm represent 46.4% of the submissions that change anything, while those that do not change the learning procedure but modify other aspects like budgets or hyperparameters represent 53.6%. [2]
- Submissions that change the learning algorithm focus on the objective, supervision signal, learning rule, and data, whereas submissions that do not change the learning procedure move budgets, checkpointing, hyperparameters, and capacity. [2]
- Submissions that reach the learning algorithm layer are where the distance to a better algorithm is closed, while most submissions never go there. [3]

**Hybrid — Complete:** Correct 0.226 versus 0.126 and the stronger-systems/noncausal caveat; extra group shares are correct.

- Submissions that change the learning algorithm achieve a mean score of 0.226, whereas submissions that do not change the learning procedure achieve a mean score of 0.126. [2][3]
- The gap between submissions that reach the learning algorithm layer and those that do not represents the difference between the two groups rather than an effect caused by reaching that layer, as the systems that reach it more often are stronger to begin with. [3]
- Submissions that change the learning algorithm represent 46.4% of the submissions that change anything, while those that do not change the learning procedure but modify other aspects like budgets or hyperparameters represent 53.6%. [2]
- Submissions that change the learning algorithm focus on the objective, supervision signal, learning rule, and data, whereas submissions that do not change the learning procedure move budgets, checkpointing, hyperparameters, and capacity. [2]
- Submissions that reach the learning algorithm layer are where the distance to a better algorithm is closed, while most submissions never go there. [3]

Vector citation concern: 46.4% and 53.6% cite [2], but are supported in [3].

Hybrid citation concern: 46.4% and 53.6% cite [2], but are supported in [3].

</details>

<details><summary>q09: What two kinds of task structure does TMI recover from computer-use traces?</summary>

**Vector — Complete:** Correct hierarchical objectives and execution control flow; repeats their definitions.

- TMI recovers task models that unify two complementary axes: an objective model representing a hierarchical decomposition of the task's goal into subobjectives, and a procedure model representing a trace-grounded composition of control-flow operators specifying how execution is organized through sequencing and iteration. [2][3]
- The task model construction stage builds complementary objective and procedure models for each task, where every node in the resulting tree structure pairs an objective with a control-flow operator over its children. [1]
- The procedure model is defined as a trace-grounded composition of control-flow operators that specify how the execution of a task is organized through sequencing and iteration. [3]
- The objective model is defined as a hierarchical decomposition of the objective of a task into the subobjectives that the user pursues. [3]

**Hybrid — Complete:** Correct objective hierarchy and execution-control-flow models; still repeats definitions.

- TMI recovers task models that unify two complementary axes: an objective model representing a hierarchical decomposition of the task's goal into subobjectives, and a procedure model representing a trace-grounded composition of control-flow operators specifying how execution is organized through sequencing and iteration. [2][3]
- The objective model for each task is defined as a hierarchical decomposition of the objective into the subobjectives the user pursues. [3]
- The procedure model for each task is defined as a trace-grounded composition of control-flow operators that specify how the execution of the task is organized through sequencing and iteration. [3]
- The resulting task model is structured as a tree in which every node pairs an objective with a control-flow operator over its children. [3]

</details>

<details><summary>q10: How well does task model induction group interleaved tasks?</summary>

**Vector — Complete:** Correct 0.974 agreement with task groupings, without confusing this with execution or task success.

- The method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings. [2][3]

**Hybrid — Complete:** Correct 0.974 agreement with task groupings, without confusing this with execution or task success.

- The method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings. [2][3]

</details>

<details><summary>q11: How much of observed execution does TMI reconstruct?</summary>

**Vector — Complete:** Correct 74.9% of observed execution steps.

- TMI reconstructs 74.9% of the observed execution steps. [2][4]

**Hybrid — Complete:** Correct 74.9% of observed execution steps.

- TMI reconstructs 74.9% of the observed execution steps. [2][4]

</details>

<details><summary>q12: What was the held-out task accuracy gain from skills derived from TMI?</summary>

**Vector — Major issue:** Starts correctly with 14.29% to 18.57% (30% relative gain), then mislabels multiple other table cells as held-out accuracy and incorrectly calls Human curated the strongest baseline. This is a factual regression despite covering the requested number.

- The held-out task accuracy gain from skills derived from TMI was an increase from 14.29% to 18.57%, representing a 30% relative accuracy gain over the strongest baseline. [1]
- The held-out task accuracy for skills derived from TMI was 18.57%, compared to 10.00% for the strongest baseline (Human curated). [1]
- The held-out task accuracy for skills derived from TMI was 18.57%, compared to 56.59% for Raw demonstration and 63.80% for Workflow summary. [1]
- The held-out task accuracy for skills derived from TMI was 18.57%, compared to 90.50% for No skill. [1]
- The held-out task accuracy for skills derived from TMI was 18.57%, compared to N/A for No skill. [1]
- The held-out task accuracy for skills derived from TMI was 18.57%, compared to 93.59% for Human curated. [1]
- The held-out task accuracy for skills derived from TMI was 18.57%, compared to 59.20% for No skill. [1]
- The held-out task accuracy for skills derived from TMI was 18.57%, compared to 63.80% for Human curated. [1]
- The held-out task accuracy for skills derived from TMI was 18.57%, compared to 90.50% for No skill. [1]
- The held-out task accuracy for skills derived from TMI was 18.57%, compared to 56.59% for Raw demonstration. [1]

**Hybrid — Complete:** Correct 30.0% held-out accuracy improvement, without the vector run's incorrect table comparisons.

- Skills derived from TMI's task models improve held-out task accuracy by 30.0% over the strongest baseline. [2][5]

Vector citation concern: All erroneous claims cite [1]; numeric-presence checks miss wrong metric/row attribution.

</details>

<details><summary>q13: How many commuters and respondent-scenario observations did the weather travel survey collect?</summary>

**Vector — Complete:** Correct 92 commuters and 454 observations; six excluded records are supported.

- The weather travel survey collected data from 92 student commuters. [2][4]
- The survey yielded a total of 454 valid respondent-scenario observations after excluding six records with missing or unparseable mode choices. [3][4]

**Hybrid — Complete:** Correct 92 commuters and 454 respondent-scenario observations.

- The weather travel survey collected data from 92 student commuters. [2][3]
- The weather travel survey collected 454 respondent-scenario observations. [1][3]

</details>

<details><summary>q14: Which travel modes responded most clearly to adverse weather and snow?</summary>

**Vector — Complete:** Correct cycling sensitivity and shift toward transit in snow, with supported descriptive context.

- Cycling was particularly sensitive to adverse weather, while public transit use increased substantially under the Snowy scenario. [2][3]
- Snowy and Rainy scenarios are generally easier to predict because they produce stronger and more consistent shifts toward public transit and away from cycling. [2][3]
- The MNL estimates were consistent with these descriptive patterns, showing lower relative utility for cycling-related modes and higher relative utility for transit and driving under adverse scenarios. [2][3]

**Hybrid — Complete:** Correct cycling sensitivity and increased transit use under snow; extra descriptive observations are supported.

- Cycling was particularly sensitive to adverse weather, while public transit use increased substantially under the Snowy scenario. [2][4]
- The MNL estimates showed lower relative utility for cycling-related modes and higher relative utility for transit and driving under adverse scenarios. [1]
- Snowy and Rainy scenarios are generally easier to predict because they produce stronger and more consistent shifts toward public transit and away from cycling. [3]

</details>

<details><summary>q15: How did random forest and the best text-only zero-shot LLM compare on five-class travel prediction?</summary>

**Vector — Complete:** Correct 69.6% random forest versus 69.9% text-only zero-shot LLM, and does not claim statistical superiority. Adds unnecessary but supported vision/context details.

- Random forest achieved the highest trained-model accuracy, while the best zero-shot LLM produced a comparable five-class point estimate without task-specific fitting to the survey records. [2][3]
- Random forest achieved 69.6% five-class accuracy, while the best text-only zero-shot LLM reached 69.9% without task-specific fitting. [4]
- The best vision-based configuration achieved a five-class accuracy point estimate of 71.5%, compared with 69.9% for the highest text-only zero-shot LLM result and 69.6% for random forest. [1]
- The conventional and LLM-based models therefore play complementary roles, with the MNL providing behavioral interpretation, while logistic regression and random forest offering data-trained predictive benchmarks. [2]
- The comparison of MNL, random forest, and LLM-based predictors highlighted their complementary roles, where random forest offered a strong trained predictive benchmark. [3]
- These small differences should be interpreted as descriptive point-estimate differences rather than evidence of statistical superiority. [1]

**Hybrid — Complete:** Correct 69.6% random forest versus 69.9% text-only zero-shot LLM, without asserting significance; much shorter than vector answer.

- Random forest achieved the highest trained-model accuracy with a five-class point estimate of 69.6%, while the best text-only zero-shot LLM produced a comparable five-class point estimate of 69.9% without task-specific fitting to the survey records. [2][3]

</details>

<details><summary>q16: What was the best vision-based travel mode prediction accuracy?</summary>

**Vector — Complete:** Correct best observed vision-based five-class accuracy of 71.5%.

- The best vision-based configuration achieved a five-class accuracy point estimate of 71.5%. [2][3]

**Hybrid — Complete:** Correct best observed vision-based five-class accuracy of 71.5%.

- The best vision-based configuration achieved a five-class accuracy point estimate of 71.5%. [2][3]

</details>

<details><summary>q17: What is the patient-oriented medical report interpretation task introduced with G-CARL?</summary>

**Vector — Complete:** Correct patient-facing, accurate and accessible multimodal report interpretation grounded in the user query and dialogue history.

- Patient-oriented Medical Report Interpretation (PMRI) is a novel open-ended multimodal generation task that requires models to explain medical reports in accurate and accessible language based on a user's query and dialogue history. [2][3]
- PMRI addresses the need for both evidence-grounded medical factuality and context-dependent patient communication, which existing medical vision-language tasks do not adequately capture. [3]
- PMRI is designed to bridge the gap where medical reports present abnormal values and descriptive findings without explaining their implications for non-expert readers, particularly in online healthcare scenarios. [4]
- Beyond simply recognizing report findings, PMRI must align professional medical knowledge with patient-centered communication by transforming report evidence into accessible explanations, addressing user-specific concerns, and offering clinically cautious guidance. [4]
- Unlike the short, deterministic outputs of medical VQA or conventional report generation tasks, PMRI produces long-form, patient-facing explanations grounded in multimodal reports and extended patient-doctor dialogue. [4]

**Hybrid — Complete:** Correct patient-facing, accurate and accessible multimodal report interpretation grounded in the user query and dialogue history.

- Patient-oriented Medical Report Interpretation (PMRI) is a novel open-ended multimodal generation task that requires models to explain medical reports in accurate and accessible language based on a user's query and dialogue history. [2][3]
- PMRI addresses the need for both evidence-grounded medical factuality and context-dependent patient communication, which existing medical vision-language tasks do not adequately capture. [3]
- PMRI is designed to bridge the gap where medical reports present abnormal values and descriptive findings without explaining their implications for non-expert readers, particularly in online healthcare scenarios. [4]
- Beyond simply recognizing report findings, PMRI must align professional medical knowledge with patient-centered communication by transforming report evidence into accessible explanations, addressing user-specific concerns, and offering clinically cautious guidance. [4]
- Unlike the short, deterministic outputs of medical VQA or conventional report generation tasks, PMRI produces long-form, patient-facing explanations grounded in multimodal reports and extended patient-doctor dialogue. [4]

</details>

<details><summary>q18: How does G-CARL supervise factuality and response coverage?</summary>

**Vector — Complete:** Covers multi-source atomic claim verification and instance-specific weighted checklists, including generation/refinement and coverage rewards. Repeats both mechanisms.

- G-CARL supervises factuality through retrieval-grounded claim verification that decomposes reward supervision by verifying atomic claims against evidence retrieved from a multi-source medical datastore. [2][3]
- G-CARL supervises response coverage by constructing instance-specific weighted checklists through MLLM generation followed by clinician refinement to explicitly supervise demand satisfaction and expression quality. [1][3]
- The retrieval-grounded branch supervises medical factuality by verifying report-grounded medical claims against evidence retrieved from a multi-source medical datastore to produce the factuality reward Rfact. [4]
- The case-specific checklist branch optimizes demand satisfaction and expression quality by constructing a weighted checklist through MLLM generation followed by clinician-guided refinement, and evaluating the checklist coverage of the generated response to produce the checklist reward Rcheck. [4]
- Each checklist item is assigned an automatically generated weight, allowing the reward to emphasize the aspects most relevant to the current report and user question. [1]

**Hybrid — Complete:** Preserves multi-source atomic claim verification and instance-specific weighted checklists; now repeats the explanation across nine claims.

- G-CARL supervises medical factuality by combining multi-source retrieval for atomic claim verification with a retrieval-grounded claim reward that provides fine-grained supervision through verifying report-grounded medical claims against evidence retrieved from a multi-source medical datastore. [2][4]
- G-CARL supervises response coverage by constructing instance-specific weighted checklists through MLLM generation followed by clinician refinement, which explicitly supervises demand satisfaction and expression quality without relying on a single reference response. [1][2][4]
- The case-specific checklist branch optimizes demand satisfaction and expression quality by constructing a weighted checklist through MLLM generation followed by clinician-guided refinement, and evaluating the checklist coverage of the generated response to produce the checklist reward. [4]
- The reward provides localized supervision for sparse factual errors and discourages unsupported elaboration, while the case-specific checklist score provides transparent supervision for whether the response addresses the user's concern appropriately. [1]
- G-CARL combines multi-source retrieval for atomic claim verification with context-aware, instance-specific weighted checklists for response coverage, providing structured supervision for factuality, user-demand satisfaction, and expression quality without constraining response diversity. [2]
- Given uploaded medical report images, dialogue history, and a user query, the policy model first generates candidate responses, after which the retrieval-grounded branch supervises medical factuality by verifying report-grounded medical claims against evidence retrieved from a multi-source medical datastore. [4]
- In parallel, the case-specific checklist branch optimizes demand satisfaction and expression quality by constructing a weighted checklist through MLLM generation followed by clinician-guided refinement, and evaluating the checklist coverage of the generated response. [4]
- Together with the format reward, these components are integrated into a multi-objective optimization task that optimizes the policy model with three reward signals. [4]
- G-CARL is a reinforcement learning framework that decomposes reward supervision through retrieval-grounded claim verification and case-specific checklist guidance. [3]

</details>

<details><summary>q19: What benchmark and evaluation dimensions accompany G-CARL?</summary>

**Vector — Complete:** Correct MMedReport and all three evaluation dimensions; adds supported objective metrics and CMB evaluation.

- G-CARL is evaluated on the MMedReport benchmark, which contains 2,450 instances of real-world multimodal patient-oriented medical report interpretation tasks. [2][3]
- The evaluation on MMedReport utilizes three clinician-defined subjective dimensions: Medical Accuracy, Demand Satisfaction, and Expression Quality. [1]
- In addition to holistic subjective assessment, G-CARL is evaluated using two objective metrics: claim-level Precision and checklist-level Recall. [2]
- G-CARL is also evaluated on an external benchmark called CMB, which assesses medical QA accuracy and the professionalism of open-ended clinical interpretation under an LLM-as-a-judge protocol. [1]
- On the external CMB benchmark, G-CARL is evaluated for QA accuracy and professionalism scores in open-ended generation. [3]

**Hybrid — Complete:** Correct MMedReport and three dimensions, with supported extra evaluation details.

- The G-CARL framework is evaluated using the MMedReport benchmark, which consists of 2,450 instances collected from online healthcare consultations and includes dialogue history, user queries, medical report images, and clinician-verified reference annotations. [2][4]
- In addition to MMedReport, G-CARL is evaluated on the external CMB benchmark, which assesses medical QA accuracy and the professionalism of open-ended clinical interpretation under an LLM-as-a-judge protocol. [1]
- The evaluation dimensions for G-CARL on MMedReport are defined by clinicians and include Medical Accuracy (Accuracy), Demand Satisfaction (Satisfaction), and Expression Quality (Expression). [1]
- Each of the three evaluation dimensions is scored according to a detailed clinician-authored rubric ranging from -2 to 3. [1]
- The evaluation protocol for G-CARL involves both subjective and objective metrics, with the subjective component focusing on holistic assessment across the three clinician-defined dimensions. [4]
- A blind pairwise human preference study was conducted on 250 held-out cases where three professional clinicians compared anonymized responses from G-CARL and baselines across the three evaluation dimensions and overall preference, using majority voting for the final decision. [2]
- Additionally, 50 participants without medical training were recruited to evaluate the comprehensibility of the generated interpretations as part of the evaluation process. [2]

Vector citation concern: The 2,450-instance claim cites [2][3] but is supported by [1].

Hybrid citation concern: The 2,450-instance claim cites [2][4] instead of supporting [1]. The subjective/objective protocol claim cites [4] rather than [1].

</details>

<details><summary>q20: What did clinician pairwise preferences find about G-CARL?</summary>

**Vector — Major issue:** Correct preference direction, but assigns 136:85 to SFT and 106:64 to MLLM-as-a-Judge, then also assigns the same numbers to evaluation dimensions. The source passage does not support that baseline-specific mapping; PDF Figure 4 presents these as metric-specific comparisons. Repeats the result ten times.

- Clinician pairwise preferences found that G-CARL is consistently preferred over both SFT and MLLM-as-a-Judge baselines, particularly in medical accuracy and demand satisfaction. [2][1]
- In the human preference study involving three professional clinicians comparing anonymized responses from G-CARL and baselines across three evaluation dimensions, majority voting was used for the final decision. [1]
- The clinician pairwise preference study included 250 held-out cases where G-CARL achieved preference margins of 136:85 over SFT and 106:64 over MLLM-as-a-Judge. [1]
- Clinician pairwise preferences indicated that G-CARL is preferred over baselines in medical accuracy and demand satisfaction, with specific preference margins of 136:85 and 106:64 respectively. [1]
- The clinician pairwise preference evaluation showed that G-CARL is consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction. [1]
- In the clinician pairwise preference study, G-CARL was consistently preferred over both SFT and MLLM-as-a-Judge baselines, particularly in medical accuracy and demand satisfaction. [1]
- Clinician pairwise preferences found that G-CARL is consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction. [1]
- The clinician pairwise preference study found that G-CARL is consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction. [1]
- Clinician pairwise preferences found that G-CARL is consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction. [1]
- In the clinician pairwise preference study, G-CARL was consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction. [1]

**Hybrid — Major issue:** Correct preference direction, but assigns 136:85 to SFT and 106:64 to MLLM-as-a-Judge, then also assigns the same numbers to evaluation dimensions. The source passage does not support that baseline-specific mapping; PDF Figure 4 presents these as metric-specific comparisons. Repeats the result ten times.

- Clinician pairwise preferences found that G-CARL is consistently preferred over both SFT and MLLM-as-a-Judge baselines, particularly in medical accuracy and demand satisfaction. [2][1]
- In the human preference study involving three professional clinicians comparing anonymized responses from G-CARL and baselines across three evaluation dimensions, majority voting was used for the final decision. [1]
- The clinician pairwise preference study included 250 held-out cases where G-CARL achieved preference margins of 136:85 over SFT and 106:64 over MLLM-as-a-Judge. [1]
- Clinician pairwise preferences indicated that G-CARL is preferred over baselines in medical accuracy and demand satisfaction, with specific preference margins of 136:85 and 106:64 respectively. [1]
- The clinician pairwise preference evaluation showed that G-CARL is consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction. [1]
- In the clinician pairwise preference study, G-CARL was consistently preferred over both SFT and MLLM-as-a-Judge baselines, particularly in medical accuracy and demand satisfaction. [1]
- Clinician pairwise preferences found that G-CARL is consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction. [1]
- The clinician pairwise preference study found that G-CARL is consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction. [1]
- Clinician pairwise preferences found that G-CARL is consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction. [1]
- In the clinician pairwise preference study, G-CARL was consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction. [1]

Vector citation concern: Existing numbers and a valid source ID do not validate which comparison each number describes.

Hybrid citation concern: Existing numbers and a valid source ID do not validate which comparison each number describes.

</details>

<details><summary>q21: Which papers address auditable AI workflows derived from user activity or survey responses?</summary>

**Vector — Partial:** Correctly associates the two arXiv paper IDs with activity-derived task models and survey workflows, but gives only numeric IDs rather than the requested recognizable paper names.

- Paper ID 2608.20320 addresses auditable AI workflows derived from survey responses by defining agents as software components that produce structured and auditable outputs. [2][3]
- Paper ID 2608.20319 addresses auditable AI workflows derived from user activity by deriving symbolic, auditable, and reusable models of how everyday work is done from naturalistic computer-use traces. [1]

**Hybrid — Partial:** Correctly identifies both papers by arXiv ID and explains their relevance, but still omits recognizable paper names required by the presentation checklist.

- Paper ID 2608.20320 addresses auditable AI workflows derived from survey responses by defining an agent as a goal-directed and researcher-supervised software component that produces structured and auditable outputs. [2][3]
- Paper ID 2608.20319 addresses auditable AI workflows derived from user activity by introducing TASKMODELINDUCTION (TMI), which derives symbolic, auditable, and reusable models of how everyday work is done from naturalistic computer-use traces. [1]

Vector citation concern: The numeric warnings concern paper IDs in metadata, not invented numerical results.

Hybrid citation concern: Paper-ID numeric warnings are metadata false positives, not fabricated result numbers.

</details>

<details><summary>q22: Which papers evaluate AI agents or model specialists using benchmark tasks and costs?</summary>

**Vector — Complete:** Identifies AI4AI-Bench and Pandora, and explains benchmark tasks and compute/estimation costs.

- The paper AI4AI-Bench evaluates AI agents using benchmark tasks that involve rewriting training algorithms within frozen research repositories, where the agent is given 4 hours on one B300 to rewrite the algorithm and its code is rerun from scratch for up to 12 hours against a fixed evaluator. [2][3]
- The paper Pandora's AI Model Routing Box evaluates model specialists using benchmark tasks that involve routing queries to heterogeneous model families with varying costs and capabilities, where the routing decision involves a cost-accuracy tradeoff between cheap but noisy estimators and accurate but expensive ones. [5]

**Hybrid — Complete:** Names both Pandora and AI4AI-Bench and correctly explains benchmark tasks and estimation/compute costs.

- Pandora's AI Model Routing Box evaluates model specialists using benchmark tasks and costs by formalizing model routing with costly value estimation as an instance of Pandora's Box, where cheap estimators are fast but noisy while accurate estimators are expensive, and experiments across three domains show that the proposed policies trade off expensive and cheap estimates to achieve strong allocative performance relative to baselines. [2][5]
- AI4AI-Bench evaluates AI agents using benchmark tasks by presenting 10 frozen research repositories spanning 10 training algorithm families, where an agent has 4 hours on one B300 to rewrite the training algorithm and its code is then rerun from scratch for up to 12 hours and scored by a fixed evaluator hidden from the agent. [3]

</details>

<details><summary>q23: Which papers discuss evidence-grounded explanations and auditable task models?</summary>

**Vector — Partial:** Repeats TMI ten times and omits G-CARL despite its evidence being present. Also emits a confusing no-unanswered-parts note.

- The paper titled Inducing Task Models from Computer-Use Traces discusses auditable task models derived from naturalistic computer-use traces. [2][1]
- The paper titled Inducing Task Models from Computer-Use Traces discusses inducing symbolic, auditable, and reusable models of how everyday work is done. [2][1]
- The paper titled Inducing Task Models from Computer-Use Traces discusses inducing task models that are auditable and reusable. [2][1]
- The paper titled Inducing Task Models from Computer-Use Traces discusses inducing task models that are symbolic and auditable. [2][1]
- The paper titled Inducing Task Models from Computer-Use Traces discusses inducing task models that are symbolic and reusable. [2][1]
- The paper titled Inducing Task Models from Computer-Use Traces discusses inducing task models that are auditable and reusable. [2][1]
- The paper titled Inducing Task Models from Computer-Use Traces discusses inducing task models that are symbolic, auditable, and reusable. [2][1]
- The paper titled Inducing Task Models from Computer-Use Traces discusses inducing task models that are symbolic, auditable, and reusable. [2][1]
- The paper titled Inducing Task Models from Computer-Use Traces discusses inducing task models that are symbolic, auditable, and reusable. [2][1]
- The paper titled Inducing Task Models from Computer-Use Traces discusses inducing task models that are symbolic, auditable, and reusable. [2][1]
Not established by the supplied passages: No unanswered parts were identified as all requested information was supported by the passages.

**Hybrid — Complete:** Names both TMI and G-CARL and assigns auditable task models and evidence-grounded medical explanations to the correct paper; fixes vector omission/repetition.

- The paper Inducing Task Models from Computer-Use Traces discusses auditable task models derived from naturalistic computer-use traces. [2][1]
- The paper G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation discusses evidence-grounded explanations that must be grounded in reports and reliable clinical knowledge. [3][5]

</details>

<details><summary>q24: What is KPMG's 2026 revenue forecast?</summary>

**Vector — Correct abstention:** Does not invent a KPMG revenue forecast.

The provided papers do not address this question.

**Hybrid — Correct abstention:** Does not invent a KPMG forecast.

The provided papers do not address this question.

</details>

<details><summary>q25: Which of these papers reports the stock price of NVIDIA on October 14, 2026?</summary>

**Vector — Correct abstention:** Does not invent a historical NVIDIA stock price.

The provided papers do not address this question.

**Hybrid — Correct abstention:** Does not invent a NVIDIA stock price.

The provided papers do not address this question.

</details>



In [4]:
display(Markdown((ROOT / "evaluation/results/optimization_loop_summary.md").read_text()))


# Optimization loop — checkpoint and best observed results

**Decision:** keep the existing tested default. The user requested wrapping up within a few minutes, so the model experiments were stopped before a complete candidate could be tested on all 25 questions. The best full-benchmark result remains **19 complete, 4 partial, 0 major issues and 2 correct abstentions**. Do not treat the examples below as a single improved pipeline or a new benchmark score.

## Results from this loop

| Candidate | Completed questions | Local validation failures | Outcome |
| --- | ---: | ---: | --- |
| llama3.1:8b, basic, quote, repair=False | 8 | 3 | Rejected: quote-copy failures and missing content. |
| llama3.1:8b, coverage, quote, repair=True | 4 | 3 | Stopped as unpromising: three failures remained after repair; high latency. |
| llama3.1:8b, coverage, citation, repair=False | 8 | 0 | Valid citation structure, but Llama still omitted key facts and numerical outcomes. |
| qwen3.5:9b, coverage, citation, repair=False | 5 | 0 | Promising individual answers; insufficient results for adoption. |

These are intentionally different pilot sizes. Local validation counts measure format/ID/quote validity, not factual correctness. No comparative accuracy percentage is inferred from these pilots.

## Best observed examples

These are selected examples from the Qwen pilot, reviewed against saved passages. They are not a full-benchmark result or outputs of a deployed automatic selector.

### q02: What is the difference between Pandora's Router and Pandora's Bidder?

Model/policy: `qwen3.5:9b` / `coverage`. Recorded time: 38.2s.

- Pandora's Router operates as a centralized policy that computes reservation prices to decide when querying a more accurate value estimate is worthwhile, interpolating between baselines by querying only when the value of information exceeds the cost. [2][3]
- Pandora's Bidder operates as a decentralized policy where the high bidder is selected in an auction, and each specialist refines its own value estimate only when the market price makes inspection worthwhile. [1][3]
- In the decentralized setting of Pandora's Bidder, maximizing local surplus does not always lead to better overall allocative efficiency, particularly when prices are inaccurate or queries are overpriced. [3]
- The tension between protecting a bidder's surplus by declining overpriced queries and assigning inputs to less capable specialists is absent in the centralized setting of Pandora's Router, where the router internalizes all costs and optimizes global welfare directly. [3]
- Pandora's Bidder corresponds to a single stage of an ascending-price mechanism with costly preference elicitation, isolating core components of decentralized prompt allocation under private value estimation with profit incentives. [4]
- Both Pandora's Router and Pandora's Bidder are derived from the same principle of balancing cheap screening and costly reasoning for routing, supporting allocation by either centralized routing or decentralized auctions. [1]

**Assessment:** The Qwen answer supplies the previously missing global-welfare/local-surplus distinction. Core content is complete, though the answer is long and citation alignment could be tighter.

- [2] Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation — chunk `2608.20316_4_22`
- [3] Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation — chunk `2608.20316_5_5`
- [1] Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation — chunk `2608.20316_4_1`

### q18: How does G-CARL supervise factuality and response coverage?

Model/policy: `qwen3.5:9b` / `coverage`. Recorded time: 33.4s.

- G-CARL supervises factuality through retrieval-grounded claim verification that decomposes reward supervision by verifying atomic claims against evidence retrieved from a multi-source medical datastore. [2][3]
- G-CARL supervises response coverage by constructing instance-specific weighted checklists through MLLM generation followed by clinician refinement to explicitly supervise demand satisfaction and expression quality. [1][3]
- The retrieval-grounded branch supervises medical factuality by verifying report-grounded medical claims against evidence retrieved from a multi-source medical datastore to produce the factuality reward Rfact. [4]
- The case-specific checklist branch optimizes demand satisfaction and expression quality by constructing a weighted checklist through MLLM generation followed by clinician-guided refinement, and evaluating the checklist coverage of the generated response to produce the checklist reward Rcheck. [4]
- Each checklist item is assigned an automatically generated weight, allowing the reward to emphasize the aspects most relevant to the current report and user question. [1]

**Assessment:** Qwen covers multi-source retrieval, atomic claim verification and weighted checklist construction. Core content is complete, though repetitive.

- [2] G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation — chunk `2608.20331_11_1`
- [3] G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation — chunk `2608.20331_1_1`
- [1] G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation — chunk `2608.20331_2_5`

### q08: How do submissions that change the learning algorithm compare with those that do not?

Model/policy: `qwen3.5:9b` / `coverage`. Recorded time: 35.2s.

- Submissions that change the learning algorithm achieve a mean score of 0.226, whereas submissions that do not change the learning procedure achieve a mean score of 0.126. [2][3]
- The gap between submissions that reach the learning algorithm layer and those that do not represents the difference between the two groups rather than an effect caused by reaching that layer, as the systems that reach it more often are stronger to begin with. [3]
- Submissions that change the learning algorithm represent 46.4% of the submissions that change anything, while those that do not change the learning procedure but modify other aspects like budgets or hyperparameters represent 53.6%. [2]
- Submissions that change the learning algorithm focus on the objective, supervision signal, learning rule, and data, whereas submissions that do not change the learning procedure move budgets, checkpointing, hyperparameters, and capacity. [2]
- Submissions that reach the learning algorithm layer are where the distance to a better algorithm is closed, while most submissions never go there. [3]

**Assessment:** Core performance comparison is now correct: 0.226 versus 0.126, with the caveat attributed to stronger systems rather than stronger submissions. The extra 46.4%/53.6% claim cites [2] but those percentages appear in [3]; this is a citation-alignment warning, not an incorrect value. This is a complete core answer with an unresolved secondary citation issue.

- [2] AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement — chunk `2608.20318_1_7`
- [3] AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement — chunk `2608.20318_9_10`

## Implemented and verified

- Experimental structured answer module with schema-constrained passage IDs, local validation, explicit generation failures and an optional single repair. Invalid output is not presented as evidence that a question is unsupported.
- Reproducible runner saving model digest, prompts/policy, generation settings, exact contexts, raw outputs, checks and timings. Citation-only mode is available alongside stricter quote-copy mode.
- Optional hybrid BM25/dense rank fusion and a retrieval-sweep runner covering candidate count, context size, per-paper limits, score gap and reranker ordering. **These new retrieval variants were not benchmarked or enabled.**
- Fifteen additional questions and criteria frozen from the PDFs before answer generation; same corpus, not an external test. The baseline retrieval run found the relevant papers for all 12 answerable additions and 24/28 evidence phrases. This is retrieval coverage, not answer accuracy.
- 140 offline tests passed. Existing notebook defaults remain unchanged.

## Work not completed

The three-stage plan is incomplete under the new time limit: no new generator had a full 25-question run, the 15 added questions have no generated-answer comparison, and retrieval/ranking candidates have not been evaluated for significant answer-quality gains. No significant retrieval improvement is claimed.

**Next continuation:** finish the Qwen structured pilot, then run the strongest candidate on the full development set before any default switch. Only after that, run the prepared retrieval sweep and hold-out answer comparison.

Implementation uses Ollama’s documented JSON-schema `format` parameter; structural validity does not prove semantic support. [Ollama API documentation](https://github.com/ollama/ollama/blob/main/docs/api.md).


In [5]:
display(Markdown((ROOT / "evaluation/results/grouped_concise_assessment.md").read_text()))


# Grouping alone: citations and concise comparisons

**Decision:** keep grouping alone and the shorter prompt as an incremental improvement. The key performance comparison now includes both scores, answers are shorter on average, and one previously uncited answer now has citations. Citation compliance remains incomplete; this is not a fully verified answer pipeline.

**Active setup:** notebook cell 77 uses `order_by_vector=True`; `ask_llm` groups passages by paper; the notebook default generator is `llama3.1:8b`. The shorter prompt is saved in `prompt_grouped_concise.txt` and is the active `PROMPT` in cell 85. No extra judge or repair model call was added.

**Controlled test:** two complete 25-question prompt runs against the same frozen passages and presentation, model weights and generation settings. Only the prompt changed between these two runs and the previous grouping-only baseline. Three completed answers from an initial literal-template attempt were preserved separately; that attempt was stopped after it copied the comparison template into an unrelated question. One request was interrupted. The longer revision was rejected for verbosity and factual regressions.

**Review basis:** assistant inspection against supplied passages, not independent human ratings. Complete means core factual content is supported, not perfect citation/format compliance. Partial includes missing material content or a correct core with an incorrect extra detail. Small single-run results do not establish general accuracy.

| Content result | Previous grouping | Longer revision (rejected) | Shorter revision (kept) |
| --- | ---: | ---: | ---: |
| Complete | 19 | 17 | 19 |
| Partial | 3 | 6 | 4 |
| Major issue | 1 | 0 | 0 |
| Correct abstention | 2 | 2 | 2 |

| Measure (23 answerable questions) | Previous grouping | Longer revision | Shorter revision |
| --- | ---: | ---: | ---: |
| Average answer words | 47.5 | 66.7 | 39.0 |
| Answers over 70 words | 6 | 8 | 3 |
| Answers containing a citation | 21 | 23 | 22 |

Word counts exclude bracketed citation numbers and appended software-generated evidence/references. Citation presence is not sentence-level compliance or proof of support.

## What improved

- **q08:** now reports **0.226 versus 0.126**, without the rejected candidate’s unsupported “significant difference” wording. Still rated partial: its caveat refers to stronger submissions, whereas the source refers to stronger systems producing them.
- **q15:** gives **69.6% versus 69.9% five-class accuracy** in one cited sentence.
- **q12:** states the **30% relative gain, from 14.29 to 18.57**, without repeated arithmetic or duplicated figures.
- **q22:** citations return to the cross-paper benchmark answer, though their positioning and source alignment remain imperfect.
- **q21/q23:** coverage of both relevant papers is retained; the longer candidate’s false attribution of auditable task models to G-CARL is avoided.
- **Unsupported questions:** both use the exact required refusal sentence.

## What remains unresolved

- **q19 still has no citations.** Several other answers put a citation only after multiple sentences or leave individual bullets uncited.
- **q01/q02/q18 remain incomplete.** Shortening also removes some previously present details: q01 loses the explicit stopping rule, q02 loses the local-surplus caveat, and q18 does not clearly specify weighted checklist construction. Their category stays partial; this should not be mistaken for no regression.
- **q17 repeats its definition in uncited bullets.** Some multi-paper answers still exceed the word target.
- Prompt-only instructions do not guarantee citation alignment or numerical reasoning. The next focused improvement would be a structured answer format with explicit claim/citation fields, validated locally; that has not been added here.

**Validation:** 132 offline tests passed after the final prompt change. Saved contexts and passages were verified identical to the grouping-only baseline. The review notebook was executed separately without model calls.

**Timing:** longer revision 305.7s; selected shorter revision 355.1s. These are recorded generation totals, not a controlled speed comparison; runtime was variable. The initial stopped template attempt is excluded from these totals.

## Per-question assessment

| ID | Before | After | Explanation |
| --- | --- | --- | --- |
| q01 | Partial | Partial | Correct value-of-information principle and final selection, but still omits reservation-price inspection order and the explicit stopping rule. First factual sentence has no adjacent citation. |
| q02 | Partial | Partial | Correct centralized versus decentralized distinction, but omits the Router’s global-welfare objective and local-surplus trade-off. Concise but less complete. |
| q03 | Complete | Complete | Correct three domains; citations appear on a separate line after the list instead of each bullet. |
| q04 | Complete | Complete | Correct possible harm from noisy competing estimates, without the longer candidate’s categorical claim. Second sentence repeats the point and citation follows only the second sentence. |
| q05 | Complete | Complete | Directly gives ten repositories and ten algorithm families as two cited bullets. |
| q06 | Complete | Complete | Correct four-hour agent budget and twelve-hour replay budget, with sentence-level citations. |
| q07 | Complete | Complete | Correct 0, 0.1 and 1.0 normalization anchors, with a supporting citation. |
| q08 | Major issue | Partial | Now gives the key 0.226 versus 0.126 scores without calling them statistically significant. However, the caveat says submissions were stronger to begin with; source [3] says the systems producing them were stronger. Adds unrequested reasoning-effort statistics. This is a substantial improvement over missing the scores, but not a fully precise comparison. |
| q09 | Complete | Complete | Correct objective hierarchy and procedure/control-flow explanation. Citation remains at paragraph end rather than after each factual sentence. |
| q10 | Complete | Complete | Correct 0.974 grouping agreement without unrelated metric comparisons. Still copies first-person wording from the paper. |
| q11 | Complete | Complete | Correct 74.9% result, now includes its metric label and citation. |
| q12 | Complete | Complete | Concise correct 30% relative accuracy gain from 14.29 to 18.57, with citation. |
| q13 | Complete | Complete | Correct 92 commuters and 454 observations, but citations still follow both sentences together. |
| q14 | Complete | Complete | Correct cycling and transit findings in one cited sentence. |
| q15 | Complete | Complete | Concise comparison with correctly assigned 69.6% and 69.9% five-class accuracies, plus the supported no-task-specific-fitting qualification. |
| q16 | Complete | Complete | Correct 71.5% accuracy with supporting citations. Extra image-context sentence is supported but not required. |
| q17 | Complete | Complete | Core PMRI definition is supported, but repeats it as three uncited bullets after two cited sentences. Fails the intended brevity and per-bullet citation format. |
| q18 | Partial | Partial | Correct atomic claim verification and case-specific checklist supervision. Still omits explicit multi-source datastore support and does not spell out the weighted checklist construction; citations are present per sentence. |
| q19 | Complete | Complete | Correct benchmark names and evaluation dimensions, and shorter than before. Still contains no passage citations; the citation-compliance goal is not met here. |
| q20 | Complete | Complete | Concise supported clinician preference finding with correct 136:85 and 106:64 margins and citation. |
| q21 | Complete | Complete | Both relevant papers are correctly identified with cited descriptions. Still adds a third bullet repeating the survey framework and exceeds the 70-word target. |
| q22 | Complete | Complete | Both benchmark papers and their roles are correct, and citations return. Citations are embedded near titles instead of ending the claims; the explicit evaluation of both Router and Bidder is in source [5], while the first bullet cites [4]. Still exceeds the length target. |
| q23 | Complete | Complete | Correctly identifies TMI and G-CARL without the longer candidate’s false claim that G-CARL induces auditable task models. Repeats TMI in two bullets; [2] would support its auditability more directly than [1]. |
| q24 | Correct abstention | Correct abstention | Correctly declines the unsupported KPMG forecast using the required exact sentence. |
| q25 | Correct abstention | Correct abstention | Correctly declines the unsupported NVIDIA stock price using the required exact sentence. |

## Answers and source checks

Source excerpts are literal saved passages shortened to 300 characters; full passages are available in the `manual_review_grouped_concise_full` worksheet.

<details><summary>q01 — How does Pandora's Router decide when an expensive specialist estimate is worth paying for?</summary>

**Answer:**

Pandora's Router decides when an expensive specialist estimate is worth paying for by computing reservation prices that quantify the expected value-of-information for querying a more costly and accurate value estimate. This is done by running the specialist for a few reasoning tokens and checking if it is on the right track [1]. The system then chooses the maximizer of the observed value estimate [1].

**Assessment:** Correct value-of-information principle and final selection, but still omits reservation-price inspection order and the explicit stopping rule. First factual sentence has no adjacent citation.

**Source [1] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> Pandora’s Router Centralized policy Inspect specialists𝑔𝑚(𝑥) in reservationprice order, stopping when no remaining query is worth its cost. Choose the maximizer of the observed𝑔𝑚. Pandora’s Bidder Decentralized policy The high bidder is selected in an auction. Each specialist refines its own value e…

</details>

<details><summary>q02 — What is the difference between Pandora's Router and Pandora's Bidder?</summary>

**Answer:**

Pandora's Router is a centralized policy that queries the more accurate value estimate 𝑔𝑚(𝑥) when the value of information exceeds the cost, while Pandora's Bidder is a decentralized policy where specialists use value-of-information reasoning to decide whether to refine their self-assessments before accepting a posted price. [1][3]

**Assessment:** Correct centralized versus decentralized distinction, but omits the Router’s global-welfare objective and local-surplus trade-off. Concise but less complete.

**Source [1] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> Pandora’s Router Centralized policy Inspect specialists𝑔𝑚(𝑥) in reservationprice order, stopping when no remaining query is worth its cost. Choose the maximizer of the observed𝑔𝑚. Pandora’s Bidder Decentralized policy The high bidder is selected in an auction. Each specialist refines its own value e…

**Source [3] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> we measure the regret versus an oracle that knows𝐺𝑚 for free. Results.Figure3plotsallocativeefficiencyandbiddersurplusfordifferent 𝑐𝑔. Pandora’sBidderclosely tracks the lower envelope of the regret of the two baselines across the full𝑐𝑔 range. When𝑐𝑔 is low, the bidder frequently refines its estimat…

</details>

<details><summary>q03 — In which three domains was the costly-value-estimation routing policy evaluated?</summary>

**Answer:**

The costly-value-estimation routing policy was evaluated in the following three domains:

• Multi-LLM routing benchmark (EmbedLLM)
• Retrieval-augmented specialists for factoid QA
• Inference-time computation for mathematical reasoning

[3][4]

**Assessment:** Correct three domains; citations appear on a separate line after the list instead of each bullet.

**Source [3] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> Bidder: each specialist faces a posted price—the current best competing offer—and must decide whether to invest in a more accurate self-assessment before claiming the query. This corresponds to a single stage of the ascending-price mechanism with costly preference elicitation studied by Parkes (2005…

**Source [4] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> 2026-08-21 Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation Adam Fisch, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate Larson, Jacob Eisenstein Google DeepMind Heterogeneous AI systems composed of multiple models, architectur…

</details>

<details><summary>q04 — What limitation appears when competing specialists' estimates are noisy?</summary>

**Answer:**

When competing estimates are noisy, value-of-information reasoning can increase the strategic specialist's utility at the expense of others. This is because the noisy estimates can lead to incorrect decisions, benefiting the strategic specialist but harming others. [1]

**Assessment:** Correct possible harm from noisy competing estimates, without the longer candidate’s categorical claim. Second sentence repeats the point and citation follows only the second sentence.

**Source [1] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> benchmark, retrieval-augmented specialists, and LLMs with variable inference-time reasoning—show that Pandora’s Router matches the routing quality of exhaustive estimation, while querying the expensive estimator far less often. In the decentralized setting, value-of-information reasoning improves al…

</details>

<details><summary>q05 — How many frozen repositories and training algorithm families are in AI4AI-Bench?</summary>

**Answer:**

• 10 frozen repositories [2]
• 10 distinct families of training algorithm [1]

**Assessment:** Directly gives ten repositories and ten algorithm families as two cited bullets.

**Source [2] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> The suite is ten frozen research repositories, listed in Table 1 and chosen so that between them they cover ten families of training algorithm rather than ten instances of one: supervised fine-tuning, multi-turn agentic RL, on-policy distillation, Bradley–Terry reward modeling, preference optimizati…

**Source [1] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> are kept. An evaluator fixed before the first run, with no access to the agent’s workspace, scores them, and the cell takes the best of the three under the task’s direction; for the two tasks that do not train, the twelve hours are only a ceiling on the verification stage (2.3).Bottom:the ten frozen…

</details>

<details><summary>q06 — What is the AI4AI-Bench time budget for an agent and for the replay?</summary>

**Answer:**

The AI4AI-Bench time budget for an agent is 4 hours on one B300 GPU [2]. The time budget for the replay is 12 hours [3].

**Assessment:** Correct four-hour agent budget and twelve-hour replay budget, with sentence-level citations.

**Source [2] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> Each task presents an agent with a research repository, a base model to start from, and an inexpensive proxy metric that can be evaluated freely during development 2.2. The agent is given 4 hours on one B300 GPU and a single objective: improve the training algorithm implemented in the repository. It…

**Source [3] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> Recursive self-improvement compounds through the algorithmic link, and AI4AI-Bench is built to measure that link on its own: ten frozen research repositories, each asking an agent to improve the training algorithm it already applies to its own model, with four hours to write code, twelve to run what…

</details>

<details><summary>q07 — How are the incommensurable AI4AI-Bench task metrics normalized?</summary>

**Answer:**

The incommensurable AI4AI-Bench task metrics are normalized to a scale where 0 is an uninformative model, 0.1 is the algorithm the repository ships, and 1.0 is the task optimum. [2]

**Assessment:** Correct 0, 0.1 and 1.0 normalization anchors, with a supporting citation.

**Source [2] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> incommensurable, every task is mapped onto one scale on which0is an uninformative model,0 .1is the algorithm the repository ships, and1.0 is the task optimum. Across 29 configurations of 6 systems on all 10 tasks the mean score is0.166, and the best system reaches0.250: even the strongest closes und…

</details>

<details><summary>q08 — How do submissions that change the learning algorithm compare with those that do not?</summary>

**Answer:**

• Submissions that change the learning algorithm average 0.226 against 0.126 for those that do not. [2]
• The share of submissions that touch the learning algorithm rises from 8.0% to 64.0% when reasoning effort is increased. [4]
• Submissions that change the learning algorithm are stronger to begin with, but the gap is the difference between those that go there and those that do not. [3]

**Assessment:** Now gives the key 0.226 versus 0.126 scores without calling them statistically significant. However, the caveat says submissions were stronger to begin with; source [3] says the systems producing them were stronger. Adds unrequested reasoning-effort statistics. This is a substantial improvement over missing the scores, but not a fully precise comparison.

**Source [2] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> …tive, the supervision signal, the learning rule, the data — average0.226 against0 .126for the rest: the algorithmic layer is where the distance gets closed, and most submissions never go there. More reasoning effort moves agents towards it, taking that share from8%to64%and the mean score from0 .094t…

**Source [3] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> …in this corpus. It is also not a randomized comparison: the systems that reach the learning procedure more often are the stronger ones to begin with, so the gap is the difference between the submissions that go there and the submissions that do not, rather than the effect of going there. Read togeth…

</details>

<details><summary>q09 — What two kinds of task structure does TMI recover from computer-use traces?</summary>

**Answer:**

TMI recovers two kinds of task structure: objective models and procedure models. Objective models are hierarchical decompositions of the objective of a task into subobjectives, while procedure models are trace-grounded compositions of control-flow operators specifying how the execution of a task is organized. [3]

**Assessment:** Correct objective hierarchy and procedure/control-flow explanation. Citation remains at paragraph end rather than after each factual sentence.

**Source [3] — Inducing Task Models from Computer-Use Traces:**

> Describing the activity conducted in a computeruse session is necessary for analyzing, learning, and auditing it. We formalize this process as the recovery, from a naturalistic computer-use trace, of the latent tasks and their task models specifying the objectives and procedures. Let X=⟨x 1, x2, . .…

</details>

<details><summary>q10 — How well does task model induction group interleaved tasks?</summary>

**Answer:**

Our method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings [2].

**Assessment:** Correct 0.974 grouping agreement without unrelated metric comparisons. Still copies first-person wording from the paper.

**Source [2] — Inducing Task Models from Computer-Use Traces:**

> into semantic actions and activities, bridging low-level signal and local intent (3.1).Latent task inductionuntangles the multi-threaded activity stream, discovering tasks and assigning possibly non-contiguous activities to each without a pre-specified task set (3.2).Task model constructionbuilds ea…

</details>

<details><summary>q11 — How much of observed execution does TMI reconstruct?</summary>

**Answer:**

TMI reconstructs 74.9% of observed execution steps. [4]

**Assessment:** Correct 74.9% result, now includes its metric label and citation.

**Source [4] — Inducing Task Models from Computer-Use Traces:**

> into semantic actions and activities, bridging low-level signal and local intent (3.1).Latent task inductionuntangles the multi-threaded activity stream, discovering tasks and assigning possibly non-contiguous activities to each without a pre-specified task set (3.2).Task model constructionbuilds ea…

</details>

<details><summary>q12 — What was the held-out task accuracy gain from skills derived from TMI?</summary>

**Answer:**

The held-out task accuracy gain from skills derived from TMI was 30% relative accuracy gain, from 14.29 to 18.57. [1]

**Assessment:** Concise correct 30% relative accuracy gain from 14.29 to 18.57, with citation.

**Source [1] — Inducing Task Models from Computer-Use Traces:**

> Exec. Safe. Align. Acc. No skill N/A N/A N/A 59.20 8.57 Human curated 93.59 63.80 90.50 56.59 10.00 Raw demonstration 52.46 53.4992.5263.15 11.43 Workflow summary 54.07 59.35 91.74 66.18 14.29 Ours54.95 67.6590.5767.99 18.57 The task model is an effective representation for skill transfer.Skills gen…

</details>

<details><summary>q13 — How many commuters and respondent-scenario observations did the weather travel survey collect?</summary>

**Answer:**

The survey collected 92 McGill student commuters. The final processed dataset contained 454 valid respondent–scenario observations. [2][3]

**Assessment:** Correct 92 commuters and 454 observations, but citations still follow both sentences together.

**Source [2] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> The Data Collection Agent was implemented as a purpose-built conversational chatbot,Travel- BehaviorSurveyBot, developed in Voiceflow and deployed as a web application accessible through a public link and QR code (Figure 2). Respondents completed the survey on their own devices through a guided dial…

**Source [3] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> The Data Processing Agent converted the chatbot’s raw, JSON-encoded response export into an analysis-ready dataset. Processing steps included parsing nested list- and slider-valued fields, validating and recoding categorical responses, constructing derived features, and flagging incomplete or incons…

</details>

<details><summary>q14 — Which travel modes responded most clearly to adverse weather and snow?</summary>

**Answer:**

Cycling was particularly sensitive to adverse weather, while public transit use increased substantially under the Snowy scenario [4].

**Assessment:** Correct cycling and transit findings in one cited sentence.

**Source [4] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> Travel-behavior research increasingly combines digital data collection with predictive modeling, yet these stages are often developed and evaluated separately. This study proposes and applies a three-agent workflow integrating conversational data collection, structured data processing, and behaviora…

</details>

<details><summary>q15 — How did random forest and the best text-only zero-shot LLM compare on five-class travel prediction?</summary>

**Answer:**

Random forest achieved a 69.6% five-class accuracy, while the best text-only zero-shot LLM reached 69.9% without task-specific fitting. [4]

**Assessment:** Concise comparison with correctly assigned 69.6% and 69.9% five-class accuracies, plus the supported no-task-specific-fitting qualification.

**Source [4] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> Travel-behavior research increasingly combines digital data collection with predictive modeling, yet these stages are often developed and evaluated separately. This study proposes and applies a three-agent workflow integrating conversational data collection, structured data processing, and behaviora…

</details>

<details><summary>q16 — What was the best vision-based travel mode prediction accuracy?</summary>

**Answer:**

The best vision-based travel mode prediction accuracy was 71.5% [2]. This was achieved by the best vision-based configuration, which used the same generated weather images shown to respondents [3].

**Assessment:** Correct 71.5% accuracy with supporting citations. Extra image-context sentence is supported but not required.

**Source [2] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> and the best vision-based configuration reached the highest observed five-class accuracy of 71.5%, indicating that visual context can provide additional predictive information for selected models. Taken together, the findings show that habitual travel information, prompting strategy, demonstrations,…

**Source [3] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> labeled travel-behavior data are limited, although the small sample and multiple evaluated configurations require cautious interpretation. The five-class task provides a clearer assessment of whether models distinguish among individual modes. Vision-based prediction.The vision experiments directly c…

</details>

<details><summary>q17 — What is the patient-oriented medical report interpretation task introduced with G-CARL?</summary>

**Answer:**

The patient-oriented medical report interpretation task, PMRI, is a novel open-ended multimodal generation task that requires models to explain medical reports in accurate and accessible language based on a user's query and dialogue history. [3] This task differs from existing medical vision-language tasks in that it requires both evidence-grounded medical factuality and context-dependent patient communication. [2]

Key properties of PMRI:
- Requires models to explain medical reports in accurate and accessible language
- Based on a user's query and dialogue history
- Combines evidence-grounded medical factuality and context-dependent patient communication

**Assessment:** Core PMRI definition is supported, but repeats it as three uncited bullets after two cited sentences. Fails the intended brevity and per-bullet citation format.

**Source [3] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> Personalized interpretation of medical reports has emerged as an increasingly important need among patients. Addressing this need requires both evidence-grounded medical factualityandcontext-dependentpatientcommunication,yetexisting medical vision-language tasks do not adequately capture these dual …

**Source [2] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> In this paper, we present PMRI as a challenging yet underexplored task that requires both evidence-grounded medical factualityandcontext-dependentpatientcommunication.We propose G-CARL, a reinforcement learning framework that decomposes reward supervision through retrieval-grounded claim verificatio…

</details>

<details><summary>q18 — How does G-CARL supervise factuality and response coverage?</summary>

**Answer:**

G-CARL supervises factuality through a retrieval-grounded claim reward that provides fine-grained supervision for medical factuality through atomic claim verification [1]. It supervises response coverage through a case-specific checklist reward that explicitly supervises demand satisfaction and expression quality without relying on a single reference response [1]. This reward is constructed through MLLM generation followed by clinician-guided refinement [4].

**Assessment:** Correct atomic claim verification and case-specific checklist supervision. Still omits explicit multi-source datastore support and does not spell out the weighted checklist construction; citations are present per sentence.

**Source [1] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> reward provides localized supervision for sparse factual errors and discouragesunsupportedelaboration.Forcontext-dependent objectives such as demand satisfaction and expression quality,G-CARLconstructsinstance-specificweightedchecklists throughMLLMgenerationfollowedbyclinicianrefinement. Each checkl…

**Source [4] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> BuiltuponGRPO,G-CARLoptimizesthepolicymodelwith three reward signals, as illustrated in Fig. 2. Given uploaded medical report imagesI, the dialogue historyh, and a user queryq,thepolicymodelfirstgeneratescandidateresponses. The retrieval-grounded branch then supervises medical factuality by verifyin…

</details>

<details><summary>q19 — What benchmark and evaluation dimensions accompany G-CARL?</summary>

**Answer:**

The benchmark accompanying G-CARL is MMedReport, a real-world multimodal PMRI benchmark, and CMB, an external benchmark that evaluates both medical QA accuracy and the professionalism of open-ended clinical interpretation. The evaluation dimensions for PMRI responses include Medical Accuracy, Demand Satisfaction, and Expression Quality.

**Assessment:** Correct benchmark names and evaluation dimensions, and shorter than before. Still contains no passage citations; the citation-compliance goal is not met here.

**Source [1] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> Datasets.We evaluate G-CARL primarily on(1)MMedReport, a real-world multimodal PMRI benchmark collectedfromonlinehealthcareconsultations.Itcontains2,450 instances, each including dialogue history, a user query, uploaded medical report images, and clinician-verified reference annotations. All instanc…

</details>

<details><summary>q20 — What did clinician pairwise preferences find about G-CARL?</summary>

**Answer:**

G-CARL was consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction, with preference margins of 136:85 and 106:64, respectively. [1]

**Assessment:** Concise supported clinician preference finding with correct 136:85 and 106:64 margins and citation.

**Source [1] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> andproducesclinicallyaccurateinterpretations.Similarly,in therightexample,SFTfailstoincorporatetheuser’ssmoking andalcoholcessationhistory,whileMLLM-as-a-Judgefails toaddresstheuser’sprimaryconcernofwhethertocontinue themedication.Incontrast,ourmethodeffectivelyintegrates historical context, directl…

</details>

<details><summary>q21 — Which papers address auditable AI workflows derived from user activity or survey responses?</summary>

**Answer:**

• Inducing Task Models from Computer-Use Traces introduces TASKMODELINDUCTION(TMI), which induces task models from computer-use traces, producing auditable and reusable models of how everyday work is done. [1]
• An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction proposes a multi-agent framework that integrates data collection, data processing, and behavioral modeling, producing structured and auditable outputs. [2]
• The framework in An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction connects AI-assisted behavioral data collection with travel-choice modeling and LLM-based prediction, providing a reproducible basis for exploring AI-assisted travel-behavior research. [4]

**Assessment:** Both relevant papers are correctly identified with cited descriptions. Still adds a third bullet repeating the survey framework and exceeds the 70-word target.

**Source [1] — Inducing Task Models from Computer-Use Traces:**

> Naturalistic computer-use traces, passively recorded screenshots and mouse or keyboard actions, are a valuable resource for deriving symbolic, auditable, and reusable models of how everyday work is done. Such models matter as computer-use agents enter real work, where agents need to learn how tasks …

**Source [2] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> We propose a multi-agent framework that integrates data collection, data processing, and behavioral modeling for commuter mode-choice research. The framework consists of three specialized agents: a Data Collection Agent, a Data Processing Agent, and a Data Modeling Agent. The agents perform distinct…

</details>

<details><summary>q22 — Which papers evaluate AI agents or model specialists using benchmark tasks and costs?</summary>

**Answer:**

• Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation [4] evaluates both Pandora’s Router and Pandora’s Bidder across three domains, including a multi-LLM routing benchmark and retrieval-augmented specialists for factoid QA.
• AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement [2] presents a benchmark that isolates the ability of an agent to design training algorithms, and [3] describes the benchmark's evaluation process, where agents rewrite training algorithms and their code is rerun from scratch for up to 12 hours.

**Assessment:** Both benchmark papers and their roles are correct, and citations return. Citations are embedded near titles instead of ending the claims; the explicit evaluation of both Router and Bidder is in source [5], while the first bullet cites [4]. Still exceeds the length target.

**Source [4] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> 2026-08-21 Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation Adam Fisch, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate Larson, Jacob Eisenstein Google DeepMind Heterogeneous AI systems composed of multiple models, architectur…

**Source [2] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> equations , AutoML-Zero evolves learning algorithms from primitive operations , and symbolic program search discovered Lion . These works establish that algorithmic design can be automated, but each demonstrates a method inside a compact search space and short proxy tasks; none is a benchmark of whe…

</details>

<details><summary>q23 — Which papers discuss evidence-grounded explanations and auditable task models?</summary>

**Answer:**

• Inducing Task Models from Computer-Use Traces discusses auditable task models [1].
• Inducing Task Models from Computer-Use Traces also discusses evidence-grounded explanations [1].
• G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation discusses evidence-grounded explanations [4].

**Assessment:** Correctly identifies TMI and G-CARL without the longer candidate’s false claim that G-CARL induces auditable task models. Repeats TMI in two bullets; [2] would support its auditability more directly than [1].

**Source [1] — Inducing Task Models from Computer-Use Traces:**

> Modeling a task t requires understanding the two types of evidence it carries, procedure and objective. Constructing a task model thus means inducing an objective model Ot and a procedure model Pt from the activities Et of each discovered task t, then reconciling them into the task model Mt (Figure …

**Source [4] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> explanations grounded inmultimodalreportsandextendedpatient–doctordialogue. cation by transforming report evidence into accessible explanations, addressing user-specific concerns, and offering clinically cautious guidance. As illustrated in Fig. 1, PMRI differs from conventional medical visual quest…

</details>

<details><summary>q24 — What is KPMG's 2026 revenue forecast?</summary>

**Answer:**

The provided papers do not address this question.

**Assessment:** Correctly declines the unsupported KPMG forecast using the required exact sentence.

</details>

<details><summary>q25 — Which of these papers reports the stock price of NVIDIA on October 14, 2026?</summary>

**Answer:**

The provided papers do not address this question.

**Assessment:** Correctly declines the unsupported NVIDIA stock price using the required exact sentence.

</details>


In [6]:
display(Markdown((ROOT / "evaluation/results/context_grouped_full_assessment.md").read_text()))


# Paper grouping alone — full benchmark assessment

**Conclusion:** grouping alone is the most promising of the tested configurations for core content, but it is not a clean upgrade: two answers lose all citations. It fixes the earlier cross-paper failures without the combined version’s new factual errors. Use it as the next optimization candidate, with citation compliance and concise output as the next targets. No production defaults were changed in this test.

**Scope:** 25 fresh Llama 3.1 8B generations with grouping enabled and selection-order reconstruction disabled. Same model weights, prompt, generation settings and exact frozen passages as the prior baseline. Grouping preserves first-seen paper order and within-paper order, but makes a paper’s passages adjacent; it is not an order-neutral change for interleaved papers. The earlier three-question pilot was rerun as part of this full run. No LLM judge or new retrieval was used.

**Review basis:** assistant assessment against saved sources, not independent human ratings. Complete refers to core factual content, not perfect citation/format compliance. Partial indicates material omissions or a correct core with a material erroneous extra detail. This small single-run comparison does not establish a stable accuracy rate.

| Content result | Baseline | Grouping + ordering | Grouping alone |
| --- | ---: | ---: | ---: |
| Complete | 17 | 18 | 19 |
| Partial | 4 | 3 | 3 |
| Major issue | 2 | 2 | 1 |
| Correct abstention | 2 | 2 | 2 |

**Citation trade-off:** q19 and q22 contain correct core content but no passage citations. Grouping alone has citations in 21/23 answerable responses, versus 22/23 for baseline (one false refusal) and 23/23 for both changes. These counts check citation presence, not whether every claim is supported.

## What improved and what remains

- **q21:** correctly names both auditable-workflow papers instead of falsely declining.
- **q23:** covers both TMI and G-CARL and avoids the combined version’s incorrect selection-operator claim.
- **q10:** retains the correct 0.974 grouping-agreement result without comparing it with unrelated 30.3% reconstruction.
- **q08:** still misses 0.226 versus 0.126; paper grouping does not solve this comparison failure.
- **q01, q02, q18:** existing omissions remain. In particular, q18 still lacks explicit multi-source retrieval, which the combined variant supplied.
- **q19 and q22:** new citation omissions; q22 is also verbose. Do not count the factual completeness increase as universal improvement.
- **q24 and q25:** both unsupported questions are correctly declined.

**Runtime:** 240.6 seconds of recorded generation time for 25 calls. Timing is descriptive, not a controlled latency benchmark.

**Recommendation:** prefer grouping alone as the next candidate, then test a small citation/conciseness prompt adjustment against this saved run. Treat q08 as a separate evidence-use/comparison problem. Keep the current configuration unchanged until those trade-offs are acceptable; this task only generated and assessed outputs.

## Per-question results

| ID | Baseline | Combined | Grouping alone | Explanation |
| --- | --- | --- | --- | --- |
| q01 | Partial | Partial | Partial | Value of information, stopping and selection are correct, but inspection in reservation-price order is still missing. |
| q02 | Partial | Partial | Partial | Correct centralized/decentralized distinction and local-surplus caveat. Still does not explicitly explain Router’s global-welfare objective and exceeds the requested three sentences. |
| q03 | Complete | Complete | Complete | All three domains are correct; one citation follows the whole list instead of each item. |
| q04 | Complete | Complete | Complete | Correctly describes increased strategic utility at others’ expense when competing estimates are noisy. |
| q05 | Complete | Complete | Complete | Correct ten repositories and ten algorithm families. |
| q06 | Complete | Complete | Complete | Correct four-hour agent budget on one B300 GPU and twelve-hour replay budget. |
| q07 | Complete | Complete | Complete | Correctly explains the 0, 0.1 and 1.0 normalization anchors. |
| q08 | Major issue | Major issue | Major issue | Still omits the performance comparison 0.226 versus 0.126 despite source [2] containing both. Submission shares and the supported non-causal caveat do not answer the main performance comparison. |
| q09 | Complete | Complete | Complete | Correct objective hierarchies and procedure/control-flow models; citation follows the paragraph. |
| q10 | Complete | Major issue | Complete | Correct 0.974 grouping agreement. Avoids the combined version’s false comparison with 30.3%. Still copies the paper’s first-person wording “Our method”. |
| q11 | Complete | Complete | Complete | Correct 74.9% execution reconstruction; a metric label would make the bare number clearer. |
| q12 | Complete | Complete | Complete | Correct 30% relative gain from 14.29 to 18.57, rather than a percentage-point change. |
| q13 | Complete | Complete | Complete | Correct 92 McGill student commuters and 454 respondent-scenario observations; citations are grouped after the second sentence. |
| q14 | Complete | Complete | Complete | Correct cycling sensitivity to adverse weather and increased transit use under snow. |
| q15 | Complete | Complete | Complete | Correct 69.6% random forest versus 69.9% best text-only zero-shot LLM; both values are supported by source [4]. |
| q16 | Complete | Complete | Complete | Correct 71.5% accuracy result; the bare number lacks a self-contained metric label. |
| q17 | Complete | Complete | Complete | Correct patient-oriented, report-grounded explanation task, using query and dialogue history. Explains patient communication and distinction from VQA/report generation. |
| q18 | Partial | Complete | Partial | Correct atomic claim verification and weighted case-specific checklists. Still omits explicit multi-source retrieval/datastore support, unlike the combined version. |
| q19 | Complete | Complete | Complete | Core benchmark and evaluation dimensions are correct, including the extra CMB and precision/recall details. However, the answer contains no passage citations: a citation regression, recorded separately from core factual completeness. |
| q20 | Complete | Complete | Complete | Correct clinician preference finding and supported margins of 136:85 and 106:64. |
| q21 | Major issue | Complete | Complete | Resolves the false refusal and identifies both relevant papers, with supported descriptions and TMI results. The answer is verbose and exposes paper IDs; claim-level citation placement can still improve. |
| q22 | Complete | Complete | Complete | Correctly identifies both papers and gives supported benchmark details. However, no passage citations are supplied, and the answer is unnecessarily long: another citation/formatting regression. |
| q23 | Partial | Partial | Complete | Covers both TMI and G-CARL with supported descriptions and citations. Avoids the combined version’s false selection-operator detail. Paper IDs are unnecessary for a business-facing answer. |
| q24 | Correct abstention | Correct abstention | Correct abstention | Correctly declines the unsupported KPMG revenue forecast. |
| q25 | Correct abstention | Correct abstention | Correct abstention | Correctly declines the unsupported NVIDIA stock-price question. |

## New answers and source checks

The excerpts are literal saved-source text, shortened to 300 characters. Full passages are in `manual_review_context_grouped_full.json` and its Markdown worksheet.

<details><summary>q01 — How does Pandora's Router decide when an expensive specialist estimate is worth paying for?</summary>

**Answer:**

Pandora's Router computes reservation prices that quantify the expected value-of-information for querying a more costly and accurate value estimate 𝑔𝑚(𝑥), e.g., by running the specialist for a few reasoning tokens and checking if it is on the right track [1]. The system stops when no remaining query is worth its cost, and chooses the maximizer of the observed 𝑔𝑚 [1].

**Assessment:** Value of information, stopping and selection are correct, but inspection in reservation-price order is still missing.

**Source [1] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> Pandora’s Router Centralized policy Inspect specialists𝑔𝑚(𝑥) in reservationprice order, stopping when no remaining query is worth its cost. Choose the maximizer of the observed𝑔𝑚. Pandora’s Bidder Decentralized policy The high bidder is selected in an auction. Each specialist refines its own value e…

</details>

<details><summary>q02 — What is the difference between Pandora's Router and Pandora's Bidder?</summary>

**Answer:**

Pandora's Router and Pandora's Bidder differ in their approach to value estimation and decision-making. Pandora's Router is a centralized policy that inspects specialists in reservation price order, stopping when no remaining query is worth its cost, and chooses the maximizer of the observed value estimate [1]. In contrast, Pandora's Bidder is a decentralized policy that selects the high bidder in an auction, with each specialist refining its own value estimate only when the market price makes inspection worthwhile [1]. Additionally, Pandora's Bidder measures regret versus an oracle that knows the true value for free, and its performance is affected by the price being inaccurate, leading to a trade-off between local surplus and overall allocative efficiency [3].

**Assessment:** Correct centralized/decentralized distinction and local-surplus caveat. Still does not explicitly explain Router’s global-welfare objective and exceeds the requested three sentences.

**Source [1] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> Pandora’s Router Centralized policy Inspect specialists𝑔𝑚(𝑥) in reservationprice order, stopping when no remaining query is worth its cost. Choose the maximizer of the observed𝑔𝑚. Pandora’s Bidder Decentralized policy The high bidder is selected in an auction. Each specialist refines its own value e…

**Source [3] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> we measure the regret versus an oracle that knows𝐺𝑚 for free. Results.Figure3plotsallocativeefficiencyandbiddersurplusfordifferent 𝑐𝑔. Pandora’sBidderclosely tracks the lower envelope of the regret of the two baselines across the full𝑐𝑔 range. When𝑐𝑔 is low, the bidder frequently refines its estimat…

</details>

<details><summary>q03 — In which three domains was the costly-value-estimation routing policy evaluated?</summary>

**Answer:**

The three domains where the costly-value-estimation routing policy was evaluated are:

1. A multi-LLM routing benchmark (EmbedLLM)
2. Retrieval-augmented specialists for factoid QA
3. Inference-time computation for mathematical reasoning

[3]

**Assessment:** All three domains are correct; one citation follows the whole list instead of each item.

**Source [3] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> Bidder: each specialist faces a posted price—the current best competing offer—and must decide whether to invest in a more accurate self-assessment before claiming the query. This corresponds to a single stage of the ascending-price mechanism with costly preference elicitation studied by Parkes (2005…

</details>

<details><summary>q04 — What limitation appears when competing specialists' estimates are noisy?</summary>

**Answer:**

When competing estimates are noisy, value-of-information reasoning can increase the strategic specialist's utility at the expense of others. [1]

**Assessment:** Correctly describes increased strategic utility at others’ expense when competing estimates are noisy.

**Source [1] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> benchmark, retrieval-augmented specialists, and LLMs with variable inference-time reasoning—show that Pandora’s Router matches the routing quality of exhaustive estimation, while querying the expensive estimator far less often. In the decentralized setting, value-of-information reasoning improves al…

</details>

<details><summary>q05 — How many frozen repositories and training algorithm families are in AI4AI-Bench?</summary>

**Answer:**

The AI4AI-Bench suite consists of 10 frozen research repositories, covering 10 distinct families of training algorithm. [2]

**Assessment:** Correct ten repositories and ten algorithm families.

**Source [2] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> The suite is ten frozen research repositories, listed in Table 1 and chosen so that between them they cover ten families of training algorithm rather than ten instances of one: supervised fine-tuning, multi-turn agentic RL, on-policy distillation, Bradley–Terry reward modeling, preference optimizati…

</details>

<details><summary>q06 — What is the AI4AI-Bench time budget for an agent and for the replay?</summary>

**Answer:**

The AI4AI-Bench time budget for an agent is 4 hours on one B300 GPU [2]. The time budget for replay is 12 hours [3].

**Assessment:** Correct four-hour agent budget on one B300 GPU and twelve-hour replay budget.

**Source [2] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> Each task presents an agent with a research repository, a base model to start from, and an inexpensive proxy metric that can be evaluated freely during development 2.2. The agent is given 4 hours on one B300 GPU and a single objective: improve the training algorithm implemented in the repository. It…

**Source [3] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> Recursive self-improvement compounds through the algorithmic link, and AI4AI-Bench is built to measure that link on its own: ten frozen research repositories, each asking an agent to improve the training algorithm it already applies to its own model, with four hours to write code, twelve to run what…

</details>

<details><summary>q07 — How are the incommensurable AI4AI-Bench task metrics normalized?</summary>

**Answer:**

Each task is mapped onto one scale where 0 is an uninformative model, 0.1 is the algorithm the repository ships, and 1.0 is the task optimum [2].

**Assessment:** Correctly explains the 0, 0.1 and 1.0 normalization anchors.

**Source [2] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> incommensurable, every task is mapped onto one scale on which0is an uninformative model,0 .1is the algorithm the repository ships, and1.0 is the task optimum. Across 29 configurations of 6 systems on all 10 tasks the mean score is0.166, and the best system reaches0.250: even the strongest closes und…

</details>

<details><summary>q08 — How do submissions that change the learning algorithm compare with those that do not?</summary>

**Answer:**

Submissions that change the learning algorithm (46.4% of 263 submissions) are compared to those that do not (53.6% of 263 submissions). The comparison is limited by the fact that the systems that reach the learning procedure more often are the stronger ones to begin with, so the gap is the difference between the submissions that go there and the submissions that do not, rather than the effect of going there [3].

**Assessment:** Still omits the performance comparison 0.226 versus 0.126 despite source [2] containing both. Submission shares and the supported non-causal caveat do not answer the main performance comparison.

**Source [2] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> … the objective, the supervision signal, the learning rule, the data — average0.226 against0 .126for the rest: the algorithmic layer is where the distance gets closed, and most submissions never go there. More reasoning effort moves agents towards it, taking that share from8%to64%and the mean score f…

</details>

<details><summary>q09 — What two kinds of task structure does TMI recover from computer-use traces?</summary>

**Answer:**

TMI recovers two kinds of task structure: objective models and procedure models. Objective models are hierarchical decompositions of the objective of a task into subobjectives, while procedure models are trace-grounded compositions of control-flow operators specifying how the execution of a task is organized. [3]

**Assessment:** Correct objective hierarchies and procedure/control-flow models; citation follows the paragraph.

**Source [3] — Inducing Task Models from Computer-Use Traces:**

> Describing the activity conducted in a computeruse session is necessary for analyzing, learning, and auditing it. We formalize this process as the recovery, from a naturalistic computer-use trace, of the latent tasks and their task models specifying the objectives and procedures. Let X=⟨x 1, x2, . .…

</details>

<details><summary>q10 — How well does task model induction group interleaved tasks?</summary>

**Answer:**

Our method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings [2].

**Assessment:** Correct 0.974 grouping agreement. Avoids the combined version’s false comparison with 30.3%. Still copies the paper’s first-person wording “Our method”.

**Source [2] — Inducing Task Models from Computer-Use Traces:**

> …sions (Wang et al., 2025), our method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings and matches 74.9% of observed execution steps against 30.3% for the strongest workflow induction baseline. Using the induced task model to generate reusable agent skills, evaluat…

</details>

<details><summary>q11 — How much of observed execution does TMI reconstruct?</summary>

**Answer:**

74.9% [4]

**Assessment:** Correct 74.9% execution reconstruction; a metric label would make the bare number clearer.

**Source [4] — Inducing Task Models from Computer-Use Traces:**

> into semantic actions and activities, bridging low-level signal and local intent (3.1).Latent task inductionuntangles the multi-threaded activity stream, discovering tasks and assigning possibly non-contiguous activities to each without a pre-specified task set (3.2).Task model constructionbuilds ea…

</details>

<details><summary>q12 — What was the held-out task accuracy gain from skills derived from TMI?</summary>

**Answer:**

The held-out task accuracy gain from skills derived from TMI was 30% relative accuracy gain, from 14.29 to 18.57, over the strongest baseline. [1]

**Assessment:** Correct 30% relative gain from 14.29 to 18.57, rather than a percentage-point change.

**Source [1] — Inducing Task Models from Computer-Use Traces:**

> Exec. Safe. Align. Acc. No skill N/A N/A N/A 59.20 8.57 Human curated 93.59 63.80 90.50 56.59 10.00 Raw demonstration 52.46 53.4992.5263.15 11.43 Workflow summary 54.07 59.35 91.74 66.18 14.29 Ours54.95 67.6590.5767.99 18.57 The task model is an effective representation for skill transfer.Skills gen…

</details>

<details><summary>q13 — How many commuters and respondent-scenario observations did the weather travel survey collect?</summary>

**Answer:**

The Data Collection Agent collected mode choices from 92 McGill student commuters. The final processed dataset contained 454 valid respondent–scenario observations. [2][3]

**Assessment:** Correct 92 McGill student commuters and 454 respondent-scenario observations; citations are grouped after the second sentence.

**Source [2] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> The Data Collection Agent was implemented as a purpose-built conversational chatbot,Travel- BehaviorSurveyBot, developed in Voiceflow and deployed as a web application accessible through a public link and QR code (Figure 2). Respondents completed the survey on their own devices through a guided dial…

**Source [3] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> The Data Processing Agent converted the chatbot’s raw, JSON-encoded response export into an analysis-ready dataset. Processing steps included parsing nested list- and slider-valued fields, validating and recoding categorical responses, constructing derived features, and flagging incomplete or incons…

</details>

<details><summary>q14 — Which travel modes responded most clearly to adverse weather and snow?</summary>

**Answer:**

Cycling was particularly sensitive to adverse weather, while public transit use increased substantially under the Snowy scenario. [4]

**Assessment:** Correct cycling sensitivity to adverse weather and increased transit use under snow.

**Source [4] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> Travel-behavior research increasingly combines digital data collection with predictive modeling, yet these stages are often developed and evaluated separately. This study proposes and applies a three-agent workflow integrating conversational data collection, structured data processing, and behaviora…

</details>

<details><summary>q15 — How did random forest and the best text-only zero-shot LLM compare on five-class travel prediction?</summary>

**Answer:**

Random forest achieved a five-class accuracy of 69.6% [4], while the best text-only zero-shot LLM reached 69.9% [4].

**Assessment:** Correct 69.6% random forest versus 69.9% best text-only zero-shot LLM; both values are supported by source [4].

**Source [4] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> Travel-behavior research increasingly combines digital data collection with predictive modeling, yet these stages are often developed and evaluated separately. This study proposes and applies a three-agent workflow integrating conversational data collection, structured data processing, and behaviora…

</details>

<details><summary>q16 — What was the best vision-based travel mode prediction accuracy?</summary>

**Answer:**

71.5% [2]

**Assessment:** Correct 71.5% accuracy result; the bare number lacks a self-contained metric label.

**Source [2] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> and the best vision-based configuration reached the highest observed five-class accuracy of 71.5%, indicating that visual context can provide additional predictive information for selected models. Taken together, the findings show that habitual travel information, prompting strategy, demonstrations,…

</details>

<details><summary>q17 — What is the patient-oriented medical report interpretation task introduced with G-CARL?</summary>

**Answer:**

Patient-oriented Medical Report Interpretation (PMRI) is a novel open-ended multimodal generation task that requires models to explain medical reports in accurate and accessible language based on a user's query and dialogue history. [3]

PMRI must align professional medical knowledge with patient-centered communication by transforming report evidence into accessible explanations, addressing user-specific concerns, and offering clinically cautious guidance. [4]

PMRI differs fundamentally from medical VQA (Task A) and conventional report generation (Task B) in that it produces long-form, patient-facing explanations grounded in multimodal reports and extended patient-doctor dialogue. [4]

**Assessment:** Correct patient-oriented, report-grounded explanation task, using query and dialogue history. Explains patient communication and distinction from VQA/report generation.

**Source [3] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> Personalized interpretation of medical reports has emerged as an increasingly important need among patients. Addressing this need requires both evidence-grounded medical factualityandcontext-dependentpatientcommunication,yetexisting medical vision-language tasks do not adequately capture these dual …

**Source [4] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> Written within a professional medical context, medical reports commonly present abnormal values and descriptive findingswithoutexplainingtheirimplicationsfornon-expert readers. This gap becomes particularly salient in online healthcare scenarios, where patients upload one or more report images and a…

</details>

<details><summary>q18 — How does G-CARL supervise factuality and response coverage?</summary>

**Answer:**

G-CARL supervises factuality through a retrieval-grounded claim reward that provides fine-grained supervision for medical factuality through atomic claim verification [1]. It also supervises response coverage through a case-specific checklist reward that explicitly supervises demand satisfaction and expression quality without relying on a single reference response [1]. This is achieved by constructing instance-specific weighted checklists through MLLM generation followed by clinician refinement [1].

**Assessment:** Correct atomic claim verification and weighted case-specific checklists. Still omits explicit multi-source retrieval/datastore support, unlike the combined version.

**Source [1] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> reward provides localized supervision for sparse factual errors and discouragesunsupportedelaboration.Forcontext-dependent objectives such as demand satisfaction and expression quality,G-CARLconstructsinstance-specificweightedchecklists throughMLLMgenerationfollowedbyclinicianrefinement. Each checkl…

</details>

<details><summary>q19 — What benchmark and evaluation dimensions accompany G-CARL?</summary>

**Answer:**

The benchmark accompanying G-CARL is MMedReport, a real-world multimodal PMRI benchmark, and CMB, an external benchmark that evaluates both medical QA accuracy and the professionalism of open-ended clinical interpretation. 

The evaluation dimensions for PMRI responses include subjective evaluation along three clinician-defined dimensions: Medical Accuracy (Accuracy), Demand Satisfaction (Satisfaction), and Expression Quality (Expression), and objective evaluation using claim-level Precision and checklist-level Recall.

**Assessment:** Core benchmark and evaluation dimensions are correct, including the extra CMB and precision/recall details. However, the answer contains no passage citations: a citation regression, recorded separately from core factual completeness.

**Source [1] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> Datasets.We evaluate G-CARL primarily on(1)MMedReport, a real-world multimodal PMRI benchmark collectedfromonlinehealthcareconsultations.Itcontains2,450 instances, each including dialogue history, a user query, uploaded medical report images, and clinician-verified reference annotations. All instanc…

**Source [2] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> (OpenAI 2025) as the judge by strictly following these evaluation criteria. The complete judging protocol is provided in theAppendix Bto facilitate reproducibility. (2) Objective Evaluation. In addition to holistic assessment, we report claim-levelPrecisionand checklist-levelRecall. Precisionisdefin…

</details>

<details><summary>q20 — What did clinician pairwise preferences find about G-CARL?</summary>

**Answer:**

G-CARL was consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction, with preference margins of 136:85 and 106:64, respectively. [1]

**Assessment:** Correct clinician preference finding and supported margins of 136:85 and 106:64.

**Source [1] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> andproducesclinicallyaccurateinterpretations.Similarly,in therightexample,SFTfailstoincorporatetheuser’ssmoking andalcoholcessationhistory,whileMLLM-as-a-Judgefails toaddresstheuser’sprimaryconcernofwhethertocontinue themedication.Incontrast,ourmethodeffectivelyintegrates historical context, directl…

</details>

<details><summary>q21 — Which papers address auditable AI workflows derived from user activity or survey responses?</summary>

**Answer:**

The provided papers address auditable AI workflows derived from user activity or survey responses.

* Inducing Task Models from Computer-Use Traces (Paper ID: 2608.20319) introduces TASKMODELINDUCTION (TMI), which induces task models from computer-use traces, allowing for the recovery of interleaved tasks with 0.974 agreement against ground-truth groupings and the reconstruction of 74.9% of observed execution steps. [1]
* An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction (Paper ID: 2608.20320) proposes a multi-agent framework that integrates data collection, data processing, and behavioral modeling, producing structured and auditable outputs. [2] The framework consists of three specialized agents: a Data Collection Agent, a Data Processing Agent, and a Data Modeling Agent, which exchange information through standardized data objects and explicit input–output interfaces. [3]

**Assessment:** Resolves the false refusal and identifies both relevant papers, with supported descriptions and TMI results. The answer is verbose and exposes paper IDs; claim-level citation placement can still improve.

**Source [1] — Inducing Task Models from Computer-Use Traces:**

> Naturalistic computer-use traces, passively recorded screenshots and mouse or keyboard actions, are a valuable resource for deriving symbolic, auditable, and reusable models of how everyday work is done. Such models matter as computer-use agents enter real work, where agents need to learn how tasks …

**Source [2] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> We propose a multi-agent framework that integrates data collection, data processing, and behavioral modeling for commuter mode-choice research. The framework consists of three specialized agents: a Data Collection Agent, a Data Processing Agent, and a Data Modeling Agent. The agents perform distinct…

</details>

<details><summary>q22 — Which papers evaluate AI agents or model specialists using benchmark tasks and costs?</summary>

**Answer:**

Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation (Paper ID: 2608.20316) evaluates AI agents or model specialists using benchmark tasks and costs. Specifically, it evaluates Pandora's Router and Pandora's Bidder across three domains: a multi-LLM routing benchmark, retrieval-augmented specialists for factoid QA, and inference-time computation for mathematical reasoning. The evaluation shows that both frameworks are able to trade-off expensive and cheap estimates to achieve strong allocative performance relative to baselines across a range of estimation costs.

AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement (Paper ID: 2608.20318) also evaluates AI agents or model specialists using benchmark tasks and costs. It presents AI4AI-Bench, a benchmark that isolates the ability of an agent to design training algorithms, and evaluates 6 systems across 29 configurations on 10 frozen research repositories spanning 10 training algorithm families. The evaluation scores the agent's ability to improve the training algorithm and maps the 10 metrics onto a single scale.

**Assessment:** Correctly identifies both papers and gives supported benchmark details. However, no passage citations are supplied, and the answer is unnecessarily long: another citation/formatting regression.

**Source [3] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement Yizhe Chi , Wenyi Li, Deyao Hong, Xiaoqiu Wang, Mingju Gao, Kaisen Yang, Bingxiang He, Y oujie Zheng, Calvin Xiao, Qinhuai Na Navers Lab, Einsia.AI T singhua University Project Lead Corresponding Author Abstrac…

**Source [5] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> Bidder: each specialist faces a posted price—the current best competing offer—and must decide whether to invest in a more accurate self-assessment before claiming the query. This corresponds to a single stage of the ascending-price mechanism with costly preference elicitation studied by Parkes (2005…

</details>

<details><summary>q23 — Which papers discuss evidence-grounded explanations and auditable task models?</summary>

**Answer:**

The provided papers discuss evidence-grounded explanations and auditable task models.

* Inducing Task Models from Computer-Use Traces (Paper ID: 2608.20319) discusses auditable task models, specifically objective models that explain why observed activities were performed, and procedure models that capture how the execution of the task was organized in time. [1]
* G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation (Paper ID: 2608.20331) discusses evidence-grounded explanations, specifically transforming report evidence into accessible explanations that address user-specific concerns and offer clinically cautious guidance. [4]

**Assessment:** Covers both TMI and G-CARL with supported descriptions and citations. Avoids the combined version’s false selection-operator detail. Paper IDs are unnecessary for a business-facing answer.

**Source [1] — Inducing Task Models from Computer-Use Traces:**

> Modeling a task t requires understanding the two types of evidence it carries, procedure and objective. Constructing a task model thus means inducing an objective model Ot and a procedure model Pt from the activities Et of each discovered task t, then reconciling them into the task model Mt (Figure …

**Source [4] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> explanations grounded inmultimodalreportsandextendedpatient–doctordialogue. cation by transforming report evidence into accessible explanations, addressing user-specific concerns, and offering clinically cautious guidance. As illustrated in Fig. 1, PMRI differs from conventional medical visual quest…

</details>

<details><summary>q24 — What is KPMG's 2026 revenue forecast?</summary>

**Answer:**

The provided papers do not address this question.

**Assessment:** Correctly declines the unsupported KPMG revenue forecast.

</details>

<details><summary>q25 — Which of these papers reports the stock price of NVIDIA on October 14, 2026?</summary>

**Answer:**

The provided papers do not address this question.

**Assessment:** Correctly declines the unsupported NVIDIA stock-price question.

</details>


In [7]:
display(Markdown((ROOT / "evaluation/results/context_presentation_assessment.md").read_text()))


# Passage grouping and ordering — assessment

**Conclusion:** mixed results, not a reliable overall upgrade. Grouping and selection order improve paper coverage, but the combined version introduces a wrong metric comparison and an incorrect extra detail. Keep these changes experimental. The next bounded test should evaluate grouping alone more broadly before adopting it.

**Scope:** 31 new generations: grouping-only and ordering-only on q08/q21/q23 (six calls), then both together on all 25 questions. Compared with the previous 25 Llama 3.1 8B answers. Model weights, prompt, generation options and passage text were verified unchanged. No new retrieval or LLM judge. Original citation IDs are preserved; the ordering experiment reconstructs reserved-paper selection from saved reranker scores. Live retrieval assigns IDs after ordering, so this is a controlled presentation test, not a fresh end-to-end retrieval run.

**Review basis:** assistant assessment against saved passages, not an independent human rating. Complete means the core factual answer is supported; citation/formatting flaws are reported separately. Partial includes missing required content or a correct core answer with a material incorrect extra detail. Major issue includes a misleading central comparison or unanswered central question. Small single-run tests do not establish a stable accuracy rate.

| Result | Previous | Grouping + ordering |
| --- | ---: | ---: |
| Complete | 17 | 18 |
| Partial | 4 | 3 |
| Major issue | 2 | 2 |
| Correct abstention | 2 | 2 |

## What changed

- **Improved:** q21 no longer falsely refuses; q18 now explains multi-source retrieval and checklist coverage. q03 gives a citation for each domain.
- **Still wrong:** q08 omits **0.226 versus 0.126**, even with the direct score passage first. Moving evidence earlier was insufficient.
- **New regression:** q10 compares **0.974 grouping agreement** with **30.3% execution reconstruction**. The latter should be compared with 74.9%, not 0.974.
- **Mixed:** q23 now covers both papers but adds an incorrect claim about selection operators, despite the source explicitly excluding selection. It is also more verbose.
- **Preserved:** both genuinely unsupported questions are declined. Citation presence still does not establish factual correctness.

## Three-question isolation test

| Variant | q08: performance comparison | q21: auditable workflows | q23: explanations and task models |
| --- | --- | --- | --- |
| Previous | Missing scores | False refusal | Omits G-CARL |
| Grouping only | Missing scores | Both papers covered | Both papers covered; no operator error |
| Ordering only | Missing scores | Both papers covered | Both papers covered; no operator error |
| Both | Missing scores | Both papers covered | Both papers covered, but adds operator error |

Grouping-only and ordering-only were tested on three selected failures, not all 25; neither can yet be declared the best general configuration. Grouping can itself change order by making a paper’s passages adjacent. No extra multi-paper prompt rule was added, keeping this test focused on presentation.

## Implementation and validation

- Moved the notebook’s standalone `format_context` into `ResearchRetriever`; the notebook call now resolves correctly. Previously the class method rejected `group_by_paper`, and the standalone function referenced an unimported `RetrievedChunk` annotation.
- Restored the accidentally replaced `format_answer` helper.
- Added optional grouping while preserving original citation IDs and the legacy default format used to validate frozen snapshots.
- Added `--group-by-paper` and `--selection-order` to the answer runner, saved exact generation context, and fingerprinted both switches to prevent incompatible resumes.
- 132 offline tests passed, including notebook wiring, grouping/citation identity, unknown-paper separation, selection ordering and resume protection. The notebook keeps the user’s grouping and `order_by_vector=False` experiment; the default generator is still llama3.2, so request llama3.1:8b explicitly.

- `answers_context_grouped.json`: 3 calls, 48.6s recorded generation time.
- `answers_context_ordered.json`: 3 calls, 45.5s recorded generation time.
- `answers_context_both_full.json`: 25 calls, 257.6s recorded generation time.

Timing is descriptive, not a controlled speed benchmark; warmth and answer length differ.

## Per-question comparison

| ID | Before | After | Explanation |
| --- | --- | --- | --- |
| q01 | Partial | Partial | Still omits inspection in reservation-price order. Value of information, stopping and selection are supported. |
| q02 | Partial | Partial | Explains centralized inspection versus decentralized auction, but still omits the global-welfare objective and now also omits the local-surplus trade-off. Citations appear only at the end. |
| q03 | Complete | Complete | All three domains are correct; each list item now has its own citation. |
| q04 | Complete | Complete | Correctly explains increased strategic utility at other specialists’ expense when competing estimates are noisy. |
| q05 | Complete | Complete | Correctly reports ten repositories and ten algorithm families. |
| q06 | Complete | Complete | Correct four-hour agent budget on one B300 and twelve-hour replay budget. |
| q07 | Complete | Complete | Correctly associates 0, 0.1 and 1.0 with the uninformative model, shipped algorithm and optimum. |
| q08 | Major issue | Major issue | Still omits the performance scores 0.226 versus 0.126, even though source [2] is presented first. Submission shares and types of changes do not cover the main performance comparison. The prior non-causal caveat is also absent. |
| q09 | Complete | Complete | Correctly explains hierarchical objective models and procedure/control-flow models. Citation is at paragraph end. |
| q10 | Complete | Major issue | New metric-mixing error: 0.974 is task-grouping agreement, but the answer compares it with 30.3%, which source [2] assigns to execution-step reconstruction. The correct reconstruction comparison is 74.9% versus 30.3%. Also retains the paper’s first-person “Our method”. |
| q11 | Complete | Complete | Correct 74.9% reconstruction result; a metric label would make the bare number more self-contained. |
| q12 | Complete | Complete | Correctly states a 30% relative gain from 14.29 to 18.57, rather than a percentage-point increase. |
| q13 | Complete | Complete | Correct 92 student commuters and 454 respondent-scenario observations. |
| q14 | Complete | Complete | Correct adverse-weather sensitivity for cycling and increased transit use under snow. |
| q15 | Complete | Complete | Correctly associates 69.6% with random forest and 69.9% with text-only zero-shot LLM; source [4] contains both. |
| q16 | Complete | Complete | Correct 71.5% result, though the bare number would benefit from a metric label. |
| q17 | Complete | Complete | Correct patient-oriented explanation task, grounded in report images, user query and dialogue history. |
| q18 | Partial | Complete | Now covers multi-source medical retrieval and claim verification plus the weighted case-specific checklist, all supported by source [4]. This fills the earlier retrieval omission. |
| q19 | Complete | Complete | Correct MMedReport benchmark and all three evaluation dimensions. Extra CMB and precision/recall details are supported. Citations remain bunched at the end instead of following each factual sentence. |
| q20 | Complete | Complete | Correct clinician preference finding and supported margins 136:85 and 106:64. |
| q21 | Major issue | Complete | False refusal is resolved: identifies both the survey workflow and TMI, explains their different inputs, and cites both papers. Citation placement is still mostly at bullet ends. |
| q22 | Complete | Complete | Correctly identifies Pandora routing and AI4AI-Bench, their evaluation domains and budgets/costs. More detailed than the earlier answer; citations remain at paragraph ends. |
| q23 | Partial | Partial | Now covers both G-CARL and TMI, but adds a false detail: says the three observable operators are sequence, selection and iteration. Source [5] explicitly excludes selection from the output and specifies SEQ, FOR and WHILE. The answer is also long and exposes paper IDs instead of readable titles. |
| q24 | Correct abstention | Correct abstention | Correctly declines the unsupported KPMG revenue forecast. |
| q25 | Correct abstention | Correct abstention | Correctly declines the unsupported NVIDIA stock price. |

## Answers and source excerpts

Excerpts below are literal saved-source text, shortened to at most 300 characters. They are spot checks; the offline worksheet contains full passages.

<details><summary>q01 — How does Pandora's Router decide when an expensive specialist estimate is worth paying for?</summary>

**New answer:**

Pandora's Router computes reservation prices that quantify the expected value-of-information for querying a more costly and accurate value estimate 𝑔𝑚(𝑥), e.g., by running the specialist for a few reasoning tokens and checking if it is on the right track [3]. The system stops when no remaining query is worth its cost [1]. The maximizer of the observed 𝑔𝑚 is chosen [1].

**Assessment:** Still omits inspection in reservation-price order. Value of information, stopping and selection are supported.

**Source [3] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> We now extend Pandora’s Router to a decentralized setting in which the specialists control their own value estimates, and use them to participate in a marketplace. Pandora’s Router assumes that a single decision-maker controls which boxes to open and which specialist to select. In many settings, how…

**Source [1] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> Pandora’s Router Centralized policy Inspect specialists𝑔𝑚(𝑥) in reservationprice order, stopping when no remaining query is worth its cost. Choose the maximizer of the observed𝑔𝑚. Pandora’s Bidder Decentralized policy The high bidder is selected in an auction. Each specialist refines its own value e…

</details>

<details><summary>q02 — What is the difference between Pandora's Router and Pandora's Bidder?</summary>

**New answer:**

Pandora's Router is a centralized policy that inspects specialists in reservation price order, stopping when no remaining query is worth its cost, and chooses the maximizer of the observed value estimate. In contrast, Pandora's Bidder is a decentralized policy that selects the high bidder in an auction, where each specialist refines its own value estimate only when the market price makes inspection worthwhile. [1][2][3]

**Assessment:** Explains centralized inspection versus decentralized auction, but still omits the global-welfare objective and now also omits the local-surplus trade-off. Citations appear only at the end.

**Source [1] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> Pandora’s Router Centralized policy Inspect specialists𝑔𝑚(𝑥) in reservationprice order, stopping when no remaining query is worth its cost. Choose the maximizer of the observed𝑔𝑚. Pandora’s Bidder Decentralized policy The high bidder is selected in an auction. Each specialist refines its own value e…

**Source [2] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> and all methods except𝑓-only (which never queries𝑔) perform similarly. As𝑐𝑔 increases, the𝑔-always baseline continues to make a fixed number of queries regardless of cost, while𝑓-only still never queries at all. Pandora’s Router interpolates between these extremes, querying𝑔 only when the value of i…

</details>

<details><summary>q03 — In which three domains was the costly-value-estimation routing policy evaluated?</summary>

**New answer:**

The three domains where the costly-value-estimation routing policy was evaluated are: 
1. a multi-LLM routing benchmark (EmbedLLM) [3]
2. retrieval-augmented specialists for factoid QA [3]
3. inference-time computation for mathematical reasoning [3]

**Assessment:** All three domains are correct; each list item now has its own citation.

**Source [3] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> Bidder: each specialist faces a posted price—the current best competing offer—and must decide whether to invest in a more accurate self-assessment before claiming the query. This corresponds to a single stage of the ascending-price mechanism with costly preference elicitation studied by Parkes (2005…

</details>

<details><summary>q04 — What limitation appears when competing specialists' estimates are noisy?</summary>

**New answer:**

When competing estimates are noisy, value-of-information reasoning can increase the strategic specialist's utility at the expense of others. [1]

**Assessment:** Correctly explains increased strategic utility at other specialists’ expense when competing estimates are noisy.

**Source [1] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> benchmark, retrieval-augmented specialists, and LLMs with variable inference-time reasoning—show that Pandora’s Router matches the routing quality of exhaustive estimation, while querying the expensive estimator far less often. In the decentralized setting, value-of-information reasoning improves al…

</details>

<details><summary>q05 — How many frozen repositories and training algorithm families are in AI4AI-Bench?</summary>

**New answer:**

10 frozen research repositories, each covering a distinct family of training algorithm, spanning 10 training algorithm families. [2]

**Assessment:** Correctly reports ten repositories and ten algorithm families.

**Source [2] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> The suite is ten frozen research repositories, listed in Table 1 and chosen so that between them they cover ten families of training algorithm rather than ten instances of one: supervised fine-tuning, multi-turn agentic RL, on-policy distillation, Bradley–Terry reward modeling, preference optimizati…

</details>

<details><summary>q06 — What is the AI4AI-Bench time budget for an agent and for the replay?</summary>

**New answer:**

The AI4AI-Bench time budget for an agent is 4 hours on one B300 GPU [2]. The time budget for replay is 12 hours [2].

**Assessment:** Correct four-hour agent budget on one B300 and twelve-hour replay budget.

**Source [2] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> Each task presents an agent with a research repository, a base model to start from, and an inexpensive proxy metric that can be evaluated freely during development 2.2. The agent is given 4 hours on one B300 GPU and a single objective: improve the training algorithm implemented in the repository. It…

</details>

<details><summary>q07 — How are the incommensurable AI4AI-Bench task metrics normalized?</summary>

**New answer:**

The AI4AI-Bench task metrics are normalized to a common scale where 0 is an uninformative model, 0.1 is the algorithm the repository ships, and 1.0 is the task optimum. [2]

**Assessment:** Correctly associates 0, 0.1 and 1.0 with the uninformative model, shipped algorithm and optimum.

**Source [2] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> incommensurable, every task is mapped onto one scale on which0is an uninformative model,0 .1is the algorithm the repository ships, and1.0 is the task optimum. Across 29 configurations of 6 systems on all 10 tasks the mean score is0.166, and the best system reaches0.250: even the strongest closes und…

</details>

<details><summary>q08 — How do submissions that change the learning algorithm compare with those that do not?</summary>

**New answer:**

Submissions that change the learning algorithm (46.4% of 263 submissions) are less common than those that do not (53.6% of 263 submissions) [3]. The former group includes changes to the loss it optimizes, supervision it learns from, update rule itself, and data it trains on [1]. In contrast, submissions that do not change the learning algorithm instead modify how the run goes, such as training time, hyperparameters, and checkpointing [1].

**Assessment:** Still omits the performance scores 0.226 versus 0.126, even though source [2] is presented first. Submission shares and types of changes do not cover the main performance comparison. The prior non-causal caveat is also absent.

**Source [2] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> …reach it — the objective, the supervision signal, the learning rule, the data — average0.226 against0 .126for the rest: the algorithmic layer is where the distance gets closed, and most submissions never go there. More reasoning effort moves agents towards it, taking that share from8%to64%and the me…

</details>

<details><summary>q09 — What two kinds of task structure does TMI recover from computer-use traces?</summary>

**New answer:**

TMI recovers two kinds of task structure: objective models and procedure models. Objective models are hierarchical decompositions of the objective of a task into subobjectives, while procedure models are trace-grounded compositions of control-flow operators specifying how the execution of a task is organized. [3]

**Assessment:** Correctly explains hierarchical objective models and procedure/control-flow models. Citation is at paragraph end.

**Source [3] — Inducing Task Models from Computer-Use Traces:**

> Describing the activity conducted in a computeruse session is necessary for analyzing, learning, and auditing it. We formalize this process as the recovery, from a naturalistic computer-use trace, of the latent tasks and their task models specifying the objectives and procedures. Let X=⟨x 1, x2, . .…

</details>

<details><summary>q10 — How well does task model induction group interleaved tasks?</summary>

**New answer:**

Our method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings [2]. This is compared to 30.3% for the strongest workflow induction baseline [2].

**Assessment:** New metric-mixing error: 0.974 is task-grouping agreement, but the answer compares it with 30.3%, which source [2] assigns to execution-step reconstruction. The correct reconstruction comparison is 74.9% versus 30.3%. Also retains the paper’s first-person “Our method”.

**Source [2] — Inducing Task Models from Computer-Use Traces:**

> …n work sessions (Wang et al., 2025), our method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings and matches 74.9% of observed execution steps against 30.3% for the strongest workflow induction baseline. Using the induced task model to generate reusable agent skill…

</details>

<details><summary>q11 — How much of observed execution does TMI reconstruct?</summary>

**New answer:**

74.9% [2]

**Assessment:** Correct 74.9% reconstruction result; a metric label would make the bare number more self-contained.

**Source [2] — Inducing Task Models from Computer-Use Traces:**

> Naturalistic computer-use traces, passively recorded screenshots and mouse or keyboard actions, are a valuable resource for deriving symbolic, auditable, and reusable models of how everyday work is done. Such models matter as computer-use agents enter real work, where agents need to learn how tasks …

</details>

<details><summary>q12 — What was the held-out task accuracy gain from skills derived from TMI?</summary>

**New answer:**

The held-out task accuracy gain from skills derived from TMI was 30% relative accuracy gain, from 14.29 to 18.57, over the strongest baseline. [1]

**Assessment:** Correctly states a 30% relative gain from 14.29 to 18.57, rather than a percentage-point increase.

**Source [1] — Inducing Task Models from Computer-Use Traces:**

> Exec. Safe. Align. Acc. No skill N/A N/A N/A 59.20 8.57 Human curated 93.59 63.80 90.50 56.59 10.00 Raw demonstration 52.46 53.4992.5263.15 11.43 Workflow summary 54.07 59.35 91.74 66.18 14.29 Ours54.95 67.6590.5767.99 18.57 The task model is an effective representation for skill transfer.Skills gen…

</details>

<details><summary>q13 — How many commuters and respondent-scenario observations did the weather travel survey collect?</summary>

**New answer:**

The study collected 92 student commuters and 454 respondent-scenario observations. [2]

**Assessment:** Correct 92 student commuters and 454 respondent-scenario observations.

**Source [2] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> The Data Collection Agent was implemented as a purpose-built conversational chatbot,Travel- BehaviorSurveyBot, developed in Voiceflow and deployed as a web application accessible through a public link and QR code (Figure 2). Respondents completed the survey on their own devices through a guided dial…

</details>

<details><summary>q14 — Which travel modes responded most clearly to adverse weather and snow?</summary>

**New answer:**

Cycling was particularly sensitive to adverse weather, while public transit use increased substantially under the Snowy scenario. [4]

**Assessment:** Correct adverse-weather sensitivity for cycling and increased transit use under snow.

**Source [4] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> Travel-behavior research increasingly combines digital data collection with predictive modeling, yet these stages are often developed and evaluated separately. This study proposes and applies a three-agent workflow integrating conversational data collection, structured data processing, and behaviora…

</details>

<details><summary>q15 — How did random forest and the best text-only zero-shot LLM compare on five-class travel prediction?</summary>

**New answer:**

Random forest achieved a five-class accuracy of 69.6% [4], while the best text-only zero-shot LLM reached 69.9% [4].

**Assessment:** Correctly associates 69.6% with random forest and 69.9% with text-only zero-shot LLM; source [4] contains both.

**Source [4] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> Travel-behavior research increasingly combines digital data collection with predictive modeling, yet these stages are often developed and evaluated separately. This study proposes and applies a three-agent workflow integrating conversational data collection, structured data processing, and behaviora…

</details>

<details><summary>q16 — What was the best vision-based travel mode prediction accuracy?</summary>

**New answer:**

71.5% [3].

**Assessment:** Correct 71.5% result, though the bare number would benefit from a metric label.

**Source [3] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> labeled travel-behavior data are limited, although the small sample and multiple evaluated configurations require cautious interpretation. The five-class task provides a clearer assessment of whether models distinguish among individual modes. Vision-based prediction.The vision experiments directly c…

</details>

<details><summary>q17 — What is the patient-oriented medical report interpretation task introduced with G-CARL?</summary>

**New answer:**

Patient-Oriented Medical Report Interpretation (PMRI), a novel open-ended multimodal generation task that requires models to explain medical reports in accurate and accessible language based on a user’s query and dialogue history [3].

**Assessment:** Correct patient-oriented explanation task, grounded in report images, user query and dialogue history.

**Source [3] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> Personalized interpretation of medical reports has emerged as an increasingly important need among patients. Addressing this need requires both evidence-grounded medical factualityandcontext-dependentpatientcommunication,yetexisting medical vision-language tasks do not adequately capture these dual …

</details>

<details><summary>q18 — How does G-CARL supervise factuality and response coverage?</summary>

**New answer:**

G-CARL supervises factuality through a retrieval-grounded branch that verifies report-grounded medical claims against evidence retrieved from a multi-source medical datastore, producing the factuality reward Rfact [4]. It also supervises response coverage through a case-specific checklist branch that constructs a weighted checklist through MLLM generation followed by clinician-guided refinement, and evaluates the checklist coverage of the generated response to produce the checklist reward Rcheck [4].

**Assessment:** Now covers multi-source medical retrieval and claim verification plus the weighted case-specific checklist, all supported by source [4]. This fills the earlier retrieval omission.

**Source [4] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> BuiltuponGRPO,G-CARLoptimizesthepolicymodelwith three reward signals, as illustrated in Fig. 2. Given uploaded medical report imagesI, the dialogue historyh, and a user queryq,thepolicymodelfirstgeneratescandidateresponses. The retrieval-grounded branch then supervises medical factuality by verifyin…

</details>

<details><summary>q19 — What benchmark and evaluation dimensions accompany G-CARL?</summary>

**New answer:**

The benchmark accompanying G-CARL is MMedReport, a real-world multimodal PMRI benchmark, and CMB, an external benchmark that evaluates both medical QA accuracy and the professionalism of open-ended clinical interpretation. 

The evaluation dimensions for PMRI responses include Medical Accuracy (Accuracy), Demand Satisfaction (Satisfaction), and Expression Quality (Expression), scored according to a clinician-authored rubric ranging from -2 to 3. 

Additionally, objective evaluation metrics include claim-level Precision and checklist-level Recall, measuring factual correctness and coverage of case-specific requirements, respectively. [1][2][3]

**Assessment:** Correct MMedReport benchmark and all three evaluation dimensions. Extra CMB and precision/recall details are supported. Citations remain bunched at the end instead of following each factual sentence.

**Source [1] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> Datasets.We evaluate G-CARL primarily on(1)MMedReport, a real-world multimodal PMRI benchmark collectedfromonlinehealthcareconsultations.Itcontains2,450 instances, each including dialogue history, a user query, uploaded medical report images, and clinician-verified reference annotations. All instanc…

**Source [2] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> (OpenAI 2025) as the judge by strictly following these evaluation criteria. The complete judging protocol is provided in theAppendix Bto facilitate reproducibility. (2) Objective Evaluation. In addition to holistic assessment, we report claim-levelPrecisionand checklist-levelRecall. Precisionisdefin…

</details>

<details><summary>q20 — What did clinician pairwise preferences find about G-CARL?</summary>

**New answer:**

G-CARL was consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction, with preference margins of 136:85 and 106:64, respectively. [1]

**Assessment:** Correct clinician preference finding and supported margins 136:85 and 106:64.

**Source [1] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> andproducesclinicallyaccurateinterpretations.Similarly,in therightexample,SFTfailstoincorporatetheuser’ssmoking andalcoholcessationhistory,whileMLLM-as-a-Judgefails toaddresstheuser’sprimaryconcernofwhethertocontinue themedication.Incontrast,ourmethodeffectivelyintegrates historical context, directl…

</details>

<details><summary>q21 — Which papers address auditable AI workflows derived from user activity or survey responses?</summary>

**New answer:**

The provided papers address auditable AI workflows derived from user activity or survey responses.

* An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction addresses auditable AI workflows derived from survey responses. The paper proposes a multi-agent framework that integrates data collection, data processing, and behavioral modeling for commuter mode-choice research, with a focus on stateful conversational survey administration and structured response capture. [4][3]
* Inducing Task Models from Computer-Use Traces addresses auditable AI workflows derived from user activity. The paper introduces TASKMODELINDUCTION(TMI), which discovers latent tasks in unconstrained traces and induces task models pairing hierarchical objective models with procedure models of control flow. [1]

**Assessment:** False refusal is resolved: identifies both the survey workflow and TMI, explains their different inputs, and cites both papers. Citation placement is still mostly at bullet ends.

**Source [4] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> a commonmulti-agent workflowremains largely unexplored for travel behavior and individual mode-choice modeling. This study addresses these gaps through a multi-agent workflow that connects AI-assisted behavioral data collection with travel-choice modeling and LLM-based prediction. The paper makes fo…

**Source [3] — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction:**

> Throughout this paper,active data collectionrefers to stateful conversational survey administration with predefined interaction logic and structured response Table 1: Roles, interfaces, and supported functions of the three agents. Agent Main inputs Authorized functions Main outputs Data Collection A…

</details>

<details><summary>q22 — Which papers evaluate AI agents or model specialists using benchmark tasks and costs?</summary>

**New answer:**

Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation evaluates both Pandora's Router and Pandora's Bidder using three domains: a multi-LLM routing benchmark, retrieval-augmented specialists for factoid QA, and inference-time computation for mathematical reasoning. The evaluation assesses their allocative performance relative to baselines across a range of estimation costs [5].

AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement presents a benchmark that evaluates AI agents' ability to design training algorithms. In each task, an agent has 4 hours to rewrite the training algorithm, and its code is then rerun from scratch for up to 12 hours and scored by a fixed evaluator [3].

**Assessment:** Correctly identifies Pandora routing and AI4AI-Bench, their evaluation domains and budgets/costs. More detailed than the earlier answer; citations remain at paragraph ends.

**Source [5] — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation:**

> Bidder: each specialist faces a posted price—the current best competing offer—and must decide whether to invest in a more accurate self-assessment before claiming the query. This corresponds to a single stage of the ascending-price mechanism with costly preference elicitation studied by Parkes (2005…

**Source [3] — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement:**

> AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement Yizhe Chi , Wenyi Li, Deyao Hong, Xiaoqiu Wang, Mingju Gao, Kaisen Yang, Bingxiang He, Y oujie Zheng, Calvin Xiao, Qinhuai Na Navers Lab, Einsia.AI T singhua University Project Lead Corresponding Author Abstrac…

</details>

<details><summary>q23 — Which papers discuss evidence-grounded explanations and auditable task models?</summary>

**New answer:**

The provided papers discuss the following concepts related to evidence-grounded explanations and auditable task models:

* Evidence-grounded explanations:
	+ Paper 2608.20331 discusses patient-oriented medical report interpretation (PMRI) as an evidence-bounded generation task that requires explanations grounded in reports and reliable clinical knowledge. [4]
	+ Paper 2608.20331 proposes a reinforcement learning framework, G-CARL, that decomposes reward supervision through retrieval-grounded claim verification and case-specific checklist guidance. [3]
* Auditable task models:
	+ Paper 2608.20319 introduces TASKMODELINDUCTION (TMI), which induces task models pairing a hierarchical objective model of recursive goal decomposition with a procedure model of the control flow that organized the execution. [2]
	+ Paper 2608.20319 describes the objective model as a hierarchy that explains why the observed activities were performed, and the procedure model as capturing how the execution of the task was organized in time. [1]
	+ Paper 2608.20319 also describes the procedure model as inducing a tree using three observable operators: sequence, selection, and iteration. [5]

**Assessment:** Now covers both G-CARL and TMI, but adds a false detail: says the three observable operators are sequence, selection and iteration. Source [5] explicitly excludes selection from the output and specifies SEQ, FOR and WHILE. The answer is also long and exposes paper IDs instead of readable titles.

**Source [5] — Inducing Task Models from Computer-Use Traces:**

> …(while). The trace records only the path enacted, not unchosen alternatives, so selection is not represented in the output. Decompose the observed activity trace into a tree using the three observable operators below. The three operators are the complete and closed vocabulary: -`SEQ`: steps in fixed…

**Source [3] — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation:**

> In this paper, we present PMRI as a challenging yet underexplored task that requires both evidence-grounded medical factualityandcontext-dependentpatientcommunication.We propose G-CARL, a reinforcement learning framework that decomposes reward supervision through retrieval-grounded claim verificatio…

</details>

<details><summary>q24 — What is KPMG's 2026 revenue forecast?</summary>

**New answer:**

The provided papers do not address this question.

**Assessment:** Correctly declines the unsupported KPMG revenue forecast.

</details>

<details><summary>q25 — Which of these papers reports the stock price of NVIDIA on October 14, 2026?</summary>

**New answer:**

The provided papers do not address this question.

**Assessment:** Correctly declines the unsupported NVIDIA stock price.

</details>


In [8]:
ANSWERS = ROOT / 'evaluation/results/answers_llama31_8b_full.json'
CHECKLISTS = ROOT / 'evaluation/answer_checklists.json'
OUTPUT = ROOT / 'evaluation/results/manual_review_llama31_8b_full.json'
MAX_WORDS = 120


In [9]:
display(Markdown((ROOT / 'evaluation/results/llama31_8b_full_assessment.md').read_text()))


# Llama 3.1 8B — full 25-question assessment

**Conclusion:** promising for direct factual questions, but still unreliable on some comparisons and cross-paper questions. Keep it as an optimization candidate with manual review; this run does not establish it as a dependable unattended system. The default model is unchanged.

**Scope:** generated the remaining 19 answers and combined them with the previous six unchanged records. Same model weights, polished prompt, generation settings and frozen retrieval inputs were verified. No LLM judge was used. Assessments below were made by the assistant against saved passages, not by an independent human reviewer. This is a small, single-run benchmark.

| Content assessment | Count |
| --- | --- |
| Complete | 17 |
| Partial | 4 |
| Major issue | 2 |
| Correct abstention | 2 |

**Rubric:** Complete means the core requested factual content is supported and adequately answered; it does not mean every citation or formatting rule passes. Partial means useful correct content with a material required point missing. Major issue means the central question is unanswered, incorrectly answered, or unjustifiably refused. Correct abstention applies only to the two unsupported benchmark questions. The checklist is a guide; optional explanatory detail is not treated as a factual error (for example q06 still supplies both requested budgets).

**Evidence and timing:** 22/23 answerable questions received a response containing citations; q21 was wrongly refused. Both unsupported questions were correctly declined. The retrieved set includes all benchmark-labeled papers for all 23 answerable questions (paper-level recall, not proof of passage relevance). New generation took 171.3s; all 25 recorded calls total 242.6s across the two runs. No repeated/warmed latency benchmark or 25-question Llama 3.2 comparison was run.

## What the broader test revealed

- **Numerical facts:** correctly reports 10 repositories/families, 4/12-hour budgets, normalized-score anchors, 0.974 grouping agreement, 74.9% reconstruction, and 92/454 survey counts. It correctly interprets the 30% TMI gain as relative, from 14.29 to 18.57. Travel prediction results 69.6%, 69.9%, and 71.5% are correctly associated with their models.
- **Remaining comparison failure:** q08 still misses the actual 0.226 versus 0.126 performance comparison, despite the direct evidence being supplied.
- **Cross-paper coverage:** q22 succeeds; q21 wrongly refuses; q23 covers only TMI and omits G-CARL. Thus this small cross-paper subset has one complete, one partial, and one major failure.
- **Abstention:** both unsupported questions are declined, but the false refusal on q21 shows that abstaining is not uniformly reliable.
- **Citations:** presence is much better than the earlier smaller-model pilot, but some sentences/bullets remain uncited or cite sources that omit a detail. Do not equate a citation count with verified support.
- **Automatic checks:** q05, q06, q07 and q19 have number warnings caused by spelled-out values or PDF extraction artifacts. q08 has a genuine missing-result problem; its 263 denominator warning is a separate citation-location issue. Manual inspection is necessary.

**Next optimization:** test context ordering and per-paper coverage on q08/q21/q23 while keeping the model and prompt fixed. Put the direct comparison passage first for q08; for cross-paper questions, present clearly named, distinct papers with their supporting passages. Treat this as an experiment, not an established fix. Improve numeric-warning handling separately so formatting artifacts do not obscure real errors.

## Per-question assessment

| ID | Result | Finding |
| --- | --- | --- |
| q01 | Partial | Correct value-of-information, stop and selection rules; inspection in reservation-price order is still missing. Sentence-level citations are present. |
| q02 | Partial | Correct centralized/decentralized distinction and local-surplus caveat. Does not explicitly explain Router’s global-welfare objective; exceeds the three-sentence instruction. |
| q03 | Complete | Correct three domains. Citations appear only after the last item, so per-bullet citation coverage still needs improvement. |
| q04 | Complete | Correctly explains that noisy competing estimates can increase the strategic specialist’s utility at others’ expense. |
| q05 | Complete | Correctly reports 10 repositories and 10 algorithm families. Source [2] spells both values as ten; the automatic digit-matching warning is not an answer error. |
| q06 | Complete | Correctly gives the requested budgets: 4 hours on one B300 and 12 hours for replay. It could clarify that replay starts from scratch; calling 12 hours a budget is not a claim that every replay lasts exactly 12 hours. The source joins 12hours, causing a mechanical warning. |
| q07 | Complete | Correctly associates 0, 0.1, and 1.0 with the uninformative model, repository baseline and task optimum. Split decimals/joined text in the PDF cause mechanical warnings. |
| q08 | Major issue | Fails to report the requested performance scores 0.226 versus 0.126. Gives submission shares instead. The non-causal caveat is supported, but does not answer the main comparison. The supplied passages contain the missing scores. |
| q09 | Complete | Correct objective and procedure models. Citation [3] supports the content, but appears only at the paragraph end. |
| q10 | Complete | Correctly reports 0.974 agreement with ground-truth groupings. Copies the paper’s first-person phrase “Our method”; use the method name in a reader-facing answer. |
| q11 | Complete | Correctly reports reconstruction of 74.9% of observed execution steps. |
| q12 | Complete | Correctly distinguishes a 30% relative gain from a percentage-point increase, giving the baseline 14.29 and result 18.57. |
| q13 | Complete | Correct participant and observation counts: 92 and 454. The McGill qualifier is in sources [1]/[2], while the answer only cites [4] after the second sentence; citation placement is incomplete. |
| q14 | Complete | Correctly identifies cycling as sensitive to adverse weather and increased transit use under snow. |
| q15 | Complete | Correctly assigns 69.6% to random forest and 69.9% to the best text-only zero-shot LLM. It makes no unsupported significance claim; these remain descriptive results. |
| q16 | Complete | Correct 71.5% result with relevant citations. The bare number answers the question, though naming five-class accuracy would make it more self-contained. |
| q17 | Complete | Correctly explains accurate, accessible, patient-oriented multimodal report interpretation using user query and dialogue history, with citations. |
| q18 | Partial | Correctly explains atomic claim verification and weighted case-specific checklists, with sentence-level citations. Does not explicitly mention multi-source retrieval/datastore support, a required checklist detail. |
| q19 | Complete | Correct MMedReport benchmark and medical accuracy, demand satisfaction and expression quality dimensions. Extra CMB details are supported. Some sentences lack adjacent citations; the 2,450 warning is caused by the joined PDF token, not a wrong value. |
| q20 | Complete | Accurately describes clinician preference for G-CARL in medical accuracy and demand satisfaction; quoted preference margins are in source [1]. |
| q21 | Major issue | Incorrectly refuses. Source [1] describes auditable task models from user activity, and sources [2]–[4] describe the auditable survey workflow. Both benchmark-relevant papers were retrieved. |
| q22 | Complete | Correctly identifies AI4AI-Bench and the Pandora routing paper and relates them to agent/specialist evaluation. Cite at the end of each factual claim rather than relying on leading passage numbers. |
| q23 | Partial | Cites two passages from the SAME TMI paper as the answer, and omits the G-CARL paper’s evidence-grounded interpretation work. Both papers are available in the retrieved context. |
| q24 | Correct abstention | Correctly declines the unsupported KPMG revenue forecast question. No revenue figure is invented. |
| q25 | Correct abstention | Correctly declines the unsupported NVIDIA stock-price question. No price is invented. |

## Answers and source checks

<details><summary>q01 — How does Pandora's Router decide when an expensive specialist estimate is worth paying for?</summary>

**Generated answer:**

Pandora's Router computes reservation prices that quantify the expected value-of-information for querying a more costly and accurate value estimate 𝑔𝑚(𝑥), e.g., by running the specialist for a few reasoning tokens and checking if it is on the right track [1]. The system stops when no remaining query is worth its cost [1]. The maximizer of the observed 𝑔𝑚 is chosen [1].

**Assessment:** Correct value-of-information, stop and selection rules; inspection in reservation-price order is still missing. Sentence-level citations are present.

**Source 1** (shortened literal excerpt):

> …Pandora’s Router Centralized policy Inspect specialists𝑔𝑚(𝑥) in reservationprice order, stopping when no remaining query is worth its cost. Choose the maximizer of the observed𝑔𝑚. Pandora’s Bidder Decentralized policy The high bidder is selected in an auction. Each specialist refines its own value estimate only when the market price makes inspection worthwhile. The same principle yields both a centralized and decentralized pol…

</details>

<details><summary>q02 — What is the difference between Pandora's Router and Pandora's Bidder?</summary>

**Generated answer:**

Pandora's Router and Pandora's Bidder differ in their approach to allocating queries. Pandora's Router is a centralized policy that inspects specialists' value estimates in reservation price order, stopping when no remaining query is worth its cost [1]. In contrast, Pandora's Bidder is a decentralized policy that uses an auction mechanism to select the high bidder, with each specialist refining its own value estimate only when the market price makes inspection worthwhile [1]. Additionally, Pandora's Bidder measures regret versus an oracle that knows the true value for free, and its performance is affected by the price being inaccurate, leading to a trade-off between local surplus and overall allocative efficiency [3].

**Assessment:** Correct centralized/decentralized distinction and local-surplus caveat. Does not explicitly explain Router’s global-welfare objective; exceeds the three-sentence instruction.

**Source 3** (shortened literal excerpt):

> …fficiency on the table at low costs by never investing in refinement. With decentralization, however, maximizing local surplus does not alwaysleadtobetteroverallallocativeefficiency. Forexample, whenthepriceisinaccurate(setusingthe cheap estimate𝑓 rather than𝑔 for the𝑀− 1competitors) the strategic bidder captures higher surplus at the cost of overall allocative efficiency (see additional results in Appendix D.2). Overpriced queries Pandora’s AI Model Routing Box: Efficient Allocation with Costly…

</details>

<details><summary>q03 — In which three domains was the costly-value-estimation routing policy evaluated?</summary>

**Generated answer:**

The three domains where the costly-value-estimation routing policy was evaluated are:  1. a multi-LLM routing benchmark (EmbedLLM) 2. retrieval-augmented specialists for factoid QA 3. inference-time computation for mathematical reasoning [3][4]

**Assessment:** Correct three domains. Citations appear only after the last item, so per-bullet citation coverage still needs improvement.

**Source 3** (shortened literal excerpt):

> … refine their self-assessments before accepting a posted price (5). • We evaluate both frameworks (see Figure 1) across three domains: a multi-LLM routing benchmark (EmbedLLM), retrieval-augmented specialists for factoid QA, and inference-time computation for mathematical reasoning. In each setting, both Pandora’s Router and Pandora’s Bidder are able to gracefully trade-off expensive and cheap estimates to achieve strong allocative performance relative to baselines across a range of estimation c…

</details>

<details><summary>q04 — What limitation appears when competing specialists' estimates are noisy?</summary>

**Generated answer:**

When competing estimates are noisy, value-of-information reasoning can increase the strategic specialist's utility at the expense of others. [1]

**Assessment:** Correctly explains that noisy competing estimates can increase the strategic specialist’s utility at others’ expense.

**Source 1** (shortened literal excerpt):

> …ve estimation, while querying the expensive estimator far less often. In the decentralized setting, value-of-information reasoning improves allocative efficiency when competing estimates are accurate; when competing estimates are noisy, however, it can increase the strategic specialist’s utility at the expense of others.…

</details>

<details><summary>q05 — How many frozen repositories and training algorithm families are in AI4AI-Bench?</summary>

**Generated answer:**

The AI4AI-Bench suite consists of 10 frozen research repositories, which cover 10 distinct families of training algorithm. [2]

**Assessment:** Correctly reports 10 repositories and 10 algorithm families. Source [2] spells both values as ten; the automatic digit-matching warning is not an answer error.

**Source 2** (shortened literal excerpt):

> …The suite is ten frozen research repositories, listed in Table 1 and chosen so that between them they cover ten families of training algorithm rather than ten instances of one: supervised fine-tuning, multi-turn agentic RL, on-policy distillation, Bradley–Terry reward modeling, preference optimization, diffusion RL, machine unlearning, discrete graph diffusion, weight averaging, and one-shot pruning. Each was admitted on three…

</details>

<details><summary>q06 — What is the AI4AI-Bench time budget for an agent and for the replay?</summary>

**Generated answer:**

The AI4AI-Bench time budget for an agent is 4 hours on one B300 GPU [2]. The time budget for the replay is 12 hours [2].

**Assessment:** Correctly gives the requested budgets: 4 hours on one B300 and 12 hours for replay. It could clarify that replay starts from scratch; calling 12 hours a budget is not a claim that every replay lasts exactly 12 hours. The source joins 12hours, causing a mechanical warning.

**Source 2** (shortened literal excerpt):

> …ive proxy metric that can be evaluated freely during development 2.2. The agent is given 4 hours on one B300 GPU and a single objective: improve the training algorithm implemented in the repository. Its submission is neither a number nor a trained model, but the repository’s source code after the agent’s modifications. After submission, the agent can no longer modify or interact with the code. The submitted repository is then run from scratch under a fixed budget, and the resulting model is scor…

**Source 2** (shortened literal excerpt):

> … a rewritten source C ′; it never evaluatesm. Execution ofC ′ under a verification budgetTv = 12hours yields a model Table 1 The ten AI4AI -Bench tasks.Each freezes a research repository and asks an agent to improve the training algorithm that repository applies to its own model.Starting modelis the modela0 the procedure begins with, and Evaluation metricthe quantity a fixed evaluator computes afterwards, with its direction. Between them the ten cover ten families of training algorithm rather th…

</details>

<details><summary>q07 — How are the incommensurable AI4AI-Bench task metrics normalized?</summary>

**Generated answer:**

The AI4AI-Bench task metrics are normalized to a scale where 0 is an uninformative model, 0.1 is the algorithm the repository ships, and 1.0 is the task optimum [2].

**Assessment:** Correctly associates 0, 0.1, and 1.0 with the uninformative model, repository baseline and task optimum. Split decimals/joined text in the PDF cause mechanical warnings.

**Source 2** (shortened literal excerpt):

> …incommensurable, every task is mapped onto one scale on which0is an uninformative model,0 .1is the algorithm the repository ships, and1.0 is the task optimum. Across 29 configurations of 6 systems on all 10 tasks the mean score is0.166, and the best system reaches0.250: even the strongest closes under a fifth of the distance between the algorithm that was already there and the optimum. The submissions show where that distance …

</details>

<details><summary>q08 — How do submissions that change the learning algorithm compare with those that do not?</summary>

**Generated answer:**

Submissions that change the learning algorithm (46.4% of 263 submissions) are compared to those that do not (53.6% of 263 submissions). The comparison is limited by the fact that the stronger systems are more likely to reach the learning procedure, so the gap is between submissions that go there and those that do not, rather than the effect of going there. [3]

**Assessment:** Fails to report the requested performance scores 0.226 versus 0.126. Gives submission shares instead. The non-causal caveat is supported, but does not answer the main comparison. The supplied passages contain the missing scores.

**Source 2** (shortened literal excerpt):

> …optimum. The submissions say where the remaining distance went. Of the 263 that change anything, 141 leave the learning procedure exactly as they found it and move budgets, checkpointing, hyperparameters and capacity instead. The 122 that do reach it — the objective, the supervision signal, the learning rule, the data — average0.226 against0 .126for the rest: the algorithmic layer is where the distance gets closed, and most submissions never go there. More reasoning effort moves agents tow…

</details>

<details><summary>q09 — What two kinds of task structure does TMI recover from computer-use traces?</summary>

**Generated answer:**

TMI recovers two kinds of task structure: objective models and procedure models. Objective models are hierarchical decompositions of the objective of a task into subobjectives, while procedure models are trace-grounded compositions of control-flow operators specifying how the execution of a task is organized. [3]

**Assessment:** Correct objective and procedure models. Citation [3] supports the content, but appears only at the paragraph end.

**Source 3** (shortened literal excerpt):

> …t∈ T, inducing a task model Mt that unifies two complementary axes. • Theobjective model Ot is a hierarchical decomposition of the objective of t into the subobjectives the user pursues. • Theprocedure model Pt is a trace-grounded composition of control-flow operators specifying how the execution of t is organized through sequencing and iteration. The task model Mt is structured as a tree, whose every node pairs an objective with a control-flow operator over its children.…

</details>

<details><summary>q10 — How well does task model induction group interleaved tasks?</summary>

**Generated answer:**

Our method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings [2].

**Assessment:** Correctly reports 0.974 agreement with ground-truth groupings. Copies the paper’s first-person phrase “Our method”; use the method name in a reader-facing answer.

**Source 2** (shortened literal excerpt):

> …m a dataset of recorded real human work sessions (Wang et al., 2025), our method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings and matches 74.9% of observed execution steps against 30.3% for the strongest workflow induction baseline. Using the induced task model to generate reusable agent skills, evaluated on heldout SkillLearnBench (Zhong et al., 2026) tasks, improves task accuracy by 30.0% over baseline. Our contributions are as follows. • We formalize the…

</details>

<details><summary>q11 — How much of observed execution does TMI reconstruct?</summary>

**Generated answer:**

TMI reconstructs 74.9% of the observed execution steps [4].

**Assessment:** Correctly reports reconstruction of 74.9% of observed execution steps.

**Source 4** (shortened literal excerpt):

> …m a dataset of recorded real human work sessions (Wang et al., 2025), our method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings and matches 74.9% of observed execution steps against 30.3% for the strongest workflow induction baseline. Using the induced task model to generate reusable agent skills, evaluated on heldout SkillLearnBench (Zhong et al., 2026) tasks, improves task accuracy by 30.0% over baseline. Our contributions are as follows. • We formalize the…

</details>

<details><summary>q12 — What was the held-out task accuracy gain from skills derived from TMI?</summary>

**Generated answer:**

The held-out task accuracy gain from skills derived from TMI was 30% relative accuracy gain, from 14.29 to 18.57, over the strongest baseline. [1]

**Assessment:** Correctly distinguishes a 30% relative gain from a percentage-point increase, giving the baseline 14.29 and result 18.57.

**Source 1** (shortened literal excerpt):

> … than skills generated from raw demonstrations or workflow summaries. As shown in Table 5, our model improves executability from 59.35 to 67.65 and held-out accuracy from 14.29 to 18.57 over the strongest baseline, a 30% relative accuracy gain. The expert-written skills score highest on skill coverage at 93.59 but reach 10.00 held-out accuracy, below every induced source, so skill coverage and held-out accuracy do not move together in this setting. We report the curated condition as a reference …

</details>

<details><summary>q13 — How many commuters and respondent-scenario observations did the weather travel survey collect?</summary>

**Generated answer:**

The study collected mode choices from 92 McGill student commuters. The chatbot-administered survey yielded 454 respondent–scenario observations [4].

**Assessment:** Correct participant and observation counts: 92 and 454. The McGill qualifier is in sources [1]/[2], while the answer only cites [4] after the second sentence; citation placement is incomplete.

**Source 2** (shortened literal excerpt):

> … ensured consistent presentation of the scenarios across respondents. The Phase 1 sample included 92 McGill student commuters.…

**Source 4** (shortened literal excerpt):

> …ta collection, structured data processing, and behavioral prediction. A chatbot-administered, image-augmented stated-preference survey collected mode choices from 92 student commuters across five predefined weather scenarios, yielding 454 respondent–scenario observations. Weather-related associations were analyzed using a multinomial logit model, while logistic regression and random forest provided machine-learning benchmarks. Nine locally deployed large language models (LLMs), ranging from 2 to…

</details>

<details><summary>q14 — Which travel modes responded most clearly to adverse weather and snow?</summary>

**Generated answer:**

Cycling was particularly sensitive to adverse weather, while public transit use increased substantially under the Snowy scenario [4].

**Assessment:** Correctly identifies cycling as sensitive to adverse weather and increased transit use under snow.

**Source 4** (shortened literal excerpt):

> … extended through persona, few-shot, and vision-based configurations. Cycling was particularly sensitive to adverse weather, while public transit use increased substantially under the Snowy scenario. Random forest achieved 69.6% five-class accuracy, while the best text-only zero-shot LLM reached 69.9% without task-specific fitting. Habitual travel information produced the most consistent improvement in LLM prediction, Expert framing generally outperformed Role-Play, and persona information was m…

</details>

<details><summary>q15 — How did random forest and the best text-only zero-shot LLM compare on five-class travel prediction?</summary>

**Generated answer:**

Random forest achieved a five-class accuracy of 69.6% [4], while the best text-only zero-shot LLM reached 69.9% [1].

**Assessment:** Correctly assigns 69.6% to random forest and 69.9% to the best text-only zero-shot LLM. It makes no unsupported significance claim; these remain descriptive results.

**Source 4** (shortened literal excerpt):

> … public transit use increased substantially under the Snowy scenario. Random forest achieved 69.6% five-class accuracy, while the best text-only zero-shot LLM reached 69.9% without task-specific fitting. Habitual travel information produced the most consistent improvement in LLM prediction, Expert framing generally outperformed Role-Play, and persona information was most beneficial when habitual travel information was unavailable. Few-shot prompting further improved five-class prediction for sev…

</details>

<details><summary>q16 — What was the best vision-based travel mode prediction accuracy?</summary>

**Generated answer:**

71.5% [2][3]

**Assessment:** Correct 71.5% result with relevant citations. The bare number answers the question, though naming five-class accuracy would make it more self-contained.

**Source 2** (shortened literal excerpt):

> …and the best vision-based configuration reached the highest observed five-class accuracy of 71.5%, indicating that visual context can provide additional predictive information for selected models. Taken together, the findings show that habitual travel information, prompting strategy, demonstrations, and visual context can meaningfully shape LLM-based travel-choice prediction. More broadly, the study demonstrates how conversational survey administration, structured data processi…

</details>

<details><summary>q17 — What is the patient-oriented medical report interpretation task introduced with G-CARL?</summary>

**Generated answer:**

Patient-oriented Medical Report Interpretation (PMRI) is a novel open-ended multimodal generation task that requires models to explain medical reports in accurate and accessible language based on a user's query and dialogue history. [3]  PMRI must align professional medical knowledge with patient-centered communication by transforming report evidence into accessible explanations, addressing user-specific concerns, and offering clinically cautious guidance. [4]  PMRI differs fundamentally from medical VQA (Task A) and conventional report generation (Task B) in that it produces long-form, patient-facing explanations grounded in multimodal reports and extended patient-doctor dialogue. [4]

**Assessment:** Correctly explains accurate, accessible, patient-oriented multimodal report interpretation using user query and dialogue history, with citations.

**Source 3** (shortened literal excerpt):

> …s gap, we introducePatient-orientedMedicalReportInterpretation(PMRI), a novel open-ended multimodal generation task that requires models to explain medical reports in accurate and accessible language based on a user’s query and dialogue history. Thesetwoobjectivesdifferfundamentallyintheirverifiability, yet remain tightly coupled, making them difficult to optimize jointly under conventional supervised fine-tuning and holistic reinforcement learning paradigms. To address this challenge, we propos…

</details>

<details><summary>q18 — How does G-CARL supervise factuality and response coverage?</summary>

**Generated answer:**

G-CARL supervises factuality through a retrieval-grounded claim reward that provides fine-grained supervision for medical factuality through atomic claim verification [1]. It also supervises response coverage through a case-specific checklist reward that explicitly supervises demand satisfaction and expression quality without relying on a single reference response [1]. This is achieved by constructing instance-specific weighted checklists through MLLM generation followed by clinician refinement [1].

**Assessment:** Correctly explains atomic claim verification and weighted case-specific checklists, with sentence-level citations. Does not explicitly mention multi-source retrieval/datastore support, a required checklist detail.

**Source 3** (shortened literal excerpt):

> …CARL, a grounded, checklist-aligned reinforcementlearningframeworkthatcombinesmulti-source retrieval for atomic claim verification with context-aware, instance-specific weighted checklists for response coverage, providing structured supervision for factuality, user-demand satisfaction, and expression quality without constraining response diversity. We further constructMMedReport, a realworld PMRI benchmark, along with a clinician-designed three-dimensionalevaluationprotocol.Extensiveexperiments …

</details>

<details><summary>q19 — What benchmark and evaluation dimensions accompany G-CARL?</summary>

**Generated answer:**

G-CARL is primarily evaluated on the MMedReport benchmark, which contains 2,450 instances of patient-oriented medical report interpretation. Additionally, it is evaluated on the CMB benchmark, which assesses both medical QA accuracy and the professionalism of open-ended clinical interpretation.  The evaluation dimensions for G-CARL on MMedReport include subjective evaluation along three clinician-defined dimensions: Medical Accuracy (Accuracy), Demand Satisfaction (Satisfaction), and Expression Quality (Expression). [1] On CMB, G-CARL is evaluated on QA accuracy and professionalism of open-ended clinical interpretation. [3]

**Assessment:** Correct MMedReport benchmark and medical accuracy, demand satisfaction and expression quality dimensions. Extra CMB details are supported. Some sentences lack adjacent citations; the 2,450 warning is caused by the joined PDF token, not a wrong value.

**Source 1** (shortened literal excerpt):

> … multimodal PMRI benchmark collectedfromonlinehealthcareconsultations.Itcontains2,450 instances, each including dialogue history, a user query, uploaded medical report images, and clinician-verified reference annotations. All instances undergo quality control, deidentification,and manualverificationto ensuredataquality andpatientprivacy.Detaileddatasetstatisticsareprovidedin Appendix A. Following the standard split, 2,200 instances areusedfortrainingand250forevaluation.Toassessbroader medical ca…

**Source 1** (shortened literal excerpt):

> …tive metrics. (1)Subjective Evaluation. Responses are evaluated along three clinician-defined dimensions:Medical Accuracy(Accuracy),Demand Satisfaction(Satisfaction), andExpression Quality(Expression). Each dimension is scored according to a detailed clinicianauthored rubric ranging from−2to3. We employ GPT- 5.2 (OpenAI 2025) as the judge by strictly following these evaluation criteria. The complete judging protocol is provided in theAppendix Bto facilitate reproducibility. (2) Objective Evaluat…

</details>

<details><summary>q20 — What did clinician pairwise preferences find about G-CARL?</summary>

**Generated answer:**

G-CARL was consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction, with preference margins of 136:85 and 106:64, respectively. [1]

**Assessment:** Accurately describes clinician preference for G-CARL in medical accuracy and demand satisfaction; quoted preference margins are in source [1].

**Source 1** (shortened literal excerpt):

> … the user’s question, and provides evidence-grounded recommendations. Human Preference Evaluation.We conduct a blind pairwise human preference study on 250 held-out cases. Three professional clinicians compare anonymized responses generated by G-CARL and the baselines across the three evaluation dimensions and overall preference, with majority voting used for the final decision. We additionally recruit 50 participants without medical training to evaluate the comprehensibility of the generated in…

</details>

<details><summary>q21 — Which papers address auditable AI workflows derived from user activity or survey responses?</summary>

**Generated answer:**

The provided papers do not address this question.

**Assessment:** Incorrectly refuses. Source [1] describes auditable task models from user activity, and sources [2]–[4] describe the auditable survey workflow. Both benchmark-relevant papers were retrieved.

**Source 1** (shortened literal excerpt):

> …Naturalistic computer-use traces, passively recorded screenshots and mouse or keyboard actions, are a valuable resource for deriving symbolic, auditable, and reusable models of how everyday work is done. Such models matter as computer-use agents enter real work, where agents need to learn how tasks are actually performed, and organizations need to audit and reuse that knowledge. However, inducing such task models is challengin…

**Source 2** (shortened literal excerpt):

> …ons, maintain task state, select from an authorized set of tools, and produce structured and auditable outputs. An agent is therefore not restricted to an LLM. Depending on the task, it may combine an LLM with deterministic rules, database operations, statistical software, machine-learning algorithms, or general-purpose programming tools. The proposed framework is consequently a multi-agent research workflow rather than an LLM-only pipeline. The termagentin this framework refers to a software ag…

</details>

<details><summary>q22 — Which papers evaluate AI agents or model specialists using benchmark tasks and costs?</summary>

**Generated answer:**

[3] and [4] evaluate AI agents or model specialists using benchmark tasks and costs. Specifically, [3] presents AI4AI-Bench, a benchmark that tests whether an agent can design training algorithms, and [4] presents Pandora's Router and Pandora's Bidder, which evaluate model specialists using benchmark tasks and costs in three domains: a multi-LLM routing benchmark, retrieval-augmented specialists for factoid QA, and inference-time computation for mathematical reasoning.

**Assessment:** Correctly identifies AI4AI-Bench and the Pandora routing paper and relates them to agent/specialist evaluation. Cite at the end of each factual claim rather than relying on leading passage numbers.

**Source 3** (shortened literal excerpt):

> …to how a run is executed apart from a change to how the model learns. We present AI4AI-Bench, 10 frozen research repositories spanning 10 training algorithm families. In each task, an agent has 4 hours on one B300 to rewrite the training algorithm; its code is then rerun from scratch for up to 12 hours and scored by a fixed evaluator hidden from the agent, against the repository’s original algorithm under the same procedure. Because the 10 metrics are incommensurable, every task is mapped onto o…

**Source 4** (shortened literal excerpt):

> …n self-assessment before accepting an offered price to claim a query. Experiments across three domains—a standard multi-LLM benchmark, retrieval-augmented specialists, and LLMs with variable inference-time reasoning—show that Pandora’s Router matches the routing quality of exhaustive estimation, while querying the expensive estimator far less often. In…

</details>

<details><summary>q23 — Which papers discuss evidence-grounded explanations and auditable task models?</summary>

**Generated answer:**

[1] and [2] discuss evidence-grounded explanations and auditable task models. [1] specifically focuses on inducing task models from computer-use traces, including objective models that explain why activities were performed and procedure models that capture how the execution of the task was organized in time. [2] introduces TASKMODELINDUCTION (TMI), which induces task models pairing hierarchical objective models with procedure models, and demonstrates its effectiveness in recovering interleaved tasks and reconstructing execution steps.

**Assessment:** Cites two passages from the SAME TMI paper as the answer, and omits the G-CARL paper’s evidence-grounded interpretation work. Both papers are available in the retrieved context.

**Source 1** (shortened literal excerpt):

> …res understanding the two types of evidence it carries, procedure and objective. Constructing a task model thus means inducing an objective model Ot and a procedure model Pt from the activities Et of each discovered task t, then reconciling them into the task model Mt (Figure 2(c)). Each is induced independently, so that its structure is resolved under its own evidence before being constrained by the other. Objective model.The objective model is the hierarchy that explains why the observed activ…

**Source 3** (shortened literal excerpt):

> …e-grounded medical factualityandcontext-dependentpatientcommunication.We propose G-CARL, a reinforcement learning framework that decomposes reward supervision through retrieval-grounded claim verification and case-specific checklist guidance. ExtensiveexperimentsonthenewlyconstructedMMedReport benchmark, together with clinician-authored evaluation and human preference studies, show that G-CARL consistently improves the quality of patient-oriented medical report interpretation.Wehopethisworklayst…

</details>

<details><summary>q24 — What is KPMG's 2026 revenue forecast?</summary>

**Generated answer:**

The provided papers do not address this question.

**Assessment:** Correctly declines the unsupported KPMG revenue forecast question. No revenue figure is invented.

The benchmark marks this question unsupported; the supplied passages were inspected and do not provide the requested fact.

</details>

<details><summary>q25 — Which of these papers reports the stock price of NVIDIA on October 14, 2026?</summary>

**Generated answer:**

The provided papers do not address this question.

**Assessment:** Correctly declines the unsupported NVIDIA stock-price question. No price is invented.

The benchmark marks this question unsupported; the supplied passages were inspected and do not provide the requested fact.

</details>


## Run automatic checks

This is an offline operation. It regenerates the worksheet from the saved answers; it does not change the answers or their sources.


In [10]:
report = manual_review.main(SimpleNamespace(
    answers=ANSWERS, checklists=CHECKLISTS, output=OUTPUT,
    benchmark=None, max_words=MAX_WORDS,
))


Checked 25 answers without model calls.
Report: /Users/alexnguyen/Documents/GitHub/KPMG-1C-ai-research-intelligence-agent-for-business-insight-translation/evaluation/results/manual_review_llama31_8b_full.md


## Review the answers

Check whether each required point is present **in the answer itself**, then verify it against the sources. Source previews are copied and truncated to 240 characters; expand the full passages when needed. A figure appearing only in a source is not covered by the answer.


In [11]:
display(Markdown(manual_review.render_report(report)))


# Answer review worksheet

25 answers · automatic checks complete · content awaits manual review.

Checklists are assistant-prepared and pending human approval. Automatic checks do not establish factual accuracy.

## q01 — Manual review needed

How does Pandora's Router decide when an expensive specialist estimate is worth paying for?

**Answer:** Pandora's Router computes reservation prices that quantify the expected value-of-information for querying a more costly and accurate value estimate 𝑔𝑚(𝑥), e.g., by running the specialist for a few reasoning tokens and checking if it is on the right track [1]. The system stops when no remaining query is worth its cost [1]. The maximizer of the observed 𝑔𝑚 is chosen [1].

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Explains that reservation prices/value of information determine whether a more accurate estimate is worth its cost.
- [ ] Explains inspection in reservation-price order, stopping when no remaining inspection is worthwhile, then selecting the best observed estimate.
- [ ] Answers about the centralized Router without attributing decentralized posted-price acceptance or specialist bidding to it.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 1**

> …order, stopping when no remaining query is worth its cost. Choose the maximizer of the observed𝑔𝑚. Pandora’s Bidder Decentralized policy The high bidder is selected in an auction. Each specialist refines its own value estimate only when…

**Source 3**

> The strategic specialist must then apply the same value-of-information (VoI) reasoning from Pandora’s Router to decide whether this price is worth accepting, and whether or not to pay to observe a refined estimate before making this…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

Pandora’s Router Centralized policy Inspect specialists𝑔𝑚(𝑥) in reservationprice order, stopping when no remaining query is worth its cost. Choose the maximizer of the observed𝑔𝑚. Pandora’s Bidder Decentralized policy The high bidder is selected in an auction. Each specialist refines its own value estimate only when the market price makes inspection worthwhile. The same principle yields both a centralized and decentralized policy. Figure 1\| Overview of thePandoraframework.The system balances cheap screening and costly reasoning for routing.(1)A concrete prompt𝑥 is received.(2)Multiple specialists 𝑚∈{ 1,...,𝑀} generate cheap, noisy value estimates𝑓𝑚(𝑥).(3)The system computesreservation pricesthat quantify the expected value-of-information for querying a more costly and accurate value estimate 𝑔𝑚(𝑥), e.g., by running the specialist for a few reasoning tokens and checking if it is on the right track. (4)This unified principle supports allocation by centralized routing and decentralized auctions. minus total estimation cost by deciding which expensive estimates𝑔𝑚(𝑥) are worth computing. Formally, let𝑋∈X be a random input prompt, and let𝑥 denote a realization. Conditional on the input 𝑋=𝑥 , specialist𝑚 samples an output𝑌𝑚∼𝑃 𝑚(·\|𝑥) ,𝑌𝑚∈Y , and receives a cost-adjusted reward 𝑅𝑚(𝑥,𝑌 𝑚). For notational convenience, we will write the cost-adjusted reward𝑅𝑚(𝑥,𝑌 𝑚) simply as𝑅𝑚. The reward may combine, for example, the graded quality of the output with the cost of producing it (which could be the inference cost, or API fees for tools called), and is

**Source 2 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

benchmark, retrieval-augmented specialists, and LLMs with variable inference-time reasoning—show that Pandora’s Router matches the routing quality of exhaustive estimation, while querying the expensive estimator far less often. In the decentralized setting, value-of-information reasoning improves allocative efficiency when competing estimates are accurate; when competing estimates are noisy, however, it can increase the strategic specialist’s utility at the expense of others.

**Source 3 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

We now extend Pandora’s Router to a decentralized setting in which the specialists control their own value estimates, and use them to participate in a marketplace. Pandora’s Router assumes that a single decision-maker controls which boxes to open and which specialist to select. In many settings, however, the specialists themselves are better positioned to estimate their own value: a retrieval-augmented specialisthasaccesstoitsowncorpus,amathspecialistcanexecutepartialcomputations,andadomainexpert may have proprietary benchmarks. None of these resources need be visible to a centralized router. Decentralization also has practical advantages: new specialists can join without retraining the routing model, and the cost of value estimation is borne by the specialists rather than the platform. As an alternative to a centralized router, consider a market-based mechanism where each specialist can purchase the right to answer a query. We formalize this via a posted-price mechanism corresponding to a single stage of the ascending-price framework with costly preference elicitation studied by Parkes (2005). For simplicity, we consider a leave-one-out setting with a single strategic specialist. A platform collects nonstrategic value estimates from𝑀− 1specialists and posts the best estimate as the price at which the remaining specialist can claim the query. The strategic specialist must then apply the same value-of-information (VoI) reasoning from Pandora’s Router to decide whether this price is worth accepting, and whether or not to pay to observe a refined estimate before making this decision. Concretely, for a given query𝑥, the platform solicits𝐺𝑗 from each of the𝑀− 1nonstrategic specialists and sets the

**Source 4 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

Bidder: each specialist faces a posted price—the current best competing offer—and must decide whether to invest in a more accurate self-assessment before claiming the query. This corresponds to a single stage of the ascending-price mechanism with costly preference elicitation studied by Parkes (2005); analyzing this simple setting isolates core components of decentralized prompt allocation under private value estimation with profit incentives. Core contributions of this work As AI systems with diverse capabilities proliferate, reasoning about the economics of model selection becomes increasingly relevant. To our knowledge, this is the first paper to explicitly connect model routing with the classical Pandora’s Box problem from economics and operations research. Specifically, this paper makes three main contributions: • We formalize model routing with costly value estimation as an instance of Pandora’s Box with non-obligatory inspection (Doval, 2018), yielding a reservation-price-based policy. Under a Gaussian signal model, this policy has a closed form that is straightforward to compute (4). • We extend to a decentralized setting where specialists use value-of-information reasoning to decide whether to refine their self-assessments before accepting a posted price (5). • We evaluate both frameworks (see Figure 1) across three domains: a multi-LLM routing benchmark (EmbedLLM), retrieval-augmented specialists for factoid QA, and inference-time computation for mathematical reasoning. In each setting, both Pandora’s Router and Pandora’s Bidder are able to gracefully trade-off expensive and cheap estimates to achieve strong allocative performance relative to baselines across a range of estimation costs.

Source references exist; whether they support each claim still needs review.

Answer is 61 words; the review target is at most 120.


</details>

## q02 — Manual review needed

What is the difference between Pandora's Router and Pandora's Bidder?

**Answer:** Pandora's Router and Pandora's Bidder differ in their approach to allocating queries. Pandora's Router is a centralized policy that inspects specialists' value estimates in reservation price order, stopping when no remaining query is worth its cost [1]. In contrast, Pandora's Bidder is a decentralized policy that uses an auction mechanism to select the high bidder, with each specialist refining its own value estimate only when the market price makes inspection worthwhile [1]. Additionally, Pandora's Bidder measures regret versus an oracle that knows the true value for free, and its performance is affected by the price being inaccurate, leading to a trade-off between local surplus and overall allocative efficiency [3].

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Explicitly contrasts the centralized Router with decentralized specialists using Bidder.
- [ ] Explains that Router accounts for overall costs/global welfare, whereas a bidder optimizes its own surplus, which can conflict with overall efficiency.
- [ ] If discussing high inspection costs, preserves the option to use a cheap estimate OR decline a posted price; does not state that high costs always cause refusal.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 1**

> …Pandora’s Bidder Decentralized policy The high bidder is selected in an auction. Each specialist refines its own value estimate only when the market price makes inspection worthwhile. The same principle yields both a centralized and…

**Source 3**

> …surplus does not alwaysleadtobetteroverallallocativeefficiency. Forexample, whenthepriceisinaccurate(setusingthe cheap estimate𝑓 rather than𝑔 for the𝑀− 1competitors) the strategic bidder captures higher surplus at the cost of overall…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

Pandora’s Router Centralized policy Inspect specialists𝑔𝑚(𝑥) in reservationprice order, stopping when no remaining query is worth its cost. Choose the maximizer of the observed𝑔𝑚. Pandora’s Bidder Decentralized policy The high bidder is selected in an auction. Each specialist refines its own value estimate only when the market price makes inspection worthwhile. The same principle yields both a centralized and decentralized policy. Figure 1\| Overview of thePandoraframework.The system balances cheap screening and costly reasoning for routing.(1)A concrete prompt𝑥 is received.(2)Multiple specialists 𝑚∈{ 1,...,𝑀} generate cheap, noisy value estimates𝑓𝑚(𝑥).(3)The system computesreservation pricesthat quantify the expected value-of-information for querying a more costly and accurate value estimate 𝑔𝑚(𝑥), e.g., by running the specialist for a few reasoning tokens and checking if it is on the right track. (4)This unified principle supports allocation by centralized routing and decentralized auctions. minus total estimation cost by deciding which expensive estimates𝑔𝑚(𝑥) are worth computing. Formally, let𝑋∈X be a random input prompt, and let𝑥 denote a realization. Conditional on the input 𝑋=𝑥 , specialist𝑚 samples an output𝑌𝑚∼𝑃 𝑚(·\|𝑥) ,𝑌𝑚∈Y , and receives a cost-adjusted reward 𝑅𝑚(𝑥,𝑌 𝑚). For notational convenience, we will write the cost-adjusted reward𝑅𝑚(𝑥,𝑌 𝑚) simply as𝑅𝑚. The reward may combine, for example, the graded quality of the output with the cost of producing it (which could be the inference cost, or API fees for tools called), and is

**Source 2 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

and all methods except𝑓-only (which never queries𝑔) perform similarly. As𝑐𝑔 increases, the𝑔-always baseline continues to make a fixed number of queries regardless of cost, while𝑓-only still never queries at all. Pandora’s Router interpolates between these extremes, querying𝑔 only when the value of information exceeds the cost. The result is a total regret + inspection cost that tracks the lower envelope of the baselines across the full𝑐𝑔 range. As a secondary analysis, Figure 6 shows the cost-performance tradeoff offered by Pandora’s Router, Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation in terms of monetary costs and routing regret. The costs are computed from the Gemini API prices described in Section 2.2. When𝑐𝑔 = 0.001, we are in a setting in which users are eager to pay for queries that improve routing; here Pandora’s Router obtains regret that almost matches𝑔-only, while incurring much lower inspection costs. When𝑐𝑔 = 0.1, we are in a setting in which users are unwilling to pay for queries to𝑔. As a result, the Pandora’s Router never queries𝑔, and routing performance and cost match the𝑓-only baseline. In the MATH domain the aggregate results for Pandora and Margin are nearly identical. Because there are only two routing targets, the regret is determined largely by the decision about how many𝑔 values to query: with two queries, both methods always select the𝑔-maximizer; with zero queries both methods almost always select the𝑓-maximizer (except in very rare cases where the reserve price ordering is different from the ordering of𝑓). Because Margin inherits the query budget from Pandora, it is unsurprising that its overall

**Source 3 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

we measure the regret versus an oracle that knows𝐺𝑚 for free. Results.Figure3plotsallocativeefficiencyandbiddersurplusfordifferent 𝑐𝑔. Pandora’sBidderclosely tracks the lower envelope of the regret of the two baselines across the full𝑐𝑔 range. When𝑐𝑔 is low, the bidder frequently refines its estimate, winning queries where it holds a true advantage. As𝑐𝑔 increases, the bidder selectively defaults to the cheap estimate or declines the price, avoiding the cost collapse of the𝑔-always baseline. The𝑓-only baseline, conversely, leaves efficiency on the table at low costs by never investing in refinement. With decentralization, however, maximizing local surplus does not alwaysleadtobetteroverallallocativeefficiency. Forexample, whenthepriceisinaccurate(setusingthe cheap estimate𝑓 rather than𝑔 for the𝑀− 1competitors) the strategic bidder captures higher surplus at the cost of overall allocative efficiency (see additional results in Appendix D.2). Overpriced queries Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation offer insufficient expected profit to incentivize the bidder; declining them protects the bidder’s surplus but assigns the input to a less capable specialist. This tension is absent in the centralized setting, where the router internalizes all costs and optimizes global welfare directly. Extending Pandora’s Bidder to multi-roundascending-pricemechanisms(Parkes,2005)mayrecoversomeofthisefficiencyloss.

**Source 4 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

Bidder: each specialist faces a posted price—the current best competing offer—and must decide whether to invest in a more accurate self-assessment before claiming the query. This corresponds to a single stage of the ascending-price mechanism with costly preference elicitation studied by Parkes (2005); analyzing this simple setting isolates core components of decentralized prompt allocation under private value estimation with profit incentives. Core contributions of this work As AI systems with diverse capabilities proliferate, reasoning about the economics of model selection becomes increasingly relevant. To our knowledge, this is the first paper to explicitly connect model routing with the classical Pandora’s Box problem from economics and operations research. Specifically, this paper makes three main contributions: • We formalize model routing with costly value estimation as an instance of Pandora’s Box with non-obligatory inspection (Doval, 2018), yielding a reservation-price-based policy. Under a Gaussian signal model, this policy has a closed form that is straightforward to compute (4). • We extend to a decentralized setting where specialists use value-of-information reasoning to decide whether to refine their self-assessments before accepting a posted price (5). • We evaluate both frameworks (see Figure 1) across three domains: a multi-LLM routing benchmark (EmbedLLM), retrieval-augmented specialists for factoid QA, and inference-time computation for mathematical reasoning. In each setting, both Pandora’s Router and Pandora’s Bidder are able to gracefully trade-off expensive and cheap estimates to achieve strong allocative performance relative to baselines across a range of estimation costs.

Source references exist; whether they support each claim still needs review.

Answer is 109 words; the review target is at most 120.


</details>

## q03 — Manual review needed

In which three domains was the costly-value-estimation routing policy evaluated?

**Answer:** The three domains where the costly-value-estimation routing policy was evaluated are:  1. a multi-LLM routing benchmark (EmbedLLM) 2. retrieval-augmented specialists for factoid QA 3. inference-time computation for mathematical reasoning [3][4]

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Names the standard multi-LLM routing benchmark, such as EmbedLLM.
- [ ] Names retrieval-augmented specialists for factoid question answering.
- [ ] Names variable inference-time computation/reasoning, including mathematical reasoning.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 3**

> …reasoning to decide whether to refine their self-assessments before accepting a posted price (5). • We evaluate both frameworks (see Figure 1) across three domains: a multi-LLM routing benchmark (EmbedLLM), retrieval-augmented…

**Source 4**

> …Pandora’s Bidder, where specialists independently decide whether to invest in self-assessment before accepting an offered price to claim a query. Experiments across three domains—a standard multi-LLM benchmark, retrieval-augmented…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

expensive boxes), the inequality reverses. 2The value of the best committing policy that holds out multiple boxes cannot exceed that of the best single-box policy (Beyhaghi and Kleinberg, 2019), so it suffices to evaluate all𝑀 single-holdout policies, in addition to the obligatory inspection policy (no holdouts). These M + 1 policies are compared by simulating their performance via MC sampling. Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation The algorithm proceeds as follows. For each candidate holdout𝑚∈{ 0, 1,...,𝑀} , define the expected payoff of the committing policy that reserves box𝑚: 𝜈𝑚 =𝔼 " 𝑉 ˆ𝑚− ∑︁ 𝑗∈𝑂𝑚 𝑐𝑗 # ,(3) where𝑉 ˆ𝑚isthevalueofthe(random)boxthatthecommittingpolicyselects,and 𝑂𝑚 isthe(random)set ofboxesopenedbyPandora-OIon {1,...,𝑀}\{𝑚} withinitialoutsideoption 𝑣 =𝑢 backup𝑚 . Thecase𝑚= 0corresponds to running Pandora-OI on all𝑀 boxes with no holdout (𝑢backup0 =−∞ ). Because the heldoutboxisunobserved, weestimate 𝜈𝑚 viaMonteCarlo(MC)samples ˜𝑉𝑚∼𝑃 𝑚(·\|𝑥) ; wedenotetheMC estimate ˆ𝜈𝑚. We select𝑚 that maximizesˆ𝜈𝑚, using𝑆= 100samples. 3 At test time, we run Pandora-OI on{1,...,𝑀}\{𝑚 } with initial value𝑣 =𝑢 backup𝑚 , opening the actual boxes. We default to𝑚 if we open no boxes or if no realized value clears𝑢backup𝑚 ; otherwise, we select the best opened box.4

**Source 2 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

=(𝐹 1,...,𝐹 𝑀). Inspecting specialist𝑚 then reveals the costly information𝑍𝑚, from which the refined estimate𝐺𝑚 =𝑔 𝑚(𝑋,𝑍 𝑚) is computed. We denote the cost of this process as𝑐𝑔. For the additional cost of𝐺𝑚 to be justified, it should be more accurate, in the sense that on calibration data we have𝔼[(𝑅 𝑚−𝐺 𝑚)2]&lt;𝔼[(𝑅 𝑚−𝐹 𝑚)2]. The router must dynamically decide which specialists to inspect in order to maximize the expected reward of the final assignment minus the total cost of the realized𝐺𝑚 estimates. To make things more concrete before presenting our routing algorithms, we now describe three real-world domains where the two-tier estimation between𝑓𝑚 versus𝑔𝑚 arises organically, each with a qualitatively different source of costly information that is useful for deciding the best allocation. 2.1. Experimental domains and value estimators We explore several empirical settings, each featuring different types of specialists and value estimators. Inallcases, thecheapestimator 𝑓 embedsthepromptwithapretrainedencoder, retrievesthe 𝑘nearest prompts from a calibration set (measured by cosine similarity in embedding space), and returns the average reward of the retrieved neighbors as the value estimate. The costly estimator is then constructed by fine-tuning a small language model encoder to predict the reward from a tailored, domain-specific input context. We describe these below. Note that in each domain, we partition the data into three splits: training (for fitting value estimators), calibration (for estimating the value-of-information model parameters in Sections 4

**Source 3 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

Bidder: each specialist faces a posted price—the current best competing offer—and must decide whether to invest in a more accurate self-assessment before claiming the query. This corresponds to a single stage of the ascending-price mechanism with costly preference elicitation studied by Parkes (2005); analyzing this simple setting isolates core components of decentralized prompt allocation under private value estimation with profit incentives. Core contributions of this work As AI systems with diverse capabilities proliferate, reasoning about the economics of model selection becomes increasingly relevant. To our knowledge, this is the first paper to explicitly connect model routing with the classical Pandora’s Box problem from economics and operations research. Specifically, this paper makes three main contributions: • We formalize model routing with costly value estimation as an instance of Pandora’s Box with non-obligatory inspection (Doval, 2018), yielding a reservation-price-based policy. Under a Gaussian signal model, this policy has a closed form that is straightforward to compute (4). • We extend to a decentralized setting where specialists use value-of-information reasoning to decide whether to refine their self-assessments before accepting a posted price (5). • We evaluate both frameworks (see Figure 1) across three domains: a multi-LLM routing benchmark (EmbedLLM), retrieval-augmented specialists for factoid QA, and inference-time computation for mathematical reasoning. In each setting, both Pandora’s Router and Pandora’s Bidder are able to gracefully trade-off expensive and cheap estimates to achieve strong allocative performance relative to baselines across a range of estimation costs.

**Source 4 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

2026-08-21 Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation Adam Fisch, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate Larson, Jacob Eisenstein Google DeepMind Heterogeneous AI systems composed of multiple models, architectures, harnesses, or inference-time settings can improve quality and efficiency by routing queries to thespecialistwho can answer most effectively at the lowest cost. Routing requires estimating each specialist’s expected return, but this value estimation has a cost. Cheap estimators (e.g., embedding-based predictors) are fast but noisy, while accurate estimators (e.g., fine-tuned models with access to retrieval results or partial reasoning traces) are expensive. We formalize this tradeoff as an instance of Pandora’s Box, the classical problem of optimal search with costly inspection. Under a Gaussian signal model, the resulting policies have closed-form value-of-information expressions that determine, for each specialist and input, whether refining the value estimate is worth its cost. We call the centralized policy Pandora’s Router. We extend this to a decentralized setting, Pandora’s Bidder, where specialists independently decide whether to invest in self-assessment before accepting an offered price to claim a query. Experiments across three domains—a standard multi-LLM benchmark, retrieval-augmented specialists, and LLMs with variable inference-time reasoning—show that Pandora’s Router matches the routing quality of exhaustive estimation, while querying the expensive estimator far less often. In

Source references exist; whether they support each claim still needs review.

Answer is 26 words; the review target is at most 120.


</details>

## q04 — Manual review needed

What limitation appears when competing specialists' estimates are noisy?

**Answer:** When competing estimates are noisy, value-of-information reasoning can increase the strategic specialist's utility at the expense of others. [1]

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] States that noisy competing estimates can increase the strategic specialist’s own utility at the expense of others/overall allocation efficiency, rather than benefiting everyone.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 1**

> …setting, value-of-information reasoning improves allocative efficiency when competing estimates are accurate; when competing estimates are noisy, however, it can increase the strategic specialist’s utility at the expense of others.

**Source 2**

> …Allocation with Costly Value Estimation 1 Prompt𝑥 Find the remainder when7 103 is divided by13. 2 Cheap screening Specialist1 𝑓1(𝑥): modest prior Specialist2 𝑓2(𝑥): promising prior ... Specialist𝑀 𝑓𝑀(𝑥): modest prior All specialists get…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

benchmark, retrieval-augmented specialists, and LLMs with variable inference-time reasoning—show that Pandora’s Router matches the routing quality of exhaustive estimation, while querying the expensive estimator far less often. In the decentralized setting, value-of-information reasoning improves allocative efficiency when competing estimates are accurate; when competing estimates are noisy, however, it can increase the strategic specialist’s utility at the expense of others.

**Source 2 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

We begin with a more precise introduction to the problem of efficient allocation with costly value estimation. Suppose that a router must assign each prompt to the best specialist, but estimating specialist quality is itself costly. We assume that each specialist𝑚 has a cheap value estimate𝑓𝑚(𝑥) by default, while a more accurate but expensive estimate𝑔𝑚(𝑥) can be queried selectively; we make these estimators concrete in Section 2.1. The goal is to maximize the expected reward of the selected specialist Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation 1 Prompt𝑥 Find the remainder when7 103 is divided by13. 2 Cheap screening Specialist1 𝑓1(𝑥): modest prior Specialist2 𝑓2(𝑥): promising prior ... Specialist𝑀 𝑓𝑀(𝑥): modest prior All specialists get an initial cheap, noisy value estimate 𝑓𝑚(𝑥).

Source references exist; whether they support each claim still needs review.

Answer is 18 words; the review target is at most 120.


</details>

## q05 — Manual review needed

How many frozen repositories and training algorithm families are in AI4AI-Bench?

**Answer:** The AI4AI-Bench suite consists of 10 frozen research repositories, which cover 10 distinct families of training algorithm. [2]

**Automatic findings:** Check figures against their sources.

**Check these points:**

- [ ] Reports 10 frozen research repositories.
- [ ] Reports 10 training algorithm families.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 4**

> AI4AI-Bench, 10 frozen research repositories spanning 10 training algorithm families. In each task, an agent has 4 hours on one B300 to rewrite the training algorithm; its code is then rerun from scratch for up to 12 hours and scored by a…

**Source 2**

> The suite is ten frozen research repositories, listed in Table 1 and chosen so that between them they cover ten families of training algorithm rather than ten instances of one: supervised fine-tuning, multi-turn agentic RL, on-policy…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

are kept. An evaluator fixed before the first run, with no access to the agent’s workspace, scores them, and the cell takes the best of the three under the task’s direction; for the two tasks that do not train, the twelve hours are only a ceiling on the verification stage (2.3).Bottom:the ten frozen repositories, chosen so that between them they cover ten distinct families of training algorithm, each shown with the final metric its own evaluator computes and the asset that metric reads (Table 1). over one fixed dataset — so what wins is feature engineering and ensembling while the learning algorithm stays a library call the agent never edits, and nothing in the submission is inherited by the next training run . PostTrainBench states its task end to end, post-training a base model against a released instruction-tuned checkpoint, and its largest levers are which data to assemble and what to initialize from rather than the objective ; RSIBench-Data makes that emphasis deliberate, freezing the post-training stack so that only data-centric decisions vary . MLS-Bench comes closest, with 140 tasks in which an agent improves one component of an ML system , but the component boundary is handed to it and the score conflates execution-level improvements with changes to the learning algorithm. Closest of all isautoresearch, which hands the agent one training file and declares everything in it fair game, architecture and optimizer included ; but a five-minute run of a single script is not a research repository, and measured against classical optimizers the agent’s edits behave like hyperparameter search and lose to

**Source 2 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

The suite is ten frozen research repositories, listed in Table 1 and chosen so that between them they cover ten families of training algorithm rather than ten instances of one: supervised fine-tuning, multi-turn agentic RL, on-policy distillation, Bradley–Terry reward modeling, preference optimization, diffusion RL, machine unlearning, discrete graph diffusion, weight averaging, and one-shot pruning. Each was admitted on three properties the design depends on. It must ship a training algorithm its authors actually run, not a tutorial or a toy; it must ship a frozen starting model, so that there is something the agent is improvingfrom; and its metric must be recomputable, reproducibly, under a half-day budget on a single B300, or the twelve-hour measurement in 2.3 could not be run at all — let alone once per cell. Two of the ten do no training. Weight averaging combines a bank of checkpoints that are handed over as data, and one-shot pruning removes weights from a released model in a single pass; neither has a training horizon that could be extended or shortened. They are kept because the algorithmic question is just as real in them — which checkpoints to combine and how, which weights to remove and by what criterion — and because a suite that quietly dropped them would be a suite about training loops rather than about algorithm design. The protocol treats them differently, and so does the baseline. The ten metrics are incommensurable: an aesthetic score, a perplexity, a solve rate, a pass rate, an unlearning balanced score. They cannot be averaged as they

**Source 3 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

Recursive self-improvement compounds through the algorithmic link, and AI4AI-Bench is built to measure that link on its own: ten frozen research repositories, each asking an agent to improve the training algorithm it already applies to its own model, with four hours to write code, twelve to run what was written, and an evaluation the agent never sees. Across 29 configurations of six systems on all ten tasks the mean score is 0.166on a scale where the algorithm the repository already ships is0.1and the task optimum is1.0, and the best system reaches0.250. The submissions say where the rest of that distance went: of the 263 that changed anything, 141 never touch how the model learns at all, and the 122 that do average0.226against 0.126for the rest — the algorithmic layer is where the distance is closed, and most submissions never go to it. More reasoning effort mostly buys the willingness to go, taking that minority from8%of submissions to64% and the mean score from0.094to0 .196, which still leaves the strongest setting a tenth of the way past the algorithm it started from. What today’s agents do at the algorithmic link, then, is recover a competent default rather than design past one; whether that changes is the measurement this benchmark exists to keep taking.

**Source 4 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement Yizhe Chi , Wenyi Li, Deyao Hong, Xiaoqiu Wang, Mingju Gao, Kaisen Yang, Bingxiang He, Y oujie Zheng, Calvin Xiao, Qinhuai Na Navers Lab, Einsia.AI T singhua University Project Lead Corresponding Author Abstract.Recursive self-improvement (RSI) asks whether an AI system can improve the process that produces AI systems, so that the next system inherits the improvement. That process is the training algorithm: a better objective or update rule improves the compute-capability exchange rate for every subsequent run, including the one that produces the next agent. Whether RSI is feasible therefore turns on whether an agent can design training algorithms. No benchmark isolates that ability: existing suites are won by collecting data or by tuning hyperparameters, and none tells a change to how a run is executed apart from a change to how the model learns. We present AI4AI-Bench, 10 frozen research repositories spanning 10 training algorithm families. In each task, an agent has 4 hours on one B300 to rewrite the training algorithm; its code is then rerun from scratch for up to 12 hours and scored by a fixed evaluator hidden from the agent, against the repository’s original algorithm under the same procedure. Because the 10 metrics are incommensurable, every task is mapped onto one scale on which0is an uninformative model,0 .1is the algorithm the repository ships, and1.0 is the task optimum. Across 29 configurations of 6 systems

Source references exist; whether they support each claim still needs review.

Check figures against their sources.

Answer is 17 words; the review target is at most 120.


</details>

## q06 — Manual review needed

What is the AI4AI-Bench time budget for an agent and for the replay?

**Answer:** The AI4AI-Bench time budget for an agent is 4 hours on one B300 GPU [2]. The time budget for the replay is 12 hours [2].

**Automatic findings:** Check figures against their sources.

**Check these points:**

- [ ] Reports 4 hours on one B300 for the agent.
- [ ] Reports a fresh replay from scratch for up to 12 hours.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 2**

> …C is the repository’s source in its frozen state,a0 the model it starts from, q an inexpensive proxy available to the agent,m the final metric, andd∈ {↑,↓}its direction. An agent observes( C, a0, q)under an exploration budget Te = 4hours…

**Source 4**

> AI4AI-Bench, 10 frozen research repositories spanning 10 training algorithm families. In each task, an agent has 4 hours on one B300 to rewrite the training algorithm; its code is then rerun from scratch for up to 12 hours and scored by a…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

AI4AI. To our knowledge, AI4AI-Bench is the first benchmark to make that step the object of evaluation: it removes the agent, reruns the submitted source from a clean start, and classifies the patch to verify whether the gain came from execution, data, or a change to the training algorithm itself.

**Source 2 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

Each task presents an agent with a research repository, a base model to start from, and an inexpensive proxy metric that can be evaluated freely during development 2.2. The agent is given 4 hours on one B300 GPU and a single objective: improve the training algorithm implemented in the repository. Its submission is neither a number nor a trained model, but the repository’s source code after the agent’s modifications. After submission, the agent can no longer modify or interact with the code. The submitted repository is then run from scratch under a fixed budget, and the resulting model is scored by a predetermined evaluator (2.3). Formally, a task is a tuple(C, a0, q, m, d), where C is the repository’s source in its frozen state,a0 the model it starts from, q an inexpensive proxy available to the agent,m the final metric, andd∈ {↑,↓}its direction. An agent observes( C, a0, q)under an exploration budget Te = 4hours on one B300 GPU and returns a rewritten source C ′; it never evaluatesm. Execution ofC ′ under a verification budgetTv = 12hours yields a model Table 1 The ten AI4AI -Bench tasks.Each freezes a research repository and asks an agent to improve the training algorithm that repository applies to its own model.Starting modelis the modela0 the procedure begins with, and Evaluation metricthe quantity a fixed evaluator computes afterwards, with its direction. Between them the ten cover ten families of training algorithm rather than ten instances of one, and their metrics are

**Source 3 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

Recursive self-improvement compounds through the algorithmic link, and AI4AI-Bench is built to measure that link on its own: ten frozen research repositories, each asking an agent to improve the training algorithm it already applies to its own model, with four hours to write code, twelve to run what was written, and an evaluation the agent never sees. Across 29 configurations of six systems on all ten tasks the mean score is 0.166on a scale where the algorithm the repository already ships is0.1and the task optimum is1.0, and the best system reaches0.250. The submissions say where the rest of that distance went: of the 263 that changed anything, 141 never touch how the model learns at all, and the 122 that do average0.226against 0.126for the rest — the algorithmic layer is where the distance is closed, and most submissions never go to it. More reasoning effort mostly buys the willingness to go, taking that minority from8%of submissions to64% and the mean score from0.094to0 .196, which still leaves the strongest setting a tenth of the way past the algorithm it started from. What today’s agents do at the algorithmic link, then, is recover a competent default rather than design past one; whether that changes is the measurement this benchmark exists to keep taking.

**Source 4 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement Yizhe Chi , Wenyi Li, Deyao Hong, Xiaoqiu Wang, Mingju Gao, Kaisen Yang, Bingxiang He, Y oujie Zheng, Calvin Xiao, Qinhuai Na Navers Lab, Einsia.AI T singhua University Project Lead Corresponding Author Abstract.Recursive self-improvement (RSI) asks whether an AI system can improve the process that produces AI systems, so that the next system inherits the improvement. That process is the training algorithm: a better objective or update rule improves the compute-capability exchange rate for every subsequent run, including the one that produces the next agent. Whether RSI is feasible therefore turns on whether an agent can design training algorithms. No benchmark isolates that ability: existing suites are won by collecting data or by tuning hyperparameters, and none tells a change to how a run is executed apart from a change to how the model learns. We present AI4AI-Bench, 10 frozen research repositories spanning 10 training algorithm families. In each task, an agent has 4 hours on one B300 to rewrite the training algorithm; its code is then rerun from scratch for up to 12 hours and scored by a fixed evaluator hidden from the agent, against the repository’s original algorithm under the same procedure. Because the 10 metrics are incommensurable, every task is mapped onto one scale on which0is an uninformative model,0 .1is the algorithm the repository ships, and1.0 is the task optimum. Across 29 configurations of 6 systems

Source references exist; whether they support each claim still needs review.

Check figures against their sources.

Answer is 25 words; the review target is at most 120.


</details>

## q07 — Manual review needed

How are the incommensurable AI4AI-Bench task metrics normalized?

**Answer:** The AI4AI-Bench task metrics are normalized to a scale where 0 is an uninformative model, 0.1 is the algorithm the repository ships, and 1.0 is the task optimum [2].

**Automatic findings:** Check figures against their sources.

**Check these points:**

- [ ] States that 0 represents an uninformative model.
- [ ] States that 0.1 represents the original algorithm shipped with the repository.
- [ ] States that 1.0 represents the task optimum on the common normalized scale.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 2**

> incommensurable, every task is mapped onto one scale on which0is an uninformative model,0 .1is the algorithm the repository ships, and1.0 is the task optimum. Across 29 configurations of 6 systems on all 10 tasks the mean score is0.166,…

**Source 5**

> …model, with four hours to write code, twelve to run what was written, and an evaluation the agent never sees. Across 29 configurations of six systems on all ten tasks the mean score is 0.166on a scale where the algorithm the repository

<details><summary>Full sources and automatic checks</summary>

**Source 1 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

differently, and so does the baseline. The ten metrics are incommensurable: an aesthetic score, a perplexity, a solve rate, a pass rate, an unlearning balanced score. They cannot be averaged as they stand, and every per-task number in this paper is reported in its own units against its own baseline.

**Source 2 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

incommensurable, every task is mapped onto one scale on which0is an uninformative model,0 .1is the algorithm the repository ships, and1.0 is the task optimum. Across 29 configurations of 6 systems on all 10 tasks the mean score is0.166, and the best system reaches0.250: even the strongest closes under a fifth of the distance between the algorithm that was already there and the optimum. The submissions show where that distance went: most never change how the model learns at all, and the minority that do average0.226against0 .126 for the rest. More reasoning effort mostly buys the willingness to go there, taking that minority from 8%of submissions to64%and the mean score from0 .094to0 .196. We release the task suite, the evaluators and every scored submission, so that the measurement can be repeated as these systems change. Date:August 21, 2026 Homepage:https://lab.einsia.ai/ai4ai Correspondence:nana@einsia.ai

**Source 3 — Inducing Task Models from Computer-Use Traces**

{2, . . . ,15}and three trajectories per condition. The bottom row reports the overall mean. dARI↑MAE↓ 2 0.980±0.024 0.29±0.34 3 0.975±0.032 0.50±0.60 4 0.968±0.029 0.64±0.61 Overall 0.974±0.028 0.48±0.54 meration. Both metrics are averaged across all trajectories per condition. Induction stays robust under heavy interleaving. Table 1 reports ARI and MAE aggregated over all task counts K∈ {2, . . . ,15}for each interleaving difficulty level. The system reaches 0.974 overall ARI, recovering non-contiguous task partitions even when activities are densely interwoven. MAE stays below 1 across all conditions, so the predicted task count tracks the true count as the number of concurrent tasks grows from 2 to 15. Appendix B.1 presents the error analysis.

**Source 4 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

AI4AI. To our knowledge, AI4AI-Bench is the first benchmark to make that step the object of evaluation: it removes the agent, reruns the submitted source from a clean start, and classifies the patch to verify whether the gain came from execution, data, or a change to the training algorithm itself.

**Source 5 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

Recursive self-improvement compounds through the algorithmic link, and AI4AI-Bench is built to measure that link on its own: ten frozen research repositories, each asking an agent to improve the training algorithm it already applies to its own model, with four hours to write code, twelve to run what was written, and an evaluation the agent never sees. Across 29 configurations of six systems on all ten tasks the mean score is 0.166on a scale where the algorithm the repository already ships is0.1and the task optimum is1.0, and the best system reaches0.250. The submissions say where the rest of that distance went: of the 263 that changed anything, 141 never touch how the model learns at all, and the 122 that do average0.226against 0.126for the rest — the algorithmic layer is where the distance is closed, and most submissions never go to it. More reasoning effort mostly buys the willingness to go, taking that minority from8%of submissions to64% and the mean score from0.094to0 .196, which still leaves the strongest setting a tenth of the way past the algorithm it started from. What today’s agents do at the algorithmic link, then, is recover a competent default rather than design past one; whether that changes is the measurement this benchmark exists to keep taking.

Source references exist; whether they support each claim still needs review.

Check figures against their sources.

Answer is 29 words; the review target is at most 120.


</details>

## q08 — Manual review needed

How do submissions that change the learning algorithm compare with those that do not?

**Answer:** Submissions that change the learning algorithm (46.4% of 263 submissions) are compared to those that do not (53.6% of 263 submissions). The comparison is limited by the fact that the stronger systems are more likely to reach the learning procedure, so the gap is between submissions that go there and those that do not, rather than the effect of going there. [3]

**Automatic findings:** Check figures against their sources.

**Check these points:**

- [ ] Reports a mean normalized score of 0.226 for submissions that change the learning algorithm.
- [ ] Reports 0.126 for the rest and correctly identifies the former as higher, without claiming a causal effect.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 1**

> Everything so far has been a score, and a score says only whether a submission worked. It does not say whether the agent designed an algorithm or tuned one that was already there, and those are the two outcomes this paper exists to…

**Source 2**

> optimum. The submissions say where the remaining distance went. Of the 263 that change anything, 141 leave the learning procedure exactly as they found it and move budgets, checkpointing, hyperparameters and capacity instead. The 122 that…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

Everything so far has been a score, and a score says only whether a submission worked. It does not say whether the agent designed an algorithm or tuned one that was already there, and those are the two outcomes this paper exists to separate. Counting lines does not separate them either: a one-line diff can replace a learning rule, and a thousand-line refactor can leave the training procedure exactly as it was. The distinction has to be read off the submitted code, by asking which part of the training procedure it modifies. We group every change into eight families on two sides of that line (Table 4). The grouping is assigned by a separate language model reading each submitted diff against the definitions that follow. Four changehow this run goes: how long it trains and how often it saves; the training hyperparameters, such as the learning rate or the batch size; which of the checkpoints it produced to keep; and how much trainable capacity to attach and where, such as the rank and placement of an adapter. Four changehow the model learns: the loss it optimizes, by adding, removing or reweighting a term; the supervision it learns from, by introducing a signal the procedure did not have before; the update rule itself, replaced by a different one; and the data the procedure trains on. The families are not exclusive — a submission matches3.13of them on average, since changing a loss usually drags a hyperparameter along with it — so we report how many submissions reach a side rather than assigning each to a single family. Of the 280 submissions, 17 made no change that could be classified. Of the remaining 263, 141 stay entirely on the run

**Source 2 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

optimum. The submissions say where the remaining distance went. Of the 263 that change anything, 141 leave the learning procedure exactly as they found it and move budgets, checkpointing, hyperparameters and capacity instead. The 122 that do reach it — the objective, the supervision signal, the learning rule, the data — average0.226 against0 .126for the rest: the algorithmic layer is where the distance gets closed, and most submissions never go there. More reasoning effort moves agents towards it, taking that share from8%to64%and the mean score from0 .094to0 .196, which is most of what effort buys. A machine learning scientist reads the training dynamics, names the mechanism that is failing, and changes that mechanism; few of these submissions do. In summary, we make the following contributions: • A benchmark that isolates the algorithmic level.Ten research repositories spanning ten families of training algorithms, evaluated under a unified protocol that separates a four-hour agent development window from the up to twelve-hour clean-start training run used for scoring. • A measurement of what agentsdo, not only whether they win.Every submission is classified by what it changes, which is what turns “the agent improved the training algorithm” into a checkable statement rather than a restatement of the score. • Revealing a gap between algorithmic exploration and actual improvement.The algorithmic design level is where the distance to a better algorithm is closed and the one these agents reach least often, and more reasoning effort mostly buys the willingness to reach it.

**Source 3 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

how much trainable capacity, and where 73 27.8% learning the loss it optimizes 87 33.1% the supervision it learns from 66 25.1% the update rule itself 23 8.7% the data it trains on 21 8.0% any learning family 122 46.4% run side only141 53.6% reaching the learning sidenone low medium high xhigh max share of submissions 8% 39% 33% 49% 65% 64% ordering holds within four of the five models in this corpus. It is also not a randomized comparison: the systems that reach the learning procedure more often are the stronger ones to begin with, so the gap is the difference between the submissions that go there and the submissions that do not, rather than the effect of going there. Read together, the two numbers say that the algorithmic layer is where the distance actually gets closed, and that most submissions never go to it. What separates a submission that reaches that layer from one that does not is not effort spent but a step taken first: reading the training dynamics as a specific failure mechanism, and then addressing that mechanism.

**Source 4 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

Is there anything that pushes agents down to that layer? The setup has exactly one knob that can be turned on its own, the reasoning-effort level, and taking it from the lowest setting to the highest shows clearly what it buys and what it does not. It buys nerve. The share of submissions that touch the learning algorithm rises from8.0%to64 .0%. At low effort the submissions move budgets, logging and optimization knobs; at high effort they operate on objectives, replace learning rules, and add supervision to the procedure. It buys attempts. Within the Codex grid, the only one that exposes the lowest setting, the median configuration goes from 4 evaluations inside its four hours to 16, from 18 edited lines to 246, and from11k output tokens to 109k. Cost follows: the median exploration cost per task rises from $1.69 to $34.60. The exploration stage of the whole evaluation consumed $5,334 of API calls, with Kimi K3 converted from Chinese yuan and with neither the GPU hours of the twelve-hour runs nor the evaluator’s compute included. It buys completion. Of the 19 cells that score zero, 8 ended their four hours without a usable patch — four of them workspaces the host found empty after the agent exited early — and 11 submitted a complete patch that started its twelve-hour run but finished with nothing satisfying the contract, most often no loadable merged model written to disk. All 19 terminated normally: the failure is in what the agent submitted. They concentrate at low effort, with 12 of the 19 at the two lowest levels and one each at the two highest. And it does buy a result, though a small one

Source references exist; whether they support each claim still needs review.

Check figures against their sources.

Answer is 61 words; the review target is at most 120.


</details>

## q09 — Manual review needed

What two kinds of task structure does TMI recover from computer-use traces?

**Answer:** TMI recovers two kinds of task structure: objective models and procedure models. Objective models are hierarchical decompositions of the objective of a task into subobjectives, while procedure models are trace-grounded compositions of control-flow operators specifying how the execution of a task is organized. [3]

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Identifies a hierarchical objective model of recursive goal decomposition.
- [ ] Identifies a procedure model representing execution control flow.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 1**

> TMI recovers task models from raw computer-use traces in three stages (Figure 2).Event grounding and activity segmentationrecovers what each event did from its visual context and groups the grounded events into semantic actions and…

**Source 3**

> …axes. • Theobjective model Ot is a hierarchical decomposition of the objective of t into the subobjectives the user pursues. • Theprocedure model Pt is a trace-grounded composition of control-flow operators specifying how the execution…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — Inducing Task Models from Computer-Use Traces**

TMI recovers task models from raw computer-use traces in three stages (Figure 2).Event grounding and activity segmentationrecovers what each event did from its visual context and groups the grounded events into semantic actions and activities (3.1). Latent task inductiondiscovers the tasks pursued in the session and assigns each activity to the task it realizes (3.2).Task model constructionbuilds complementary objective and procedure models for each task and reconciles them into a single task model, in which every node pairs an objective with the control-flow operator over its children (3.3).

**Source 2 — Inducing Task Models from Computer-Use Traces**

We introduced TASKMODELINDUCTION, which abstracts raw computer-use events into activities, discovers the latent tasks they realize, and reconciles separately induced objective and procedure models into one task model per task. It turns everyday computer-use activity traces into durable, auditable records of how work are carried out. Experiments show that this representation recovers interleaved execution faithfully and yields better downstream agent skills than raw traces or workflow summaries. TMI has the potential to facilitate the study of how work is carried out across domains and to make these records reusable as knowledge for both people and agents.

**Source 3 — Inducing Task Models from Computer-Use Traces**

Describing the activity conducted in a computeruse session is necessary for analyzing, learning, and auditing it. We formalize this process as the recovery, from a naturalistic computer-use trace, of the latent tasks and their task models specifying the objectives and procedures. Let X=⟨x 1, x2, . . . , xN ⟩ be a user’s computeruse trace in an interactive computer environment, where each event xi = (si, ai, τi) contains the state represented as a screenshot si, a low-level operation (e.g.,click), and a timestampτ i. The trajectory reflects the user pursuing a set of latent tasks T={t 1, t2, . . .}, where a possibly non-contiguous subsequence of events collectively realizes a task tj. However, inducing these latent tasks remains an open challenge, as none of the tasks, their boundaries, or the event-to-task assignments are predefined in realistic human traces. We define the induction problem as jointly discovering the set of latent tasks T and, for each task t∈ T, inducing a task model Mt that unifies two complementary axes. • Theobjective model Ot is a hierarchical decomposition of the objective of t into the subobjectives the user pursues. • Theprocedure model Pt is a trace-grounded composition of control-flow operators specifying how the execution of t is organized through sequencing and iteration. The task model Mt is structured as a tree, whose every node pairs an objective with a control-flow operator over its children.

**Source 4 — Inducing Task Models from Computer-Use Traces**

Naturalistic computer-use traces, passively recorded screenshots and mouse or keyboard actions, are a valuable resource for deriving symbolic, auditable, and reusable models of how everyday work is done. Such models matter as computer-use agents enter real work, where agents need to learn how tasks are actually performed, and organizations need to audit and reuse that knowledge. However, inducing such task models is challenging, as activity is observed only as low-level events and realworld work is multi-threaded with interleaved goals. Existing methods assume a given task or a single workflow, and produce step-level summaries rather than structured task models. We introduce TASKMODELINDUCTION(TMI), which (i) discovers the latent tasks in an unconstrained trace, disentangling concurrent activity, and (ii) for each latent task, induces a task model pairing a hierarchical objective model of recursive goal decomposition with a procedure model of the control flow that organized the execution. Intrinsically, on controlled human and agent trajectories, TMI recovers interleaved tasks with 0.974 agreement against ground-truth groupings and reconstructs 74.9% of the observed execution steps, far more than the strongest workflow induction baseline. Extrinsically, skills derived from TMI’s task models improve held-out task accuracy by 30.0% over the strongest baseline. 1

Source references exist; whether they support each claim still needs review.

Answer is 43 words; the review target is at most 120.


</details>

## q10 — Manual review needed

How well does task model induction group interleaved tasks?

**Answer:** Our method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings [2].

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Reports 0.974 agreement with ground-truth interleaved-task groupings; does not describe this as general task success accuracy.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 2**

> …real human work sessions (Wang et al., 2025), our method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings and matches 74.9% of observed execution steps against 30.3% for the strongest workflow induction…

**Source 1**

> Model Induction (TMI), a method that grounds raw events into semantic activities, untangles interleaved sessions into latent tasks, and induces each task’s objective and procedure models independently before reconciling them into a…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — Inducing Task Models from Computer-Use Traces**

Model Induction (TMI), a method that grounds raw events into semantic activities, untangles interleaved sessions into latent tasks, and induces each task’s objective and procedure models independently before reconciling them into a unified model under formal validity constraints. • Intrinsic and extrinsic evaluation show that our method recovers interleaved tasks and their execution structure more faithfully than workflow induction baselines, and that the induced task models yield more effective skills for downstream agents.

**Source 2 — Inducing Task Models from Computer-Use Traces**

into semantic actions and activities, bridging low-level signal and local intent (3.1).Latent task inductionuntangles the multi-threaded activity stream, discovering tasks and assigning possibly non-contiguous activities to each without a pre-specified task set (3.2).Task model constructionbuilds each task’s objective and procedure models under formal validity constraints and reconciles them into a unified model in which objective scope and control flow are mutually consistent (3.3). We evaluate TMI both intrinsically via controlled reconstructions of human computer-use sessions (4), and extrinsically via downstream agent learning (5). On trajectories built from a dataset of recorded real human work sessions (Wang et al., 2025), our method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings and matches 74.9% of observed execution steps against 30.3% for the strongest workflow induction baseline. Using the induced task model to generate reusable agent skills, evaluated on heldout SkillLearnBench (Zhong et al., 2026) tasks, improves task accuracy by 30.0% over baseline. Our contributions are as follows. • We formalize thetask modelfor induction from unconstrained traces, a representation that pairs hierarchical objective decomposition with structured control flow, grounded in computational thinking and the structured programming theorem. • We proposeTask Model Induction (TMI), a method that grounds raw events into semantic activities, untangles interleaved sessions into latent tasks, and induces each task’s objective and procedure models independently before reconciling them into a

**Source 3 — Inducing Task Models from Computer-Use Traces**

Naturalistic activity is multi-threaded (Figure 2(b)), so the activities of one task are interleaved with those of others and spread across applications and artifacts (Czerwinski et al., 2004; González and Mark, 2004). Latent task induction recovers the task set T , in which every activity belongs to exactly one task and each task states the objective its activities jointly support. Activities are processed in the order they appear in the trace, and each is assigned to a task. The tasks discovered this way are consolidated once the trace ends. Assigning an activity requires a compact representation of each task that the activity can be compared against. Each task maintains a profile with a summary of what the task achieves and a small Task C. Task B. Visit world monitor Event Grounding &amp; Activity Segmentation 3.1 Click(1900.8, 829.5) … … (a) raw input events … … (b) semantic actions … … (c) activities Bring Codex to the foreground Submit UI change request Latent Task Induction 3.2 xi ei X E Task A. Build a user study website Anchors study-frontend review-trial.web.app Classify each activity Does it match any existing tasks? YesNo Create a new task Assign + update anchors Task Model Construction 3.3 Objective Model Procedure Model do x do y do z setup for useri setup for userj do z setup users Reconcile Root SEQ FOR SEQ do x do y WHILE …. user∈{i,j} {cond} Root do x SEQ FOR user∈{i,j} setup users SEQ WHILE {cond} … do y SEQ do z SEQ For

**Source 4 — Inducing Task Models from Computer-Use Traces**

interleaved tasks entangled, sub-objectives flattened, and iteration and branching unexpressed (Wang et al., 2025; Zang et al., 2025). Trace-analysis methods assume the root task is given in advance (Wang et al., 2024; Grohs et al., 2024) and do not extend to sessions with latent and diverse tasks. We address these gaps by defining atask model, an explicit representation of a single task that pairs its goal hierarchy with the control flow of its execution. Theobjective modelrepresents the goal hierarchy as a recursive decomposition of the task into the (sub-)objectives the user pursued (Wing, 2006; Stanton, 2006). Theprocedure modelrepresents the control flow as a composition of sequencing and iteration operators, the structured programming constructs that remain observable in a trace (Böhm and Jacopini, 1966). Recovering task models from naturalistic computer-use traces (2) requires a system to jointly discover the latent tasks of an unconstrained session and induce a task model for each, with no tasks, boundaries, or descriptions given in advance (Figure 1). We introduce TASKMODELINDUCTION(TMI) (3), a method that addresses the three challenges in turn.Event grounding and activity segmentation recovers what each raw event did from its visual context and groups the results into semantic actions and activities, bridging low-level signal and local intent (3.1).Latent task inductionuntangles the multi-threaded activity stream, discovering tasks and assigning possibly non-contiguous activities to each without a

Source references exist; whether they support each claim still needs review.

Figures match available source values; their meaning and attribution still need review.

Answer is 13 words; the review target is at most 120.


</details>

## q11 — Manual review needed

How much of observed execution does TMI reconstruct?

**Answer:** TMI reconstructs 74.9% of the observed execution steps [4].

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Reports reconstruction of 74.9% of observed execution steps, not 74.9% of tasks.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 2**

> …flow that organized the execution. Intrinsically, on controlled human and agent trajectories, TMI recovers interleaved tasks with 0.974 agreement against ground-truth groupings and reconstructs 74.9% of the observed execution steps, far…

**Source 4**

> …On trajectories built from a dataset of recorded real human work sessions (Wang et al., 2025), our method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings and matches 74.9% of observed execution steps…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — Inducing Task Models from Computer-Use Traces**

The induced latent task set should reflect the actual task being performed and remain stable across different execution strategies. Since induction never observes the task label, agreement with the assigned task indicates that induced tasks capture task identity rather than execution idiosyncrasies. Metric.We evaluate on all 38 Human- Work (Wang et al., 2025) sessions and on the SkillsBench runs whose task outcome is marked successful. For each run i, TMI observes only the trajectory and produces an induced latent task set Ti. The dataset’s ground-truth assigned task yi is withheld from induction and used only for evaluation. We give Ti and yi to the LLM judge, which returns a binary label indicating whether the induced task set correctly captures the assigned task. We report the average match rate across runs. To control for judge-family bias, we repeat the evaluation with judges from two model families, gpt-5.5 and claude-sonnet-5. Induced tasks track task identity, not execution idiosyncrasies.Table 2 reports match rates for both datasets. On human sessions, the induced task set matches the assigned task in 94.74% of runs, indicating that induction recovers the task Table 2: Latent task identity match rate (%). Each judge compares the induced task set with the held-out assigned task. We report results from gpt-5.5 and claude-sonnet- 5. SkillsBench includes only successful runs. Dataset SubsetNJudge gpt-5.5 claude-sonnet-5 HumanWork All 38 94.74 89.47 SkillsBench All 74 93.24 98.65 By agent Claude Code 23

**Source 2 — Inducing Task Models from Computer-Use Traces**

Naturalistic computer-use traces, passively recorded screenshots and mouse or keyboard actions, are a valuable resource for deriving symbolic, auditable, and reusable models of how everyday work is done. Such models matter as computer-use agents enter real work, where agents need to learn how tasks are actually performed, and organizations need to audit and reuse that knowledge. However, inducing such task models is challenging, as activity is observed only as low-level events and realworld work is multi-threaded with interleaved goals. Existing methods assume a given task or a single workflow, and produce step-level summaries rather than structured task models. We introduce TASKMODELINDUCTION(TMI), which (i) discovers the latent tasks in an unconstrained trace, disentangling concurrent activity, and (ii) for each latent task, induces a task model pairing a hierarchical objective model of recursive goal decomposition with a procedure model of the control flow that organized the execution. Intrinsically, on controlled human and agent trajectories, TMI recovers interleaved tasks with 0.974 agreement against ground-truth groupings and reconstructs 74.9% of the observed execution steps, far more than the strongest workflow induction baseline. Extrinsically, skills derived from TMI’s task models improve held-out task accuracy by 30.0% over the strongest baseline. 1

**Source 3 — Inducing Task Models from Computer-Use Traces**

Model Induction (TMI), a method that grounds raw events into semantic activities, untangles interleaved sessions into latent tasks, and induces each task’s objective and procedure models independently before reconciling them into a unified model under formal validity constraints. • Intrinsic and extrinsic evaluation show that our method recovers interleaved tasks and their execution structure more faithfully than workflow induction baselines, and that the induced task models yield more effective skills for downstream agents.

**Source 4 — Inducing Task Models from Computer-Use Traces**

into semantic actions and activities, bridging low-level signal and local intent (3.1).Latent task inductionuntangles the multi-threaded activity stream, discovering tasks and assigning possibly non-contiguous activities to each without a pre-specified task set (3.2).Task model constructionbuilds each task’s objective and procedure models under formal validity constraints and reconciles them into a unified model in which objective scope and control flow are mutually consistent (3.3). We evaluate TMI both intrinsically via controlled reconstructions of human computer-use sessions (4), and extrinsically via downstream agent learning (5). On trajectories built from a dataset of recorded real human work sessions (Wang et al., 2025), our method recovers interleaved tasks with 0.974 agreement against ground-truth task groupings and matches 74.9% of observed execution steps against 30.3% for the strongest workflow induction baseline. Using the induced task model to generate reusable agent skills, evaluated on heldout SkillLearnBench (Zhong et al., 2026) tasks, improves task accuracy by 30.0% over baseline. Our contributions are as follows. • We formalize thetask modelfor induction from unconstrained traces, a representation that pairs hierarchical objective decomposition with structured control flow, grounded in computational thinking and the structured programming theorem. • We proposeTask Model Induction (TMI), a method that grounds raw events into semantic activities, untangles interleaved sessions into latent tasks, and induces each task’s objective and procedure models independently before reconciling them into a

Source references exist; whether they support each claim still needs review.

Figures match available source values; their meaning and attribution still need review.

Answer is 9 words; the review target is at most 120.


</details>

## q12 — Manual review needed

What was the held-out task accuracy gain from skills derived from TMI?

**Answer:** The held-out task accuracy gain from skills derived from TMI was 30% relative accuracy gain, from 14.29 to 18.57, over the strongest baseline. [1]

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Reports a 30.0% improvement in held-out task accuracy over the strongest baseline; does not relabel it as a 30-percentage-point gain.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 1**

> model improves executability from 59.35 to 67.65 and held-out accuracy from 14.29 to 18.57 over the strongest baseline, a 30% relative accuracy gain. The expert-written skills score highest on skill coverage at 93.59 but reach 10.00…

**Source 3**

> …agent trajectory alignment; Acc. is held-out task accuracy. All values are percentages (↑). No skill produces no generated artifact, so the skillquality columns are not applicable. Human curated baseline judges the expert-written skills…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — Inducing Task Models from Computer-Use Traces**

Exec. Safe. Align. Acc. No skill N/A N/A N/A 59.20 8.57 Human curated 93.59 63.80 90.50 56.59 10.00 Raw demonstration 52.46 53.4992.5263.15 11.43 Workflow summary 54.07 59.35 91.74 66.18 14.29 Ours54.95 67.6590.5767.99 18.57 The task model is an effective representation for skill transfer.Skills generated from our task models are more executable and transfer better to held-out tasks than skills generated from raw demonstrations or workflow summaries. As shown in Table 5, our model improves executability from 59.35 to 67.65 and held-out accuracy from 14.29 to 18.57 over the strongest baseline, a 30% relative accuracy gain. The expert-written skills score highest on skill coverage at 93.59 but reach 10.00 held-out accuracy, below every induced source, so skill coverage and held-out accuracy do not move together in this setting. We report the curated condition as a reference point for skill quality rather than as an upper bound on accuracy.

**Source 2 — Inducing Task Models from Computer-Use Traces**

The induced latent task set should reflect the actual task being performed and remain stable across different execution strategies. Since induction never observes the task label, agreement with the assigned task indicates that induced tasks capture task identity rather than execution idiosyncrasies. Metric.We evaluate on all 38 Human- Work (Wang et al., 2025) sessions and on the SkillsBench runs whose task outcome is marked successful. For each run i, TMI observes only the trajectory and produces an induced latent task set Ti. The dataset’s ground-truth assigned task yi is withheld from induction and used only for evaluation. We give Ti and yi to the LLM judge, which returns a binary label indicating whether the induced task set correctly captures the assigned task. We report the average match rate across runs. To control for judge-family bias, we repeat the evaluation with judges from two model families, gpt-5.5 and claude-sonnet-5. Induced tasks track task identity, not execution idiosyncrasies.Table 2 reports match rates for both datasets. On human sessions, the induced task set matches the assigned task in 94.74% of runs, indicating that induction recovers the task Table 2: Latent task identity match rate (%). Each judge compares the induced task set with the held-out assigned task. We report results from gpt-5.5 and claude-sonnet- 5. SkillsBench includes only successful runs. Dataset SubsetNJudge gpt-5.5 claude-sonnet-5 HumanWork All 38 94.74 89.47 SkillsBench All 74 93.24 98.65 By agent Claude Code 23

**Source 3 — Inducing Task Models from Computer-Use Traces**

family. We compare three sources of learning, each fed into the same skill creator. Raw demonstration passes the grounded trace directly, the workflow summary baseline (Wang et al., 2025) passes a phase-based summary of the demonstration, and Ourspasses the task model. ANo skillcondition bounds performance without any generated skill, and aHuman curatedcondition substitutes the expert-written skills released with the benchmark for a generated one. All skills are generated and all held-out instances executed with gpt-5-mini, so performance differences isolate the source of learning. We report the five SkillLearnBench metrics of skill coverage, executability, safety, agent trajectory alignment, and held-out task accuracy. 2The skill creation component of the Codex CLI agent, https://github.com/openai/codex. Table 5: Extrinsic evaluation on SkillLearnBench. Each row is a source of learning fed into the same skill creator. Cov. is skill coverage; Exec. is executability; Safe. is safety; Align. is agent trajectory alignment; Acc. is held-out task accuracy. All values are percentages (↑). No skill produces no generated artifact, so the skillquality columns are not applicable. Human curated baseline judges the expert-written skills released with the original benchmark. Skill quality Execution Source of learning Cov. Exec. Safe. Align. Acc. No skill N/A N/A N/A 59.20 8.57 Human curated 93.59 63.80 90.50 56.59 10.00 Raw demonstration 52.46

**Source 4 — Inducing Task Models from Computer-Use Traces**

A useful task model should transfer beyond the demonstration it was induced from. We test whether an induced task model is an effective source for generating reusable skills that improve a downstream agent’s accuracy on held-out tasks. Experiment Protocol.We test this by using the induced task model as the source of learning for agent skill generation on SkillLearnBench (Zhong et al., 2026), a benchmark for continual learning methods for agent skill generation on real-world Table 4: Task model fidelity. Task is latent task recovery; Cov. is objective coverage; Coh. is objective coherence; P-Ch. is parent-child consistency; Desc. is step description accuracy; Op. is operator correctness. Task and Cov. are 5-point Likert scores (↑); remaining columns are binary pass rates (%,↑). Each ablation removes one component of our method, the objective model, the procedure model, or the reconciliation step where the two models are induced jointly in a single pass. Best per judge in bold. Objective Procedure Model Task Cov. Coh. P-Ch. Desc. Op. gpt-5.5 judge Workflow summary 2.87 3.24 65.7 62.2 30.3 N/A Direct gen. 3.63 4.00 87.0 97.3 23.4 52.7 Ours 3.71 4.34 85.7 92.674.9 88.5 w/o objective model 3.37 3.76 77.9 85.1 63.2 84.1 w/o procedure model3.744.34 78.9 96.7

**Source 5 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

input context. We describe these below. Note that in each domain, we partition the data into three splits: training (for fitting value estimators), calibration (for estimating the value-of-information model parameters in Sections 4 and 5), and test. Table 1 reports the MSE of each estimator on held-out calibration data. Additional dataset and implementation details are included in Appendix B. Math: Inference-time scaling in mathematical reasoning tasks.In inference-time scaling, extended reasoning improves accuracy on hard problems but increases latency and cost. The costly estimator𝑔 treatstheearlytokensofareasoningtraceasprivateinformation: itreceivesthepromptalongwiththe first 𝑘 tokens of a model’s chain-of-thought, effectively peeking at a partial solution to judge whether the model’s reasoning looks promising before committing to it. This is passed with the prompt to a small language model (Gemini 3.1 Flash-Lite) with a regression loss to predict the specialist’s reward directly from the prompt text. The router chooses between Gemma3-4B (Team et al., 2025) (low-cost) and Gemini-3.1-Flash-Lite (Google DeepMind, 2026) (cost0.66, discounted from correctness in𝑅𝑚). This high cost is chosen so that it is beneficial to route to Gemini only when confident that it can answer correctly and Gemma cannot. We evaluate on a corpus of 16,512 mathematical problems spanning MATH (Hendrycks et al., 2021), Omni-Math (Gao et al., 2025), AIME

Source references exist; whether they support each claim still needs review.

Figures match available source values; their meaning and attribution still need review.

Answer is 23 words; the review target is at most 120.


</details>

## q13 — Manual review needed

How many commuters and respondent-scenario observations did the weather travel survey collect?

**Answer:** The study collected mode choices from 92 McGill student commuters. The chatbot-administered survey yielded 454 respondent–scenario observations [4].

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Reports 92 student commuters.
- [ ] Reports 454 respondent–scenario observations, keeping this distinct from participant count.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 1**

> …choice and machinelearning models, and LLM-based prediction within a unified travel-behavior research framework. The workflow was demonstrated using stated mode choices from 92 McGill student commuters across five image-augmented weather…

**Source 4**

> …conversational data collection, structured data processing, and behavioral prediction. A chatbot-administered, image-augmented stated-preference survey collected mode choices from 92 student commuters across five predefined weather…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

This study developed and evaluated a multi-agent workflow that integrates conversational survey design, AI-generated visual scenarios, structured data processing, conventional choice and machinelearning models, and LLM-based prediction within a unified travel-behavior research framework. The workflow was demonstrated using stated mode choices from 92 McGill student commuters across five image-augmented weather scenarios. The empirical results support three main conclusions. First, the conversational survey produced behaviorally coherent responses across weather conditions, with adverse weather reducing cycling-related choices and increasing reliance on public transit. Second, the comparison of MNL, random forest, and LLM-based predictors highlighted their complementary roles. MNL provided interpretable behavioral relationships, random forest offered a strong trained predictive benchmark, and LLMs enabled flexible incorporation of habitual travel information, persona descriptions, examples, and visual context without requiring a separate model specification for each information type. Third, LLM performance depended strongly on prompt design and input configuration. Habitual travel information and expert-oriented framing generally improved prediction, while selected vision-capable models benefited from the inclusion of the same weather images shown to respondents. The best zero-shot LLM achieved a point estimate comparable to the trained machine-learning benchmark without task-specific fitting to the survey observations. The main contribution of this study is therefore not the replacement of established travelchoice models, but the demonstration of an integrated and extensible workflow in which data collection, processing, behavioral analysis, predictive modeling, and multimodal reasoning can be coordinated through specialized agents. This framework provides a practical basis for incorporating heterogeneous behavioral information into travel-choice analysis while retaining conventional models for interpretation, benchmarking, and

**Source 2 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

The Data Collection Agent was implemented as a purpose-built conversational chatbot,Travel- BehaviorSurveyBot, developed in Voiceflow and deployed as a web application accessible through a public link and QR code (Figure 2). Respondents completed the survey on their own devices through a guided dialogue that collected demographic and mobility-resource information, usual summer and winter travel patterns, weekly mode-use frequencies, and ratings of factors influencing mode choice, followed by five SP weather scenarios. Each scenario presented one photorealistic image generated using defined prompts to represent Sunny, Hot–humid, Rainy, Foggy/cold, and Snowy commuting conditions. All respondents were shown the same fixed image for each scenario. For each scenario, respondents selected among five modeled alternatives: walking, cycling, public transit, bike and public transit combined, and driving. The chatbot followed predefined survey logic and did not alter the experimental attributes or choice set during deployment. Figure 2 illustrates the chatbot interface and the weather images presented during the survey. This design required neither app installation nor an interviewer and ensured consistent presentation of the scenarios across respondents. The Phase 1 sample included 92 McGill student commuters.

**Source 3 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

The Data Processing Agent converted the chatbot’s raw, JSON-encoded response export into an analysis-ready dataset. Processing steps included parsing nested list- and slider-valued fields, validating and recoding categorical responses, constructing derived features, and flagging incomplete or inconsistent records for exclusion. Of the 460 potential observations from 92 respondents and Figure 2: Chatbot survey interface and sample weather-scenario question with embedded generated images. five scenarios, six were excluded because the recorded mode choice was missing or unparseable. The final processed dataset therefore contained 454 valid respondent–scenario observations. Key derived features include weekly mode-use frequency by season, the seven factor-importance scores, binary vehicle-, license-, and bicycle-ownership flags, and years of cycling experience, which are reported in Section 5.1.

**Source 4 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

Travel-behavior research increasingly combines digital data collection with predictive modeling, yet these stages are often developed and evaluated separately. This study proposes and applies a three-agent workflow integrating conversational data collection, structured data processing, and behavioral prediction. A chatbot-administered, image-augmented stated-preference survey collected mode choices from 92 student commuters across five predefined weather scenarios, yielding 454 respondent–scenario observations. Weather-related associations were analyzed using a multinomial logit model, while logistic regression and random forest provided machine-learning benchmarks. Nine locally deployed large language models (LLMs), ranging from 2 to 35 billion parameters, were evaluated across four zero-shot prompt-and-context conditions and further extended through persona, few-shot, and vision-based configurations. Cycling was particularly sensitive to adverse weather, while public transit use increased substantially under the Snowy scenario. Random forest achieved 69.6% five-class accuracy, while the best text-only zero-shot LLM reached 69.9% without task-specific fitting. Habitual travel information produced the most consistent improvement in LLM prediction, Expert framing generally outperformed Role-Play, and persona information was most beneficial when habitual travel information was unavailable. Few-shot prompting further improved five-class prediction for several models, with gains generally stabilizing after a small number of examples. Vision-capable models were also evaluated using the same weather images presented to respondents, and the best vision-based configuration reached the highest observed five-class accuracy of 71.5%, indicating that visual context can provide additional predictive information for selected models. Taken together, the findings show that habitual travel information, prompting strategy,

Source references exist; whether they support each claim still needs review.

Figures match available source values; their meaning and attribution still need review.

Answer is 18 words; the review target is at most 120.


</details>

## q14 — Manual review needed

Which travel modes responded most clearly to adverse weather and snow?

**Answer:** Cycling was particularly sensitive to adverse weather, while public transit use increased substantially under the Snowy scenario [4].

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] States that cycling was particularly sensitive to adverse weather.
- [ ] States that public-transit use increased under the Snowy scenario.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 4**

> adverse weather, while public transit use increased substantially under the Snowy scenario. Random forest achieved 69.6% five-class accuracy, while the best text-only zero-shot LLM reached 69.9% without task-specific fitting. Habitual…

**Source 1**

> …but also by the context in which travel occurs. Weather is an important example, as precipitation, extreme temperatures, and adverse road conditions can alter the attractiveness of walking and cycling and shift travelers toward transit…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

Understanding and predicting travel behavior is a fundamental component of transportation planning, encompassing interrelated decisions regarding when, where, and how individuals travel. Among these decisions, mode choice is particularly consequential because it determines how demand is distributed across transportation systems and influences infrastructure needs, environmental impacts, andtheeffectivenessofpoliciesaimedatencouragingsustainablemobility(McFadden,1974; Ben-Akiva and Lerman, 1985). Mode choice is shaped not only by individual and alternative-specific characteristics but also by the context in which travel occurs. Weather is an important example, as precipitation, extreme temperatures, and adverse road conditions can alter the attractiveness of walking and cycling and shift travelers toward transit or private vehicles, with effects that vary across individuals and seasons (Hyland et al., 2018; Böcker et al., 2013; Saneinejad et al., 2012). Understanding such context-dependent responses therefore requires approaches capable of capturing heterogeneous travel behavior under conditions that may not be adequately represented in observed travel data. Collecting behavioral data that capture these contextual variations remains challenging. Traditional travel surveys, whether administered through interviews, paper questionnaires, or static web forms, can be costly and time-consuming to implement and scale (Krueger et al., 2016; Bansal and Kockelman, 2018). Stated-preference (SP) surveys provide an important alternative by allowing researchers to examine choices under controlled or hypothetical conditions, including weather scenarios that may be difficult to observe systematically in revealed-preference data. However, conventional SP instruments typically describe such situations using fixed textual descriptions, requiring respondents to construct the intended context themselves and limiting the incorporation of richer

**Source 2 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

approximately 70%, whereas binary accuracy varies less across models. Together, these results indicate that the information supplied to the model and the framing of the prediction task can be as important as the selected model. Prediction difficulty also varies systematically across weather conditions and travel modes. As shown in Figure 6, Snowy and Rainy scenarios are generally easier to predict because they produce stronger and more consistent shifts toward public transit and away from cycling. In contrast, Sunny and Warm conditions yield more heterogeneous choices and lower prediction accuracy. At the mode level, Public Transit is predicted relatively well across models, whereas Bike+Transit is consistently the most difficult alternative. These differences suggest that LLM accuracy depends Figure 5: Zero-shot accuracy across all four base conditions (Expert/Role-Play× Base/Richer Context) for nine models, shown separately for five-class and binary accuracy. Figure 6: Accuracy patterns by weather condition and mode across all evaluated models, with a bar-chart breakdown of weather- and mode-level accuracy under the EXP-RC experiment. not only on model and prompt configuration but also on the behavioral regularity of the scenario and alternative being predicted.

**Source 3 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

The MNL estimates were consistent with these descriptive patterns, showing lower relative utility for cycling-related modes and higher relative utility for transit and driving under adverse scenarios. This agreement supports the internal coherence of the survey responses, although it does not by itself establish external validity or actual travel behavior under observed weather. LLM configuration and comparison with conventional models.LLM performance was strongly influenced by the information included in the prompt. RC, which adds habitual travel information to BC, produced the most consistent improvement, showing that habitual travel behavior contains predictive information not fully captured by demographics and mobility resources. EXP framing generally outperformed RP, particularly for smaller models, while gains from increasing model size diminished among larger models. Persona augmentation and few-shot prompting were most useful when they supplied missing information. Persona descriptions improved prediction mainly under BC, where habitual travel information was unavailable, but added little under RC. Few-shot examples improved five-class accuracy for several models, particularly smaller ones, although the gains generally stabilized after a small number of demonstrations. The conventional and LLM-based models therefore play complementary roles. The MNL provides behavioral interpretation, while logistic regression and random forest offer data-trained predictive benchmarks. Random forest achieved the highest trained-model accuracy, while the best zero-shot LLM produced a comparable five-class point estimate without task-specific fitting to the survey records. This result suggests potential value for LLM-based prediction when labeled travel-behavior data are limited, although the small sample and multiple evaluated configurations require cautious interpretation. The five-class task provides a clearer assessment of whether models distinguish among individual modes. Vision-based prediction.The vision experiments directly connect the survey and

**Source 4 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

Travel-behavior research increasingly combines digital data collection with predictive modeling, yet these stages are often developed and evaluated separately. This study proposes and applies a three-agent workflow integrating conversational data collection, structured data processing, and behavioral prediction. A chatbot-administered, image-augmented stated-preference survey collected mode choices from 92 student commuters across five predefined weather scenarios, yielding 454 respondent–scenario observations. Weather-related associations were analyzed using a multinomial logit model, while logistic regression and random forest provided machine-learning benchmarks. Nine locally deployed large language models (LLMs), ranging from 2 to 35 billion parameters, were evaluated across four zero-shot prompt-and-context conditions and further extended through persona, few-shot, and vision-based configurations. Cycling was particularly sensitive to adverse weather, while public transit use increased substantially under the Snowy scenario. Random forest achieved 69.6% five-class accuracy, while the best text-only zero-shot LLM reached 69.9% without task-specific fitting. Habitual travel information produced the most consistent improvement in LLM prediction, Expert framing generally outperformed Role-Play, and persona information was most beneficial when habitual travel information was unavailable. Few-shot prompting further improved five-class prediction for several models, with gains generally stabilizing after a small number of examples. Vision-capable models were also evaluated using the same weather images presented to respondents, and the best vision-based configuration reached the highest observed five-class accuracy of 71.5%, indicating that visual context can provide additional predictive information for selected models. Taken together, the findings show that habitual travel information, prompting strategy,

Source references exist; whether they support each claim still needs review.

Answer is 18 words; the review target is at most 120.


</details>

## q15 — Manual review needed

How did random forest and the best text-only zero-shot LLM compare on five-class travel prediction?

**Answer:** Random forest achieved a five-class accuracy of 69.6% [4], while the best text-only zero-shot LLM reached 69.9% [1].

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Reports 69.6% five-class accuracy for random forest.
- [ ] Reports 69.9% for the best text-only zero-shot LLM without task-specific fitting, slightly higher than random forest.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 4**

> …Snowy scenario. Random forest achieved 69.6% five-class accuracy, while the best text-only zero-shot LLM reached 69.9% without task-specific fitting. Habitual travel information produced the most consistent improvement in LLM prediction,…

**Source 2**

> regression and random forest offer data-trained predictive benchmarks. Random forest achieved the highest trained-model accuracy, while the best zero-shot LLM produced a comparable five-class point estimate without task-specific fitting…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

labeled travel-behavior data are limited, although the small sample and multiple evaluated configurations require cautious interpretation. The five-class task provides a clearer assessment of whether models distinguish among individual modes. Vision-based prediction.The vision experiments directly connect the survey and prediction stages of the multi-agent workflow by providing the models with the same generated weather images shown to respondents. The best vision-based configuration achieved a five-class accuracy point estimate of 71.5%, compared with 69.9% for the highest text-only zero-shot LLM result and 69.6% for random forest. These small differences should be interpreted as descriptive point-estimate differences rather than evidence of statistical superiority. Vision combined with few-shot prompting also produced gains for selected models. Because each weather condition was represented by one fixed image, the results indicate that these specific images provided additional predictive signal for selected models; they do not establish generalization to unseen weather images or visual environments. Limitations and future research.The findings should be interpreted in light of several limitations. The sample consists of 92 students from one university and is not representative of the wider population. The analysis is based on stated choices rather than observed travel under actual weather conditions, and uncontrolled visual features within the generated images may have influenced both respondents and models. Each weather condition was represented by one fixed image, so weather condition and image identity were not separately identified. The relatively small and imbalanced dataset also limits detailed analysis of minority modes and traveler subgroups, while the results may vary across other LLM families and inference settings. The framework itself was not experimentally compared with a conventional research workflow; its contribution therefore concerns integration, traceability, and modularity rather than a measured

**Source 2 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

The MNL estimates were consistent with these descriptive patterns, showing lower relative utility for cycling-related modes and higher relative utility for transit and driving under adverse scenarios. This agreement supports the internal coherence of the survey responses, although it does not by itself establish external validity or actual travel behavior under observed weather. LLM configuration and comparison with conventional models.LLM performance was strongly influenced by the information included in the prompt. RC, which adds habitual travel information to BC, produced the most consistent improvement, showing that habitual travel behavior contains predictive information not fully captured by demographics and mobility resources. EXP framing generally outperformed RP, particularly for smaller models, while gains from increasing model size diminished among larger models. Persona augmentation and few-shot prompting were most useful when they supplied missing information. Persona descriptions improved prediction mainly under BC, where habitual travel information was unavailable, but added little under RC. Few-shot examples improved five-class accuracy for several models, particularly smaller ones, although the gains generally stabilized after a small number of demonstrations. The conventional and LLM-based models therefore play complementary roles. The MNL provides behavioral interpretation, while logistic regression and random forest offer data-trained predictive benchmarks. Random forest achieved the highest trained-model accuracy, while the best zero-shot LLM produced a comparable five-class point estimate without task-specific fitting to the survey records. This result suggests potential value for LLM-based prediction when labeled travel-behavior data are limited, although the small sample and multiple evaluated configurations require cautious interpretation. The five-class task provides a clearer assessment of whether models distinguish among individual modes. Vision-based prediction.The vision experiments directly connect the survey and

**Source 3 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

This study developed and evaluated a multi-agent workflow that integrates conversational survey design, AI-generated visual scenarios, structured data processing, conventional choice and machinelearning models, and LLM-based prediction within a unified travel-behavior research framework. The workflow was demonstrated using stated mode choices from 92 McGill student commuters across five image-augmented weather scenarios. The empirical results support three main conclusions. First, the conversational survey produced behaviorally coherent responses across weather conditions, with adverse weather reducing cycling-related choices and increasing reliance on public transit. Second, the comparison of MNL, random forest, and LLM-based predictors highlighted their complementary roles. MNL provided interpretable behavioral relationships, random forest offered a strong trained predictive benchmark, and LLMs enabled flexible incorporation of habitual travel information, persona descriptions, examples, and visual context without requiring a separate model specification for each information type. Third, LLM performance depended strongly on prompt design and input configuration. Habitual travel information and expert-oriented framing generally improved prediction, while selected vision-capable models benefited from the inclusion of the same weather images shown to respondents. The best zero-shot LLM achieved a point estimate comparable to the trained machine-learning benchmark without task-specific fitting to the survey observations. The main contribution of this study is therefore not the replacement of established travelchoice models, but the demonstration of an integrated and extensible workflow in which data collection, processing, behavioral analysis, predictive modeling, and multimodal reasoning can be coordinated through specialized agents. This framework provides a practical basis for incorporating heterogeneous behavioral information into travel-choice analysis while retaining conventional models for interpretation, benchmarking, and

**Source 4 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

Travel-behavior research increasingly combines digital data collection with predictive modeling, yet these stages are often developed and evaluated separately. This study proposes and applies a three-agent workflow integrating conversational data collection, structured data processing, and behavioral prediction. A chatbot-administered, image-augmented stated-preference survey collected mode choices from 92 student commuters across five predefined weather scenarios, yielding 454 respondent–scenario observations. Weather-related associations were analyzed using a multinomial logit model, while logistic regression and random forest provided machine-learning benchmarks. Nine locally deployed large language models (LLMs), ranging from 2 to 35 billion parameters, were evaluated across four zero-shot prompt-and-context conditions and further extended through persona, few-shot, and vision-based configurations. Cycling was particularly sensitive to adverse weather, while public transit use increased substantially under the Snowy scenario. Random forest achieved 69.6% five-class accuracy, while the best text-only zero-shot LLM reached 69.9% without task-specific fitting. Habitual travel information produced the most consistent improvement in LLM prediction, Expert framing generally outperformed Role-Play, and persona information was most beneficial when habitual travel information was unavailable. Few-shot prompting further improved five-class prediction for several models, with gains generally stabilizing after a small number of examples. Vision-capable models were also evaluated using the same weather images presented to respondents, and the best vision-based configuration reached the highest observed five-class accuracy of 71.5%, indicating that visual context can provide additional predictive information for selected models. Taken together, the findings show that habitual travel information, prompting strategy,

Source references exist; whether they support each claim still needs review.

Figures match available source values; their meaning and attribution still need review.

Answer is 18 words; the review target is at most 120.


</details>

## q16 — Manual review needed

What was the best vision-based travel mode prediction accuracy?

**Answer:** 71.5% [2][3]

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Reports 71.5% as the best observed vision-based five-class travel-mode prediction accuracy, not a guaranteed general result.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 2**

> and the best vision-based configuration reached the highest observed five-class accuracy of 71.5%, indicating that visual context can provide additional predictive information for selected models. Taken together, the findings show that…

**Source 4**

> …stabilizing after a small number of examples. Vision-capable models were also evaluated using the same weather images presented to respondents, and the best vision-based configuration reached the highest observed five-class accuracy of…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

The MNL estimates were consistent with these descriptive patterns, showing lower relative utility for cycling-related modes and higher relative utility for transit and driving under adverse scenarios. This agreement supports the internal coherence of the survey responses, although it does not by itself establish external validity or actual travel behavior under observed weather. LLM configuration and comparison with conventional models.LLM performance was strongly influenced by the information included in the prompt. RC, which adds habitual travel information to BC, produced the most consistent improvement, showing that habitual travel behavior contains predictive information not fully captured by demographics and mobility resources. EXP framing generally outperformed RP, particularly for smaller models, while gains from increasing model size diminished among larger models. Persona augmentation and few-shot prompting were most useful when they supplied missing information. Persona descriptions improved prediction mainly under BC, where habitual travel information was unavailable, but added little under RC. Few-shot examples improved five-class accuracy for several models, particularly smaller ones, although the gains generally stabilized after a small number of demonstrations. The conventional and LLM-based models therefore play complementary roles. The MNL provides behavioral interpretation, while logistic regression and random forest offer data-trained predictive benchmarks. Random forest achieved the highest trained-model accuracy, while the best zero-shot LLM produced a comparable five-class point estimate without task-specific fitting to the survey records. This result suggests potential value for LLM-based prediction when labeled travel-behavior data are limited, although the small sample and multiple evaluated configurations require cautious interpretation. The five-class task provides a clearer assessment of whether models distinguish among individual modes. Vision-based prediction.The vision experiments directly connect the survey and

**Source 2 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

and the best vision-based configuration reached the highest observed five-class accuracy of 71.5%, indicating that visual context can provide additional predictive information for selected models. Taken together, the findings show that habitual travel information, prompting strategy, demonstrations, and visual context can meaningfully shape LLM-based travel-choice prediction. More broadly, the study demonstrates how conversational survey administration, structured data processing, conventional behavioral modeling, machine learning, and multimodal LLM prediction can be coordinated within a single auditable multi-agent workflow, while broader application requires validation with larger and more representative traveler samples. Keywords:Large language models, travel behavior, travel survey, conversational agents, mode choice, multi-agent system framework Corresponding author. Email:narges.ahmadi@mail.mcgill.ca

**Source 3 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

labeled travel-behavior data are limited, although the small sample and multiple evaluated configurations require cautious interpretation. The five-class task provides a clearer assessment of whether models distinguish among individual modes. Vision-based prediction.The vision experiments directly connect the survey and prediction stages of the multi-agent workflow by providing the models with the same generated weather images shown to respondents. The best vision-based configuration achieved a five-class accuracy point estimate of 71.5%, compared with 69.9% for the highest text-only zero-shot LLM result and 69.6% for random forest. These small differences should be interpreted as descriptive point-estimate differences rather than evidence of statistical superiority. Vision combined with few-shot prompting also produced gains for selected models. Because each weather condition was represented by one fixed image, the results indicate that these specific images provided additional predictive signal for selected models; they do not establish generalization to unseen weather images or visual environments. Limitations and future research.The findings should be interpreted in light of several limitations. The sample consists of 92 students from one university and is not representative of the wider population. The analysis is based on stated choices rather than observed travel under actual weather conditions, and uncontrolled visual features within the generated images may have influenced both respondents and models. Each weather condition was represented by one fixed image, so weather condition and image identity were not separately identified. The relatively small and imbalanced dataset also limits detailed analysis of minority modes and traveler subgroups, while the results may vary across other LLM families and inference settings. The framework itself was not experimentally compared with a conventional research workflow; its contribution therefore concerns integration, traceability, and modularity rather than a measured

**Source 4 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

Travel-behavior research increasingly combines digital data collection with predictive modeling, yet these stages are often developed and evaluated separately. This study proposes and applies a three-agent workflow integrating conversational data collection, structured data processing, and behavioral prediction. A chatbot-administered, image-augmented stated-preference survey collected mode choices from 92 student commuters across five predefined weather scenarios, yielding 454 respondent–scenario observations. Weather-related associations were analyzed using a multinomial logit model, while logistic regression and random forest provided machine-learning benchmarks. Nine locally deployed large language models (LLMs), ranging from 2 to 35 billion parameters, were evaluated across four zero-shot prompt-and-context conditions and further extended through persona, few-shot, and vision-based configurations. Cycling was particularly sensitive to adverse weather, while public transit use increased substantially under the Snowy scenario. Random forest achieved 69.6% five-class accuracy, while the best text-only zero-shot LLM reached 69.9% without task-specific fitting. Habitual travel information produced the most consistent improvement in LLM prediction, Expert framing generally outperformed Role-Play, and persona information was most beneficial when habitual travel information was unavailable. Few-shot prompting further improved five-class prediction for several models, with gains generally stabilizing after a small number of examples. Vision-capable models were also evaluated using the same weather images presented to respondents, and the best vision-based configuration reached the highest observed five-class accuracy of 71.5%, indicating that visual context can provide additional predictive information for selected models. Taken together, the findings show that habitual travel information, prompting strategy,

Source references exist; whether they support each claim still needs review.

Figures match available source values; their meaning and attribution still need review.

Answer is 1 words; the review target is at most 120.


</details>

## q17 — Manual review needed

What is the patient-oriented medical report interpretation task introduced with G-CARL?

**Answer:** Patient-oriented Medical Report Interpretation (PMRI) is a novel open-ended multimodal generation task that requires models to explain medical reports in accurate and accessible language based on a user's query and dialogue history. [3]  PMRI must align professional medical knowledge with patient-centered communication by transforming report evidence into accessible explanations, addressing user-specific concerns, and offering clinically cautious guidance. [4]  PMRI differs fundamentally from medical VQA (Task A) and conventional report generation (Task B) in that it produces long-form, patient-facing explanations grounded in multimodal reports and extended patient-doctor dialogue. [4]

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Describes PMRI as explaining medical reports accurately in accessible language for patients.
- [ ] Explains that the interpretation responds to the user query and dialogue history using multimodal report information.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 4**

> …Yujin Wang and Xiandong Li. A C B D Task A : Medical VQA Task B : Medical Report Generation Task C : Patient-Oriented Medical Report Interpretation Short-form description Findings: Impressions: Doctor-patient dialogue history User query…

**Source 3**

> …medical vision-language tasks do not adequately capture these dual requirements. To bridge this gap, we introducePatient-orientedMedicalReportInterpretation(PMRI), a novel open-ended multimodal generation task that requires models to…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation Shiao Xie1, Siyu Chen1, Jianwei Lv1, Bo Yuan1, Yujin Wang1, Xiandong Li1 1Baidu Inc., China

**Source 2 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

In this paper, we present PMRI as a challenging yet underexplored task that requires both evidence-grounded medical factualityandcontext-dependentpatientcommunication.We propose G-CARL, a reinforcement learning framework that decomposes reward supervision through retrieval-grounded claim verification and case-specific checklist guidance. ExtensiveexperimentsonthenewlyconstructedMMedReport benchmark, together with clinician-authored evaluation and human preference studies, show that G-CARL consistently improves the quality of patient-oriented medical report interpretation.Wehopethisworklaysthefoundationformore reliable and patient-centered multimodal medical assistants.

**Source 3 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

Personalized interpretation of medical reports has emerged as an increasingly important need among patients. Addressing this need requires both evidence-grounded medical factualityandcontext-dependentpatientcommunication,yetexisting medical vision-language tasks do not adequately capture these dual requirements. To bridge this gap, we introducePatient-orientedMedicalReportInterpretation(PMRI), a novel open-ended multimodal generation task that requires models to explain medical reports in accurate and accessible language based on a user’s query and dialogue history. Thesetwoobjectivesdifferfundamentallyintheirverifiability, yet remain tightly coupled, making them difficult to optimize jointly under conventional supervised fine-tuning and holistic reinforcement learning paradigms. To address this challenge, we propose G-CARL, a grounded, checklist-aligned reinforcementlearningframeworkthatcombinesmulti-source retrieval for atomic claim verification with context-aware, instance-specific weighted checklists for response coverage, providing structured supervision for factuality, user-demand satisfaction, and expression quality without constraining response diversity. We further constructMMedReport, a realworld PMRI benchmark, along with a clinician-designed three-dimensionalevaluationprotocol.Extensiveexperiments demonstrate that G-CARL consistently outperforms existing post-training baselines in overall quality, claim-level precision, and checklist recall. Pairwise preference evaluation by clinicians further confirms that G-CARL produces interpretations that are more accurate and better aligned with patient needs.

**Source 4 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

Written within a professional medical context, medical reports commonly present abnormal values and descriptive findingswithoutexplainingtheirimplicationsfornon-expert readers. This gap becomes particularly salient in online healthcare scenarios, where patients upload one or more report images and ask open-ended questions shaped by their personal concerns. To bridge this gap, we introduce Patient-oriented Medical Report Interpretation(PMRI), a novel open-ended multimodal generation task. Beyond simply recognizing report findings, PMRI must align professional medical knowledge with patient-centered communi- These authors contributed equally. Corresponding authors: Yujin Wang and Xiandong Li. A C B D Task A : Medical VQA Task B : Medical Report Generation Task C : Patient-Oriented Medical Report Interpretation Short-form description Findings: Impressions: Doctor-patient dialogue history User query Long-form report interpretation Medical reports Reference answer Findings: Impressions: Which lobe is primarily affected by the abnormal hyperintense lesion? : Frontal : Parietal : Temporal : Occipital Can you tell the baby's sex from this? Abnormal Findings Risk Stratification Health Advice Evaluation Metrics Medical accuracy Demand satisfaction Expression quality Matching Semantic Figure1:ComparisonofPMRI(Task C)withmedicalVQA (Task A)andconventionalreportgeneration(Task B).Unlike the short, deterministic outputs of Tasks A and B, PMRI produces long-form, patient-facing explanations grounded inmultimodalreportsandextendedpatient–doctordialogue. cation by transforming report evidence into accessible explanations, addressing user-specific concerns, and offering clinically cautious guidance. As illustrated in Fig. 1,

Source references exist; whether they support each claim still needs review.

Answer is 85 words; the review target is at most 120.


</details>

## q18 — Manual review needed

How does G-CARL supervise factuality and response coverage?

**Answer:** G-CARL supervises factuality through a retrieval-grounded claim reward that provides fine-grained supervision for medical factuality through atomic claim verification [1]. It also supervises response coverage through a case-specific checklist reward that explicitly supervises demand satisfaction and expression quality without relying on a single reference response [1]. This is achieved by constructing instance-specific weighted checklists through MLLM generation followed by clinician refinement [1].

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Explains multi-source retrieval for verifying atomic claims.
- [ ] Explains context-aware, instance-specific weighted checklists for covering user needs/response content.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 3**

> verification with context-aware, instance-specific weighted checklists for response coverage, providing structured supervision for factuality, user-demand satisfaction, and expression quality without constraining response diversity. We…

**Source 4**

> …historyh, and a user queryq,thepolicymodelfirstgeneratescandidateresponses. The retrieval-grounded branch then supervises medical factuality by verifying report-grounded medical claims against evidence retrieved from a multi-source…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

reward provides localized supervision for sparse factual errors and discouragesunsupportedelaboration.Forcontext-dependent objectives such as demand satisfaction and expression quality,G-CARLconstructsinstance-specificweightedchecklists throughMLLMgenerationfollowedbyclinicianrefinement. Each checklist item is assigned an automatically generated weight, allowing the reward to emphasize the aspects most relevant to the current report and user question. This yields an explicit checklist score that provides transparent supervision for whether the response addresses the user’s concern appropriately. In summary, our contributions are as follows: •WeformulatePMRIasanevidence-groundedandpatientfacingmultimodalgenerationtask,andproposeG-CARL, a reinforcement learning framework that decomposes reward supervision according to the heterogeneous verifiability of different objectives. •We propose a retrieval-grounded claim reward that provides fine-grained supervision for medical factuality through atomic claim verification, and a case-specific checklist reward that explicitly supervises demand satisfactionandexpressionqualitywithoutrelyingonasingle reference response. •We constructMMedReport, a real-world PMRI benchmark with clinician-designed evaluation protocols. Extensive experiments across multiple LVLM backbones demonstrate that G-CARL consistently outperforms supervised and reinforcement learning baselines, with the gains further validated by clinician preference and user comprehension studies.

**Source 2 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

In this paper, we present PMRI as a challenging yet underexplored task that requires both evidence-grounded medical factualityandcontext-dependentpatientcommunication.We propose G-CARL, a reinforcement learning framework that decomposes reward supervision through retrieval-grounded claim verification and case-specific checklist guidance. ExtensiveexperimentsonthenewlyconstructedMMedReport benchmark, together with clinician-authored evaluation and human preference studies, show that G-CARL consistently improves the quality of patient-oriented medical report interpretation.Wehopethisworklaysthefoundationformore reliable and patient-centered multimodal medical assistants.

**Source 3 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

Personalized interpretation of medical reports has emerged as an increasingly important need among patients. Addressing this need requires both evidence-grounded medical factualityandcontext-dependentpatientcommunication,yetexisting medical vision-language tasks do not adequately capture these dual requirements. To bridge this gap, we introducePatient-orientedMedicalReportInterpretation(PMRI), a novel open-ended multimodal generation task that requires models to explain medical reports in accurate and accessible language based on a user’s query and dialogue history. Thesetwoobjectivesdifferfundamentallyintheirverifiability, yet remain tightly coupled, making them difficult to optimize jointly under conventional supervised fine-tuning and holistic reinforcement learning paradigms. To address this challenge, we propose G-CARL, a grounded, checklist-aligned reinforcementlearningframeworkthatcombinesmulti-source retrieval for atomic claim verification with context-aware, instance-specific weighted checklists for response coverage, providing structured supervision for factuality, user-demand satisfaction, and expression quality without constraining response diversity. We further constructMMedReport, a realworld PMRI benchmark, along with a clinician-designed three-dimensionalevaluationprotocol.Extensiveexperiments demonstrate that G-CARL consistently outperforms existing post-training baselines in overall quality, claim-level precision, and checklist recall. Pairwise preference evaluation by clinicians further confirms that G-CARL produces interpretations that are more accurate and better aligned with patient needs.

**Source 4 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

BuiltuponGRPO,G-CARLoptimizesthepolicymodelwith three reward signals, as illustrated in Fig. 2. Given uploaded medical report imagesI, the dialogue historyh, and a user queryq,thepolicymodelfirstgeneratescandidateresponses. The retrieval-grounded branch then supervises medical factuality by verifying report-grounded medical claims against evidence retrieved from a multi-source medical datastore, producing the factuality rewardRfact (Sec. 3.2). In parallel, the case-specific checklist branch optimizes demand satisfaction and expression quality by constructing a weighted checklist through MLLM generation followed by clinicianguided refinement, and evaluating the checklist coverage of thegeneratedresponsetoproducethechecklistrewardR check (Sec. 3.3). Together with the format rewardRformat, these components are integrated into a multi-objective optimization task (Sec. 3.4).

Source references exist; whether they support each claim still needs review.

Answer is 62 words; the review target is at most 120.


</details>

## q19 — Manual review needed

What benchmark and evaluation dimensions accompany G-CARL?

**Answer:** G-CARL is primarily evaluated on the MMedReport benchmark, which contains 2,450 instances of patient-oriented medical report interpretation. Additionally, it is evaluated on the CMB benchmark, which assesses both medical QA accuracy and the professionalism of open-ended clinical interpretation.  The evaluation dimensions for G-CARL on MMedReport include subjective evaluation along three clinician-defined dimensions: Medical Accuracy (Accuracy), Demand Satisfaction (Satisfaction), and Expression Quality (Expression). [1] On CMB, G-CARL is evaluated on QA accuracy and professionalism of open-ended clinical interpretation. [3]

**Automatic findings:** Check figures against their sources.

**Check these points:**

- [ ] Names MMedReport as the PMRI benchmark.
- [ ] Names medical accuracy, demand satisfaction, and expression quality as the three evaluation dimensions.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 1**

> three clinician-defined dimensions:Medical Accuracy(Accuracy),Demand Satisfaction(Satisfaction), andExpression Quality(Expression). Each dimension is scored according to a detailed clinicianauthored rubric ranging from−2to3. We employ…

**Source 3**

> …its holistic rubric struggles to jointly optimizemedicalaccuracy,demandsatisfaction,andexpression quality. By decomposing reward supervision into externally verifiable medical claims and internally grounded checklist objectives, G-CARL…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

Datasets.We evaluate G-CARL primarily on(1)MMedReport, a real-world multimodal PMRI benchmark collectedfromonlinehealthcareconsultations.Itcontains2,450 instances, each including dialogue history, a user query, uploaded medical report images, and clinician-verified reference annotations. All instances undergo quality control, deidentification,and manualverificationto ensuredataquality andpatientprivacy.Detaileddatasetstatisticsareprovidedin Appendix A. Following the standard split, 2,200 instances areusedfortrainingand250forevaluation.Toassessbroader medical capability beyond PMRI, we further evaluate the trained models on an external benchmark,(2)CMB(Wang et al. 2024), which evaluates both medical QA accuracy and theprofessionalismofopen-endedclinicalinterpretationunder an LLM-as-a-judge protocol. Evaluation Protocol.We evaluate PMRI responses using both subjective and objective metrics. (1)Subjective Evaluation. Responses are evaluated along three clinician-defined dimensions:Medical Accuracy(Accuracy),Demand Satisfaction(Satisfaction), andExpression Quality(Expression). Each dimension is scored according to a detailed clinicianauthored rubric ranging from−2to3. We employ GPT- 5.2 (OpenAI 2025) as the judge by strictly following these evaluation criteria. The complete judging protocol is provided in theAppendix Bto facilitate reproducibility. (2) Objective Evaluation. In addition to holistic assessment, we report

**Source 2 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

(OpenAI 2025) as the judge by strictly following these evaluation criteria. The complete judging protocol is provided in theAppendix Bto facilitate reproducibility. (2) Objective Evaluation. In addition to holistic assessment, we report claim-levelPrecisionand checklist-levelRecall. Precisionisdefinedastheratioofsupportedclaimstoextracted claims,measuringfactualcorrectness,whileRecallisdefined as the ratio of satisfied checklist items to the total checklist items, measuring coverage of case-specific requirements. Implementation Details.Our policy models are initialized from the Qwen3-VL (Bai et al. 2025) and InternVL3 (Zhu et al. 2025) series. G-CARL is trained for three epochs on 8 NVIDIAH100GPUswithabatchsizeof192andalearning rate of1×10 −5. We set the number of rollout responses to G= 8duringtraining.Therewardweightsareconfiguredas λfact = 0.4,λcheck = 0.3, andλformat = 0.3; detailed hyperparameter analysis is provided inAppendix C. The verifier Vis initialized with Qwen3.5-35B-A3B, with implementation details reported inAppendix D. We compare G-CARL withthecorrespondingbasemodels(Base),supervisedfinetuning (SFT), and an MLLM-as-a-Judge reward baseline. Additionally,weevaluateadiverserangeofgeneral-purpose and

**Source 3 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

QuantitativeComparison.Table1comparesG-CARLwith general-purpose LVLMs, specialized medical LVLMs, and different training paradigms based on the Qwen3-VL and InternVL3 backbones. While the MLLM-as-a-Judge reward improves over SFT, its holistic rubric struggles to jointly optimizemedicalaccuracy,demandsatisfaction,andexpression quality. By decomposing reward supervision into externally verifiable medical claims and internally grounded checklist objectives, G-CARL achieves the highest scores acrossbothobjectiveandsubjectivemetrics.OnQwen3-VL- 8B, G-CARL improves the overall subjective score, while boosting claim-level precision (+0.77%) and checklist-level recall (+6.71%), indicating more informative and richer interpretations. Moreover, G-CARL consistently outperforms specialized medical LVLMs and remains competitive with substantiallylargergeneral-purposeLVLMsunderzero-shot evaluation, demonstrating the effectiveness of G-CARL. External Generalization Study.We further transfer the trained models to CMB without any adaptation. As shown in Table 2, G-CARL improves QA accuracy on both splits (+0.63)andattainsthehighestprofessionalismscoreinopenended generation (3.61), whereas SFT and MLLM-as-a- Judge bring marginal gains and even degrade professionalism. This indicates that grounding rewards in verifiable

Source references exist; whether they support each claim still needs review.

Check figures against their sources.

Answer is 76 words; the review target is at most 120.


</details>

## q20 — Manual review needed

What did clinician pairwise preferences find about G-CARL?

**Answer:** G-CARL was consistently preferred over both SFT and MLLM-as-a-Judge, particularly in medical accuracy and demand satisfaction, with preference margins of 136:85 and 106:64, respectively. [1]

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Reports that clinicians preferred G-CARL interpretations as more accurate and better aligned with patient needs; does not turn preference into a claim of proven clinical outcomes.

**Sources found:** 1 of 1 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 4**

> quality, claim-level precision, and checklist recall. Pairwise preference evaluation by clinicians further confirms that G-CARL produces interpretations that are more accurate and better aligned with patient needs.

**Source 1**

> Three professional clinicians compare anonymized responses generated by G-CARL and the baselines across the three evaluation dimensions and overall preference, with majority voting used for the final decision. We additionally recruit 50…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

andproducesclinicallyaccurateinterpretations.Similarly,in therightexample,SFTfailstoincorporatetheuser’ssmoking andalcoholcessationhistory,whileMLLM-as-a-Judgefails toaddresstheuser’sprimaryconcernofwhethertocontinue themedication.Incontrast,ourmethodeffectivelyintegrates historical context, directly answers the user’s question, and provides evidence-grounded recommendations. Human Preference Evaluation.We conduct a blind pairwise human preference study on 250 held-out cases. Three professional clinicians compare anonymized responses generated by G-CARL and the baselines across the three evaluation dimensions and overall preference, with majority voting used for the final decision. We additionally recruit 50 participants without medical training to evaluate the comprehensibility of the generated interpretations. As shown in Fig. 4, G-CARL is consistently preferred over both SFT and MLLM-as-a-Judge,particularlyinmedicalaccuracyanddemand satisfaction, with preference margins of 136:85 and 106:64, respectively. It also achieves substantially higher comprehensibilityratingsfromnon-expertparticipants,suggestingthatitsresponsesareeasierforpatientstounderstand.

**Source 2 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

Datasets.We evaluate G-CARL primarily on(1)MMedReport, a real-world multimodal PMRI benchmark collectedfromonlinehealthcareconsultations.Itcontains2,450 instances, each including dialogue history, a user query, uploaded medical report images, and clinician-verified reference annotations. All instances undergo quality control, deidentification,and manualverificationto ensuredataquality andpatientprivacy.Detaileddatasetstatisticsareprovidedin Appendix A. Following the standard split, 2,200 instances areusedfortrainingand250forevaluation.Toassessbroader medical capability beyond PMRI, we further evaluate the trained models on an external benchmark,(2)CMB(Wang et al. 2024), which evaluates both medical QA accuracy and theprofessionalismofopen-endedclinicalinterpretationunder an LLM-as-a-judge protocol. Evaluation Protocol.We evaluate PMRI responses using both subjective and objective metrics. (1)Subjective Evaluation. Responses are evaluated along three clinician-defined dimensions:Medical Accuracy(Accuracy),Demand Satisfaction(Satisfaction), andExpression Quality(Expression). Each dimension is scored according to a detailed clinicianauthored rubric ranging from−2to3. We employ GPT- 5.2 (OpenAI 2025) as the judge by strictly following these evaluation criteria. The complete judging protocol is provided in theAppendix Bto facilitate reproducibility. (2) Objective Evaluation. In addition to holistic assessment, we report

**Source 3 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

In this paper, we present PMRI as a challenging yet underexplored task that requires both evidence-grounded medical factualityandcontext-dependentpatientcommunication.We propose G-CARL, a reinforcement learning framework that decomposes reward supervision through retrieval-grounded claim verification and case-specific checklist guidance. ExtensiveexperimentsonthenewlyconstructedMMedReport benchmark, together with clinician-authored evaluation and human preference studies, show that G-CARL consistently improves the quality of patient-oriented medical report interpretation.Wehopethisworklaysthefoundationformore reliable and patient-centered multimodal medical assistants.

**Source 4 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

Personalized interpretation of medical reports has emerged as an increasingly important need among patients. Addressing this need requires both evidence-grounded medical factualityandcontext-dependentpatientcommunication,yetexisting medical vision-language tasks do not adequately capture these dual requirements. To bridge this gap, we introducePatient-orientedMedicalReportInterpretation(PMRI), a novel open-ended multimodal generation task that requires models to explain medical reports in accurate and accessible language based on a user’s query and dialogue history. Thesetwoobjectivesdifferfundamentallyintheirverifiability, yet remain tightly coupled, making them difficult to optimize jointly under conventional supervised fine-tuning and holistic reinforcement learning paradigms. To address this challenge, we propose G-CARL, a grounded, checklist-aligned reinforcementlearningframeworkthatcombinesmulti-source retrieval for atomic claim verification with context-aware, instance-specific weighted checklists for response coverage, providing structured supervision for factuality, user-demand satisfaction, and expression quality without constraining response diversity. We further constructMMedReport, a realworld PMRI benchmark, along with a clinician-designed three-dimensionalevaluationprotocol.Extensiveexperiments demonstrate that G-CARL consistently outperforms existing post-training baselines in overall quality, claim-level precision, and checklist recall. Pairwise preference evaluation by clinicians further confirms that G-CARL produces interpretations that are more accurate and better aligned with patient needs.

Source references exist; whether they support each claim still needs review.

Figures match available source values; their meaning and attribution still need review.

Answer is 24 words; the review target is at most 120.


</details>

## q21 — Manual review needed

Which papers address auditable AI workflows derived from user activity or survey responses?

**Answer:** The provided papers do not address this question.

**Automatic findings:** Add source references to the answer itself.

**Check these points:**

- [ ] Identifies Inducing Task Models from Computer-Use Traces/TMI and connects it to auditable task models derived from user activity.
- [ ] Identifies the agentic travel-behavior paper and connects it to an auditable workflow integrating survey collection, processing, and prediction.

**Sources found:** 2 of 2 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 4**

> …workflow that connects AI-assisted behavioral data collection with travel-choice modeling and LLM-based prediction. The paper makes four main contributions. First, it develops a conversational, image-augmented SP survey in which an…

**Source 5**

> …travel-behavior research. Solid arrows denote forward data and artifact flows, whereas dashed arrows denote refinements implemented in a subsequent workflow iteration. from survey design through data processing to behavioral prediction…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — Inducing Task Models from Computer-Use Traces**

Naturalistic computer-use traces, passively recorded screenshots and mouse or keyboard actions, are a valuable resource for deriving symbolic, auditable, and reusable models of how everyday work is done. Such models matter as computer-use agents enter real work, where agents need to learn how tasks are actually performed, and organizations need to audit and reuse that knowledge. However, inducing such task models is challenging, as activity is observed only as low-level events and realworld work is multi-threaded with interleaved goals. Existing methods assume a given task or a single workflow, and produce step-level summaries rather than structured task models. We introduce TASKMODELINDUCTION(TMI), which (i) discovers the latent tasks in an unconstrained trace, disentangling concurrent activity, and (ii) for each latent task, induces a task model pairing a hierarchical objective model of recursive goal decomposition with a procedure model of the control flow that organized the execution. Intrinsically, on controlled human and agent trajectories, TMI recovers interleaved tasks with 0.974 agreement against ground-truth groupings and reconstructs 74.9% of the observed execution steps, far more than the strongest workflow induction baseline. Extrinsically, skills derived from TMI’s task models improve held-out task accuracy by 30.0% over the strongest baseline. 1

**Source 2 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

We propose a multi-agent framework that integrates data collection, data processing, and behavioral modeling for commuter mode-choice research. The framework consists of three specialized agents: a Data Collection Agent, a Data Processing Agent, and a Data Modeling Agent. The agents perform distinct research functions and exchange information through standardized data objects and explicit input–output interfaces. Data transfer, version control, and execution records are organized through a shared workspace and predefined workflow rules. These supporting functions are system infrastructure rather than additional agents. Figure 1 illustrates the forward workflow and the associated refinement paths. In this study, an agent is defined as a goal-directed and researcher-supervised software component that can interpret user instructions, maintain task state, select from an authorized set of tools, and produce structured and auditable outputs. An agent is therefore not restricted to an LLM. Depending on the task, it may combine an LLM with deterministic rules, database operations, statistical software, machine-learning algorithms, or general-purpose programming tools. The proposed framework is consequently a multi-agent research workflow rather than an LLM-only pipeline. The termagentin this framework refers to a software agent that supports the research process and should not be confused with a traveler agent in conventional agent-based transport simulation. Human respondents remain the source of stated survey choices, while researchers retain control over the survey constructs, data-processing requirements, model specifications, and interpretation of results. Throughout this paper,active data collectionrefers to stateful conversational survey administration with predefined interaction logic and structured response Table 1: Roles, interfaces, and supported functions of the three agents. Agent Main inputs Authorized functions Main outputs Data Collection

**Source 3 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

Throughout this paper,active data collectionrefers to stateful conversational survey administration with predefined interaction logic and structured response Table 1: Roles, interfaces, and supported functions of the three agents. Agent Main inputs Authorized functions Main outputs Data Collection Agent Survey specification and collection instructions Chatbot construction, survey logic implementation, procedural question answering, response validation, clarification, deployment, and response aggregation Deployable survey interface, distribution link or QR code, raw responses, interaction logs, and collection metadata Data Processing Agent Raw survey data, optional external data, codebook, and processing instructions De-identification, validation, recoding, missing-data handling, text coding, data integration, feature construction, and quality assessment Processed dataset, quality flags, derived variables, supplementary data, and processing records Data Modeling Agent Processed data, modeling instructions, candidate model library, and evaluation requirements Model specification, code generation, statistical estimation, machine-learning training, LLM-based choice prediction, validation, and result organization Fitted model, parameter estimates, predictions, uncertainty measures, diagnostics, evaluation metrics, and structured

**Source 4 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

a commonmulti-agent workflowremains largely unexplored for travel behavior and individual mode-choice modeling. This study addresses these gaps through a multi-agent workflow that connects AI-assisted behavioral data collection with travel-choice modeling and LLM-based prediction. The paper makes four main contributions. First, it develops a conversational, image-augmented SP survey in which an AI-based chatbot collects traveler characteristics and repeated mode choices across predefined weather scenarios, providing a structured dataset specifically designed for downstream behavioral prediction. Second, it provides empirical evidence on weather-related variation in commuter mode choice by examining how the same individuals change their stated choices across systematically varied weather scenarios. Third, it systematically evaluates LLMs as mode-choice predictors under multiple prediction configurations, including zero-shot, few-shot, persona- and travel-history-enhanced prompting, as well as vision-based prediction using the same weather images presented to respondents, and benchmarks their performance against conventional DCM and ML approaches. Fourth, it demonstrates amulti-agent workflowthat links conversational survey administration, structured data processing, and parallel discrete-choice, machine-learning, and LLM-based modeling within an integrated data-collection-to-prediction process, providing a reproducible basis for exploring AI-assisted travel-behavior research. The remainder of the paper is organized as follows. Section 2 reviews related work on travelbehavior data collection and conversational agents, mode-choice modeling, LLM-based choice prediction, and multi-agent applications. Section 3 presents the proposed multi-agent methodology. Section 4 describes the case-study implementation, and Section 5 reports the descriptive and

**Source 5 — An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and Weather-Sensitive Demand Prediction**

Iterative feedback Survey refinement Processing refinement Modeling refinement Figure 1: Multi-agent framework for commuter travel-behavior research. Solid arrows denote forward data and artifact flows, whereas dashed arrows denote refinements implemented in a subsequent workflow iteration. from survey design through data processing to behavioral prediction on a common empirical dataset. The present paper addresses this gap by instantiating the three stages as specialized agents and applying them to the same commuter mode-choice dataset. Note:The table summarizes functions supported by the general framework. The subset implemented in the present case study is described in Section 4. capture; it does not refer to active learning or adaptive sampling. Weather-sensitive demand prediction refers to individual mode-choice prediction across the five predefined weather scenarios. The framework follows the information flow: Q Acol(·;ucol)− − − − − − − →(Draw,B col) Apro(·,E;u pro)− − − − − − − − − →(Dpro,Z pro) Amod(·;umod)− − − − − − − − → cM,R mod (1) whereQdenotes the researcher-defined survey specification;u col,u pro, andu mod denote the user instructions provided to the three agents;Draw is the collected raw dataset;Bcol is the survey deployment package; E represents optional external data; Dpro is the processed dataset; Zpro contains supplementary processing outputs; cM is the fitted model or set of models; andRmod contains model estimates, predictions, diagnostics, and evaluation results. Equation (1) represents one forward execution pass. The dashed arrows in Figure

Add source references to the answer itself.

Answer is 8 words; the review target is at most 120.


</details>

## q22 — Manual review needed

Which papers evaluate AI agents or model specialists using benchmark tasks and costs?

**Answer:** [3] and [4] evaluate AI agents or model specialists using benchmark tasks and costs. Specifically, [3] presents AI4AI-Bench, a benchmark that tests whether an agent can design training algorithms, and [4] presents Pandora's Router and Pandora's Bidder, which evaluate model specialists using benchmark tasks and costs in three domains: a multi-LLM routing benchmark, retrieval-augmented specialists for factoid QA, and inference-time computation for mathematical reasoning.

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Identifies Pandora’s AI Model Routing Box and connects its specialist routing evaluation to estimation costs and benchmark tasks.
- [ ] Identifies AI4AI-Bench and connects its algorithm-design agent evaluation to benchmark tasks and fixed compute/time budgets.

**Sources found:** 2 of 2 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 2**

> …that algorithmic design can be automated, but each demonstrates a method inside a compact search space and short proxy tasks; none is a benchmark of whether a general research agent can diagnose and improve the learning algorithm in an…

**Source 3**

> …algorithm: a better objective or update rule improves the compute-capability exchange rate for every subsequent run, including the one that produces the next agent. Whether RSI is feasible therefore turns on whether an agent can design…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

AI model providers now often offer heterogeneous model families spanning a wide range of costs and capabilities: smallandfastmodelsforsimpleprompts, largeandexpensiveonesforcomplextasks, and augmented variants with tools, retrieval, or extended reasoning for more specialized tasks. This raises the question of how to allocate each input to the model best suited for it, for a given level of cost sensitivity. This is theroutingproblem, and a growing body of work addresses it by estimating each model’s expected return on the input and selecting the maximizer (Feng et al., 2026; Hu et al., 2024; Shnitzer et al., 2023). Butvalue estimationis neither free nor particularly easy. An embedding-based predictor is cheap but noisy; a fine-tuned scoring model is more accurate but more expensive; and computing partial solutions, executing tool calls, or running retrieval pipelines can be more expensive still. The routing decision itself thus involves a cost-accuracy tradeoff, which is typically ignored. Consequently, this paper asks the underlying question: when is it worth paying for a better value estimate? The question has a clean analogy in thePandora’s Boxproblem from search theory (Weitzman, 1979). Pandoraispresentedwith 𝑀 boxes, eachwithahiddenvaluewhichisknowntoheronlyindistribution. For a cost𝑐𝑚 she can open any box𝑚 and observe its value. At any point, she can stop searching and claim the best value seen thus far. Her goal is to maximize the difference between the value obtained minus the costs paid.

**Source 2 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

equations , AutoML-Zero evolves learning algorithms from primitive operations , and symbolic program search discovered Lion . These works establish that algorithmic design can be automated, but each demonstrates a method inside a compact search space and short proxy tasks; none is a benchmark of whether a general research agent can diagnose and improve the learning algorithm in an existing repository. Existing agent benchmarks instead score broader outcomes. MLE-Bench and MLE-Dojo reward competition submissions, while ML-Bench exercises repository-level ML tasks ; they mix data and feature engineering with model selection, hyperparameter tuning, debugging, and ensembling. Research-agent systems and benchmarks further cover idea generation, experimentation, paper writing, replication, and open-ended workshop problems , and Frontier-Eng extends executable, verifierdriven improvement to real-world engineering designs . The closest benchmarks begin with an AI system and ask an agent to improve it , but their scores aggregate gains from execution, data, capacity, hyperparameters, and learning rules; accordingly, they find tuning and engineering easier than method invention. Evenautoresearch, which opens architecture, optimizer, and training-loop code , yields edits that behave largely as hyperparameter optimization in controlled comparison . These benchmarks therefore test whether an agent can produce a better artifact, not whether it improved the learning rule that produces its successor—the algorithmic-design step at the core of AI4AI. To our knowledge, AI4AI-Bench is the first benchmark to make that step the object of evaluation: it removes the agent, reruns the submitted source from a clean start, and classifies the patch to verify

**Source 3 — AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement**

AI4AI-Bench: Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Improvement Yizhe Chi , Wenyi Li, Deyao Hong, Xiaoqiu Wang, Mingju Gao, Kaisen Yang, Bingxiang He, Y oujie Zheng, Calvin Xiao, Qinhuai Na Navers Lab, Einsia.AI T singhua University Project Lead Corresponding Author Abstract.Recursive self-improvement (RSI) asks whether an AI system can improve the process that produces AI systems, so that the next system inherits the improvement. That process is the training algorithm: a better objective or update rule improves the compute-capability exchange rate for every subsequent run, including the one that produces the next agent. Whether RSI is feasible therefore turns on whether an agent can design training algorithms. No benchmark isolates that ability: existing suites are won by collecting data or by tuning hyperparameters, and none tells a change to how a run is executed apart from a change to how the model learns. We present AI4AI-Bench, 10 frozen research repositories spanning 10 training algorithm families. In each task, an agent has 4 hours on one B300 to rewrite the training algorithm; its code is then rerun from scratch for up to 12 hours and scored by a fixed evaluator hidden from the agent, against the repository’s original algorithm under the same procedure. Because the 10 metrics are incommensurable, every task is mapped onto one scale on which0is an uninformative model,0 .1is the algorithm the repository ships, and1.0 is the task optimum. Across 29 configurations of 6 systems

**Source 4 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

2026-08-21 Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation Adam Fisch, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate Larson, Jacob Eisenstein Google DeepMind Heterogeneous AI systems composed of multiple models, architectures, harnesses, or inference-time settings can improve quality and efficiency by routing queries to thespecialistwho can answer most effectively at the lowest cost. Routing requires estimating each specialist’s expected return, but this value estimation has a cost. Cheap estimators (e.g., embedding-based predictors) are fast but noisy, while accurate estimators (e.g., fine-tuned models with access to retrieval results or partial reasoning traces) are expensive. We formalize this tradeoff as an instance of Pandora’s Box, the classical problem of optimal search with costly inspection. Under a Gaussian signal model, the resulting policies have closed-form value-of-information expressions that determine, for each specialist and input, whether refining the value estimate is worth its cost. We call the centralized policy Pandora’s Router. We extend this to a decentralized setting, Pandora’s Bidder, where specialists independently decide whether to invest in self-assessment before accepting an offered price to claim a query. Experiments across three domains—a standard multi-LLM benchmark, retrieval-augmented specialists, and LLMs with variable inference-time reasoning—show that Pandora’s Router matches the routing quality of exhaustive estimation, while querying the expensive estimator far less often. In

**Source 5 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

Bidder: each specialist faces a posted price—the current best competing offer—and must decide whether to invest in a more accurate self-assessment before claiming the query. This corresponds to a single stage of the ascending-price mechanism with costly preference elicitation studied by Parkes (2005); analyzing this simple setting isolates core components of decentralized prompt allocation under private value estimation with profit incentives. Core contributions of this work As AI systems with diverse capabilities proliferate, reasoning about the economics of model selection becomes increasingly relevant. To our knowledge, this is the first paper to explicitly connect model routing with the classical Pandora’s Box problem from economics and operations research. Specifically, this paper makes three main contributions: • We formalize model routing with costly value estimation as an instance of Pandora’s Box with non-obligatory inspection (Doval, 2018), yielding a reservation-price-based policy. Under a Gaussian signal model, this policy has a closed form that is straightforward to compute (4). • We extend to a decentralized setting where specialists use value-of-information reasoning to decide whether to refine their self-assessments before accepting a posted price (5). • We evaluate both frameworks (see Figure 1) across three domains: a multi-LLM routing benchmark (EmbedLLM), retrieval-augmented specialists for factoid QA, and inference-time computation for mathematical reasoning. In each setting, both Pandora’s Router and Pandora’s Bidder are able to gracefully trade-off expensive and cheap estimates to achieve strong allocative performance relative to baselines across a range of estimation costs.

Source references exist; whether they support each claim still needs review.

Answer is 60 words; the review target is at most 120.


</details>

## q23 — Manual review needed

Which papers discuss evidence-grounded explanations and auditable task models?

**Answer:** [1] and [2] discuss evidence-grounded explanations and auditable task models. [1] specifically focuses on inducing task models from computer-use traces, including objective models that explain why activities were performed and procedure models that capture how the execution of the task was organized in time. [2] introduces TASKMODELINDUCTION (TMI), which induces task models pairing hierarchical objective models with procedure models, and demonstrates its effectiveness in recovering interleaved tasks and reconstructing execution steps.

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Identifies G-CARL and connects it to evidence-grounded patient-facing interpretations.
- [ ] Identifies TMI/Inducing Task Models from Computer-Use Traces and connects it to auditable models of user activity.

**Sources found:** 2 of 2 expected papers. This measures paper retrieval, not whether the answer uses the right evidence.

**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 2**

> …models matter as computer-use agents enter real work, where agents need to learn how tasks are actually performed, and organizations need to audit and reuse that knowledge. However, inducing such task models is challenging, as activity…

**Source 4**

> explanations grounded inmultimodalreportsandextendedpatient–doctordialogue. cation by transforming report evidence into accessible explanations, addressing user-specific concerns, and offering clinically cautious guidance. As illustrated…

<details><summary>Full sources and automatic checks</summary>

**Source 1 — Inducing Task Models from Computer-Use Traces**

Modeling a task t requires understanding the two types of evidence it carries, procedure and objective. Constructing a task model thus means inducing an objective model Ot and a procedure model Pt from the activities Et of each discovered task t, then reconciling them into the task model Mt (Figure 2(c)). Each is induced independently, so that its structure is resolved under its own evidence before being constrained by the other. Objective model.The objective model is the hierarchy that explains why the observed activities were performed. Its root is the task objective, its leaves are the activities Et, and its internal nodes are latent sub-objectives that jointly explain their descendants. A language model induces Ot from the task objective and Et by recursive decomposition (Wing, 2006), breaking a goal that cannot be pursued directly into sub-goals whose union covers the relevant evidence. Each node denotes the outcome to be achieved, abstracting over the particular strategy used to reach it (e.g.,provision reviewer accountsinstead ofset up each account and verify its sign-in). The children of a node are necessary components or preconditions of their parent and jointly account for its observed activities. A node spanning many activities is refined further (e.g.,iterate on the design decomposes intoimplement changesandvalidate the fix), whereas a node grounded in a single activity has reached the level of a local objective and remains a leaf. The resulting hierarchy is abstract enough to transfer across executions yet grounded enough that every leaf stays tied to an observed activity. Objective-side validity constraints are detailed in Appendix A. Procedure model.The procedure model captures how the execution of the task was organized in time.

**Source 2 — Inducing Task Models from Computer-Use Traces**

Naturalistic computer-use traces, passively recorded screenshots and mouse or keyboard actions, are a valuable resource for deriving symbolic, auditable, and reusable models of how everyday work is done. Such models matter as computer-use agents enter real work, where agents need to learn how tasks are actually performed, and organizations need to audit and reuse that knowledge. However, inducing such task models is challenging, as activity is observed only as low-level events and realworld work is multi-threaded with interleaved goals. Existing methods assume a given task or a single workflow, and produce step-level summaries rather than structured task models. We introduce TASKMODELINDUCTION(TMI), which (i) discovers the latent tasks in an unconstrained trace, disentangling concurrent activity, and (ii) for each latent task, induces a task model pairing a hierarchical objective model of recursive goal decomposition with a procedure model of the control flow that organized the execution. Intrinsically, on controlled human and agent trajectories, TMI recovers interleaved tasks with 0.974 agreement against ground-truth groupings and reconstructs 74.9% of the observed execution steps, far more than the strongest workflow induction baseline. Extrinsically, skills derived from TMI’s task models improve held-out task accuracy by 30.0% over the strongest baseline. 1

**Source 3 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

In this paper, we present PMRI as a challenging yet underexplored task that requires both evidence-grounded medical factualityandcontext-dependentpatientcommunication.We propose G-CARL, a reinforcement learning framework that decomposes reward supervision through retrieval-grounded claim verification and case-specific checklist guidance. ExtensiveexperimentsonthenewlyconstructedMMedReport benchmark, together with clinician-authored evaluation and human preference studies, show that G-CARL consistently improves the quality of patient-oriented medical report interpretation.Wehopethisworklaysthefoundationformore reliable and patient-centered multimodal medical assistants.

**Source 4 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

explanations grounded inmultimodalreportsandextendedpatient–doctordialogue. cation by transforming report evidence into accessible explanations, addressing user-specific concerns, and offering clinically cautious guidance. As illustrated in Fig. 1, PMRI differs from conventional medical visual question answering (VQA) (Lu et al. 2025; Jiang et al. 2025a) and report generation (Lin et al. 2026; Li et al. 2025) in two important ways. First, it is anevidenceboundedgenerationtaskwhoseresponsesmustbegrounded not only in the reports but also in reliable clinical knowledge and medically coherent diagnostic reasoning, particularlywhenexplainingabnormalitiesorsuggestingfollow-up actions. This requirement is essential for patient safety because unsupported interpretations and inappropriate recommendations may mislead patients or delay necessary care. Second, PMRI is apatient-facingcommunication task that mustgobeyondclinicalcorrectnesstoaddresspatients’individual information needs while clearly explaining potential abnormalities and risks in an emotionally adaptive and reassuring manner. Accordingly, we model PMRI quality along three core dimensions: medical accuracy, demand satisfaction, and expression quality. Giventhe natureof PMRI,astraightforward strategyisto collectlarge-scalephysician-writteninterpretationsandtrain multimodalmodelswithsupervisedfine-tuning(SFT)(Chen et al. 2024b; Rotstein et al. 2024). However, SFT can overfit to the specific wording of the reference interpretation and

**Source 5 — Inducing Task Models from Computer-Use Traces**

evidence-grounded summary; may reference specific tools, files, or actions observed&gt;", "subgoal_segments": ["&lt;single integer id such as 16 or closed integer range string such as 16-23&gt;"], "decomposition": [&lt;child nodes with the same schema&gt; or &lt;empty list if no further decomposition is needed&gt;] } """ Figure 9: Objective-model induction prompt. PROCEDURE_GENERATION_PROMPT_TEXT = """ You induce a procedure model from a task-thread objective JSON by applying the Structured Programming Theorem (Bohm-Jacopini, 1966). Return only a valid JSON object. The Structured Programming Theorem motivates three control constructs: 1. Sequence -- steps executed one after another in order. 2. Selection -- a choice between mutually exclusive alternative paths. 3. Iteration -- a body repeated either over a named collection (for-each) or until a condition holds (while). The trace records only the path enacted, not unchosen alternatives, so selection is not represented in the output. Decompose the observed activity trace into a tree using the three observable operators below. The three operators are the complete and closed vocabulary: -`SEQ`: steps in fixed order with no repetition and no branching. -`FOR`: the same procedure body is applied to each member of a named, enumerable collection. -`WHILE`: a body is repeated until an observable objective-state condition is satisfied. FOR and WHILE bodies are ABSTRACT TEMPLATES -- they describe what happens per item / per pass using named steps with`name`,` description`,

Source references exist; whether they support each claim still needs review.

Answer is 67 words; the review target is at most 120.


</details>

## q24 — Manual review needed

What is KPMG's 2026 revenue forecast?

**Answer:** The provided papers do not address this question.

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Clearly states that these papers do not provide KPMG’s 2026 revenue forecast, without inventing a forecast.
**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 2**

> …because it offers a single source of prices for LLM inference, embedding, and retrieval. The resulting estimates are meant only to show that approaches to value estimation can incur vastly different costs; we do not claim that these…

**Source 3**

> …is always a clear best box to use based on cheap signals alone; or we can have a few boxes that are hard to choose between, and it is expensive to know more. We picked experimental settings that try to cover key aspects of each of these

<details><summary>Full sources and automatic checks</summary>

**Source 1 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

(StepFun AI 2026) 1.842±0.011 1.175±0.012 0.412±0.002 0.255±0.001 95.96 72.11 Kimi K2.5 (Team et al. 2026a) 1.903±0.002 1.221±0.002 0.418±0.001 0.264±0.001 97.63 76.42 Gemini 3.1 Pro (Google DeepMind 2026) 1.964±0.008 1.229±0.008 0.438±0.003 0.297±0.002 98.10 80.29 Medical LVLMs Hulumed (Jiang et al. 2025b) 1.020±0.018 0.480±0.013 0.282±0.004 0.258±0.002 72.28 32.65 Medgemma (Sellergren et al. 2026) 1.004±0.005 0.460±0.008 0.289±0.004 0.255±0.001 74.30 33.86 Lingshu (Xu et al. 2025) 1.527±0.003 0.902±0.002 0.357±0.003 0.268±0.002 89.69 39.41 Qwen-VL series Qwen3-VL-4B-Instruct (Base) 1.527±0.010 0.897±0.007

**Source 2 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

costs To estimate monetary costs for the value estimators in Table 1, we use prices fromhttps://ai. google.dev/gemini-api/docs/pricing (retrieved August 1, 2026), because it offers a single source of prices for LLM inference, embedding, and retrieval. The resulting estimates are meant only to show that approaches to value estimation can incur vastly different costs; we do not claim that these specific prices are optimal or even typical. Because the cost-accuracy tradeoff is auser characteristic rather than an objectively-measurable property of the domain or method, we focus on theratiobetween the prices of 𝑓 and 𝑔. Please see Section D.3 for the derivation of these ratios. 3. The Pandora’s Box Problem We begin by briefly reviewing the Pandora’s Box problem (Weitzman, 1979), which we will connect to model routing and model bidding in the following sections. In the Pandora’s Box problem, a decision-maker (i.e., Pandora) is presented with𝑀 options, called “boxes”, each with an unknown reward (but with known distribution). For a price she can inspect any box to observe its reward, and 1Compare, e.g. Fable 5 vs Sonnet 5 athttps://platform.claude.com/docs/en/about-claude/pricing (retrieved August 13, 2026). Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation at any time she can terminate the search and take the best reward that she has observed. The optimal policy depends on a mathematical object called thereservation price, which is

**Source 3 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

the embedding space misses, but more expensive to run per query in terms of inference. EmbedLLM incorporates more than 100 open-weights models as routing targets; following Jitkrittum et al. (2025), we assign each model a cost proportional to its parameter count. Queries are drawn from standard benchmarks like MMLU (Hendrycks et al., 2020) and GSM8K (Cobbe et al., 2021). While the Math and RAG settings have only two and three target models respectively, both routing and efficient value estimation are still challenging and important problems even when the number of target models is small. The strongest frontier models can be up to five times as expensive as cheaper models from the same provider,1 so there are strong financial incentives to reserve the most expensive models for situations where they yield meaningful improvements. Similarly, as shown in Table 1, more computationally-intensive value estimation strategies can yield significant improvements, but at high cost. The difficulty of the information acquisition problem depends not primarily on the number of targets, but on how discriminable the options are: we can have many options, but this doesn’t matter if there is always a clear best box to use based on cheap signals alone; or we can have a few boxes that are hard to choose between, and it is expensive to know more. We picked experimental settings that try to cover key aspects of each of these considerations. 2.2. Value estimator costs To estimate monetary costs for the value estimators in Table 1, we use prices fromhttps://ai. google.dev/gemini-api/docs/pricing (retrieved August 1, 2026),

**Source 4 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

2026-08-21 Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation Adam Fisch, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate Larson, Jacob Eisenstein Google DeepMind Heterogeneous AI systems composed of multiple models, architectures, harnesses, or inference-time settings can improve quality and efficiency by routing queries to thespecialistwho can answer most effectively at the lowest cost. Routing requires estimating each specialist’s expected return, but this value estimation has a cost. Cheap estimators (e.g., embedding-based predictors) are fast but noisy, while accurate estimators (e.g., fine-tuned models with access to retrieval results or partial reasoning traces) are expensive. We formalize this tradeoff as an instance of Pandora’s Box, the classical problem of optimal search with costly inspection. Under a Gaussian signal model, the resulting policies have closed-form value-of-information expressions that determine, for each specialist and input, whether refining the value estimate is worth its cost. We call the centralized policy Pandora’s Router. We extend this to a decentralized setting, Pandora’s Bidder, where specialists independently decide whether to invest in self-assessment before accepting an offered price to claim a query. Experiments across three domains—a standard multi-LLM benchmark, retrieval-augmented specialists, and LLMs with variable inference-time reasoning—show that Pandora’s Router matches the routing quality of exhaustive estimation, while querying the expensive estimator far less often. In

**Source 5 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

Feng et al., 2026; Hu et al., 2024), of which we use EmbedLLM (Zhuang et al., 2025). Most routing approachesrequirepredictingtheanswerqualityforeachmodelonagiveninput,usingpointwise(Jitkrittumetal.,2025)orpairwise(Ongetal.,2025)data. Inallofthispriorwork,valueestimationistreated as a cost-free operation with a fixed error profile. Our contribution in Pandora’s Router in Section 4 is to treat value estimation as a decision involving a cost-performance tradeoff of its own. Adaptive retrieval and tool use.Closely related to routing is the decision of whether to invoke external tools or retrievers. Adaptive RAG frameworks (Jeong et al., 2024) dynamically adjust retrieval strategiesdependingonthequerycomplexitytosavecomputationalcosts. Similarly,recentevaluations of adaptive retrieval (Moskvoretskii et al., 2025) highlight the tension between using internal LLM uncertainty estimation vs. lightweight external heuristics. We formalize this exact tradeoff—that is, deciding whether to spend computational power on self-assessment or tool-planning—through a value-of-information perspective that is both theoretically principled and empirically effective. Pandora’sBoxandoptimalsearch.Priorworkonsequentialsearchwithcostlyinspectionisreviewed in 3. The Pandora’s Box problem is also used as a motivating framework for LLM reasoning about

No source reference is required for this refusal.

Answer is 8 words; the review target is at most 120.


</details>

## q25 — Manual review needed

Which of these papers reports the stock price of NVIDIA on October 14, 2026?

**Answer:** The provided papers do not address this question.

**Automatic findings:** No mechanical issues flagged. This does not mean the answer is correct.

**Check these points:**

- [ ] Clearly states that none of these papers supplies NVIDIA’s stock price on October 14, 2026, without inventing a price.
**Source previews** — copied from saved passages, at most 240 characters each. Selected by word overlap for navigation; inspect full passages to verify claims.

**Source 2**

> …because it offers a single source of prices for LLM inference, embedding, and retrieval. The resulting estimates are meant only to show that approaches to value estimation can incur vastly different costs; we do not claim that these…

**Source 4**

> …is always a clear best box to use based on cheap signals alone; or we can have a few boxes that are hard to choose between, and it is expensive to know more. We picked experimental settings that try to cover key aspects of each of these

<details><summary>Full sources and automatic checks</summary>

**Source 1 — G-CARL: Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical Report Interpretation**

(StepFun AI 2026) 1.842±0.011 1.175±0.012 0.412±0.002 0.255±0.001 95.96 72.11 Kimi K2.5 (Team et al. 2026a) 1.903±0.002 1.221±0.002 0.418±0.001 0.264±0.001 97.63 76.42 Gemini 3.1 Pro (Google DeepMind 2026) 1.964±0.008 1.229±0.008 0.438±0.003 0.297±0.002 98.10 80.29 Medical LVLMs Hulumed (Jiang et al. 2025b) 1.020±0.018 0.480±0.013 0.282±0.004 0.258±0.002 72.28 32.65 Medgemma (Sellergren et al. 2026) 1.004±0.005 0.460±0.008 0.289±0.004 0.255±0.001 74.30 33.86 Lingshu (Xu et al. 2025) 1.527±0.003 0.902±0.002 0.357±0.003 0.268±0.002 89.69 39.41 Qwen-VL series Qwen3-VL-4B-Instruct (Base) 1.527±0.010 0.897±0.007

**Source 2 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

costs To estimate monetary costs for the value estimators in Table 1, we use prices fromhttps://ai. google.dev/gemini-api/docs/pricing (retrieved August 1, 2026), because it offers a single source of prices for LLM inference, embedding, and retrieval. The resulting estimates are meant only to show that approaches to value estimation can incur vastly different costs; we do not claim that these specific prices are optimal or even typical. Because the cost-accuracy tradeoff is auser characteristic rather than an objectively-measurable property of the domain or method, we focus on theratiobetween the prices of 𝑓 and 𝑔. Please see Section D.3 for the derivation of these ratios. 3. The Pandora’s Box Problem We begin by briefly reviewing the Pandora’s Box problem (Weitzman, 1979), which we will connect to model routing and model bidding in the following sections. In the Pandora’s Box problem, a decision-maker (i.e., Pandora) is presented with𝑀 options, called “boxes”, each with an unknown reward (but with known distribution). For a price she can inspect any box to observe its reward, and 1Compare, e.g. Fable 5 vs Sonnet 5 athttps://platform.claude.com/docs/en/about-claude/pricing (retrieved August 13, 2026). Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation at any time she can terminate the search and take the best reward that she has observed. The optimal policy depends on a mathematical object called thereservation price, which is

**Source 3 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

As noted in 2.2, we estimate monetary costs from prices listed athttps://ai.google.dev/ gemini-api/docs/pricing (retrieved August 1, 2026). In all evaluations, the𝑓 value estimator is a small MLP applied to a question embedding. The MLP itself is very cheap to run, and can be assumed to be cost-free. The embedding costs $0.20 per million tokens, and this can be amortized across target models, although to be conservative about𝑐𝑔/𝑐𝑓, we do not account for this amortization here. The𝑔value estimators are priced as follows: • SFT-CoT-20, used in the MATH domain, requires running the target model LLM reasoning process for 20 tokens. The cheapest frontier LLM (Gemini 3.5 flash-lite) costs $2.50 per million outputtokens,and$0.25permillioninputtokens. IntheMATHdomain,thereareapproximately 43 tokens per question. This gives𝑐𝑓≈ 8.6× 10−6 and 𝑐𝑔≈ 2.5× 20× 10−6 = 5× 10−5, yielding 𝑐𝑔/𝑐𝑓≈5.8. Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation 10−2 10−1 0.05 0.06 Surplus Regret MATH 10−2 10−1 0.1 0.15 RAG 10−5 10−4 10−3 10−2 10−1 0.09 0.095 0.1 EmbedLLM 10−2 10−1 0.02 0.025 0.03 𝑐𝑔 Efficiency Regret 10−2

**Source 4 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

the embedding space misses, but more expensive to run per query in terms of inference. EmbedLLM incorporates more than 100 open-weights models as routing targets; following Jitkrittum et al. (2025), we assign each model a cost proportional to its parameter count. Queries are drawn from standard benchmarks like MMLU (Hendrycks et al., 2020) and GSM8K (Cobbe et al., 2021). While the Math and RAG settings have only two and three target models respectively, both routing and efficient value estimation are still challenging and important problems even when the number of target models is small. The strongest frontier models can be up to five times as expensive as cheaper models from the same provider,1 so there are strong financial incentives to reserve the most expensive models for situations where they yield meaningful improvements. Similarly, as shown in Table 1, more computationally-intensive value estimation strategies can yield significant improvements, but at high cost. The difficulty of the information acquisition problem depends not primarily on the number of targets, but on how discriminable the options are: we can have many options, but this doesn’t matter if there is always a clear best box to use based on cheap signals alone; or we can have a few boxes that are hard to choose between, and it is expensive to know more. We picked experimental settings that try to cover key aspects of each of these considerations. 2.2. Value estimator costs To estimate monetary costs for the value estimators in Table 1, we use prices fromhttps://ai. google.dev/gemini-api/docs/pricing (retrieved August 1, 2026),

**Source 5 — Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation**

2026-08-21 Pandora’s AI Model Routing Box: Efficient Allocation with Costly Value Estimation Adam Fisch, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate Larson, Jacob Eisenstein Google DeepMind Heterogeneous AI systems composed of multiple models, architectures, harnesses, or inference-time settings can improve quality and efficiency by routing queries to thespecialistwho can answer most effectively at the lowest cost. Routing requires estimating each specialist’s expected return, but this value estimation has a cost. Cheap estimators (e.g., embedding-based predictors) are fast but noisy, while accurate estimators (e.g., fine-tuned models with access to retrieval results or partial reasoning traces) are expensive. We formalize this tradeoff as an instance of Pandora’s Box, the classical problem of optimal search with costly inspection. Under a Gaussian signal model, the resulting policies have closed-form value-of-information expressions that determine, for each specialist and input, whether refining the value estimate is worth its cost. We call the centralized policy Pandora’s Router. We extend this to a decentralized setting, Pandora’s Bidder, where specialists independently decide whether to invest in self-assessment before accepting an offered price to claim a query. Experiments across three domains—a standard multi-LLM benchmark, retrieval-augmented specialists, and LLMs with variable inference-time reasoning—show that Pandora’s Router matches the routing quality of exhaustive estimation, while querying the expensive estimator far less often. In

No source reference is required for this refusal.

Answer is 8 words; the review target is at most 120.


</details>
